In [ ]:
# SH01 integrated editor — paste this entire file into ONE Kaggle Python cell.
# Enable GPU + Internet in Kaggle. Models download on the first run; no paid API key.
SOURCE_PATH = '/kaggle/input/datasets/vivanjaiswal/video1/military_seth_raw_cut1.mp4'
BRIEF = ('Create a polished, informative vertical short. Select a coherent story and a strong opening; '
         'preserve the speaker’s meaning and voice. Add brief grounded narration in separate intro, bridge '
         'and outro sections. Use relevant supplementary images or video, readable motion graphics, '
         'purposeful camera moves and transitions, restrained SFX and a quiet music bed.')
TARGET_SECONDS = 35
NARRATION = 'intro_bridge_outro'   # 'intro_outro', 'intro', 'auto', or 'off'
REQUIRED_FEATURES = ['voiceover','sfx','music','camera_motion','transitions','graphics','supplementary_media','captions']
REFINEMENT_ROUNDS = 2
OUTPUT_SIZE = (1080, 1920)       # Use (720,1280) for a faster draft.
LOCAL_ASSET_LIBRARY = ''        # Optional annotated media directory; uses Commons otherwise.
MUSIC_PATH = ''                 # Optional local track; empty => original procedural ambient bed.
PRIVATE_ENV_FILE = '/kaggle/working/sh01-fastapi-server/.env'  # Read if present; never overwrite.
TEXT_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
VISION_MODEL = 'Qwen/Qwen2.5-VL-3B-Instruct'
VOICE_NAME = 'en_US-ljspeech-high'  # English public voice; change for another language.
ASR_DEVICE = 'cpu'             # 'cuda' is optional; director and vision use the GPU either way.
KEEP_LOCAL_API = False         # True leaves the loopback API running after this job.
TIMEOUT_SECONDS = 7200

# -------------------- automated setup and execution --------------------
import base64, hashlib, io, json, os, shutil, subprocess, sys, tarfile, time, urllib.request, venv, zipfile
from pathlib import Path

# Embedded, inspectable Python source from the matching repository patch.
BUNDLE = (
'P)h>@6aWAK2mk;8VqLIRB-*_W0049{000dD003cWUt@1=aBp*EE^v9pTWxdOxDo#D8UF{C9Z$#`QC6JXwbfLeDz<XbMSiL5T<_|7I4}uHI8!7K'
'BxS3v|9f`<kOW^O+c#~h4>3tB7K_EcJiBB})BH%%Y<w{_c${Z6HAv(ee$L5lNS$BeTZ2UzU2u}II5L8m8{?_r#j7Mv89wTprn$ct@OWjoZjfg='
'r><)-80Q(VisDSnO!xQp_ll?KCh}NZ-Y-eI4B2gUzan0F6UK`Lix%a5oR*iImX{e@RX2Igd?ivsGGtxIa{;$%B-<pwwip;k8x&BC1fUL*uyW{P'
'I!DFup7DtiU;Mqgx|lq8G{K$KXZ1Nv6WA{3QO*CzLONFud&Uo!=V!%Txgue%)F^dn1iPih+++E4!g&nK@;q`i&9^5rUQC?$EV~Fv2TkfN>57z_'
'o2n(|^w*rGSx+q~7C;6)!7ApAgcothyp7_xqCO)on1`FfXeqF`sgZy_;+%VxKGKBTN3I;qNP5>Z)+A&;$*8+zi)A$d)sykfaDUH$pT^Q73GrCA'
'DL9@^PNrjbJ~=y?%-z}f@#p<LDo)3k3Id_Np<(>qNH|%n$k2#lFyS@jhGXO@4f4>qU((1>y2)bLDcVMgDmO?NM^p;YTFDp{*4&@P-koKm&}g#2'
'dsNtNe?haEcxIW}Nl14yVT!oI38k%B38w*j7@3D+%Vy8W$RZtGo8WvuGM~o!Ihi+N+40mpK0m!Uznr)~oPTgXPtQ((rM4pY>B*P*_`_`CUQJIY'
'=U=Ye%gOQi*~ul)t;mD*Z4aH@ic^%BMr+H|cy5^rqM7dorfq`^xa@p#H95PQo}V>%zx!@5*u}ncE;4`nhoe6Z)B^i^XmaV$@R^rgrx`d!bp<|r'
'a|5eAJ}ELcgd^!VA@`PecMDK@<O>S~ekska3$iPCgnZzcJkR|-p9Y2-#rKwNIDY{n=gKHKg?JZ<=+lR%0bW?<!OzaY$~o|_4n7SJPKO7V|G@J@'
'n#D>`*g9Kj(|2^Ug}JUTX@AGF>4giX;(ng|j2bd+T%0Aq0K&6Iel?j*zMqdz-OH=<`S|-u_at2y4q<>n1V?eeEps|VfsB6}XW#=uKD<x@(uah4'
'8TCO|`Yz-k3&6{=l15Dt`ywo*^%!bF?9B2M4ArNjZ_!YEnh_R`I(Ce1iJZMS3fKbDjc?YM6mp{`@CI{9AMKyOu6(wD7$-N@D@C(kA=~@GTTM2W'
'rr_R(ED{mS_{jkg9n`O`QjC$zx3~<ja4A&UIBQT`LyNcp@Li|!UCk!Mt|EGG0coE`D4J#Wj6j<(e5_V{GBqNVZTsnx9DVzp-CxoNr5F2p`0nVY'
'A^MuZGH@|$c$(Ni{ZZX+A3?UEIt+vjaNAS2EHZ#&pJjBF!cNt>;a))3JWKCcwzN#ww4bCf!ZQsaQbKvkK-Lb4AVJC<4_j$X88E20wBuU{@=D+;'
'$>1*`n=mG5)Q@JRLUSlY2G6qer=ZW`7AVwJe+8-zT|ZwX7G93bUs8zD_7m<@(HJxYLpv)o3;xJGVI?~1<Bbc7vnm2Xq=vKD&6XShNn;#;A`ADE'
'Ji`<0b&YdW;I%1j6Y@s~QAax2&PpmqdWtD+H;?WhV#2mNW+@$ajMQwdYihX|#jM`q-n_?n3mJ8^C3|VqwiX4X1z+);7LiWCi83a@Dd02BYGlrb'
'L*o&I1k7v-B+~**8_IM6DFPz&aU-JxNIeHBgyV8o>O^b|PhMN3k0g}1vE~%0OIIypNe)$zsXK}Cgig!n@jYhCZYyltowPELT3DGO*R^}ksobuq'
'v`o5kl7MP$?LMC}-c@0_L-k=C-{pyr%r{^eFj0XgVU=3yKn=k9@JF7;9gE;BjNgsAykXcnoywg7#O7fJ+ALYgCt9jvkUL-xkw+~#Cb5j3jX+Xx'
'V&H-UimFf_*g+TNV_`u<bs!L+lv@rfVG2iuLFq=?nDVP9$y_qNHkBq#1V)u#G)CP#K^*V9Bol-sPwlx~OA?YnJIVv@rUJuRP0-=xSsVnwTmUKw'
'rvS!x!bdqn^2MBDRIpIf9VjEj)Noyo98!HEbrqVF)v)AJWUu-pIM;SGgei|Enq&yL46&Dt0OnI8vP0b+fGcNJKy?Eav2+QVqe<*7Wn|ShZX17A'
'3aMD#RF{?PD1+GNu)e0;N)jhlu}6_0H4Kn(0LvD_<e{M><mS1wKy{z^zL=*}gqQft^<fg!aAAlutoA&gr5@qF-LeY$Kw-1B2)85WHwApo4DfJA'
'a_CyFpftG)5M1j|Ze9fjaC4PhxIyV9);He>lmw{(0(Ns3O47cfm6Vzh%IK}bq*6t8v0^>yYkFywx5Ji86bnn^9+}{%nDD@=;mHDaClFlugiEg<'
'$>OQuEW2fGQ#Se^1*8^!*;Cr9XP4&_HZAjso;PYK+AMQh=MfZ|P)=PcqP!OYb%a0(Q#xSm!C(NUQo44jo}SV0_mS@EFF#}Hxhryq6;GOw%v+*a'
'ztC@b2FW0_B`^fldd~<c!Pe4NM4@zQn&vV1G-&O1qxv+(;*^U>lr5`-3@bq$aLXb6EoBE0c@&jHwbfQO(}rrkSa@6Y(5M2pHmS^omF3n)7ZeAD'
'e12^#;qd}eFwPzH2l|N87YwK?eb57}q*C8d^$_%7nE;XL^-Wt&H|HS?1?FQ_)xWv;@?kbT7Ul4V@#W-A&v^63?sHkqzd3wkKgp)iVO0!bA2&UG'
'l<Ih@ZL932xy3=lg2kmGY?Wc0F2M3AZ`oI8mH3JvT!!DOHbEc(*HVPP%2pX<EHHdi+acnLN<OJ{w`zS^JIwvTB?)sywOaK$U{TrawSXT1+%>|d'
';2R+$;OttdUjVjR@1*3Z+;&Tgvk<GFM<H7=w3Bj7%`BW<LqvtMl$Jg+tZw_qpo87;KK}C_f=Jo)kvtSQ7<F_h?0SCMeq{6kOaa@aDtQXz(h8UW'
'j7mZTI>k*FR4SZL^HHP1Pfv}cq=JUtSg+LvM}vXBdSk)SNMkLF@}t2~54%g?Z4jBudbD6rMB*Zp6%|ug6;`4Bf?9jIvP+V7ZLw>AmAH!amQ>$W'
';BD(8$m&iUg-avxw^3TRf-C->?;1RI25woYyc|7kE8KUNUFtBlm0sd>T1C=U{0>+6ZjAFh$q>%J-Y~0!s}*-Io2^;tV}0cvSyK#eJAek+*1EA*'
'Cdg*HrOcqOm{O(Ba*}$4(*|8je3jHF7Tc1a#aK}~i_B1sJ-rx%f+=>t3<2ytp1B{#v)PC7@n=+C1~s>7_rv%eQj#s26~ZjvmV@wjL({euzM^#!'
'0ECO;h(U2CA)LZ#c|_JQj*(?(Ji;EHS`8HZ$qMR!Y>OSxtY&!I#<#+^h7Gocb_2L)jpVCZs|?qHs;yx`Y}M$d*;)2=rgi+)is1r=l?+}~y{Y~|'
'^QEuVDCVoF<y9vp`mF)7-d48jU{Zvf?Mkj8OKf*)?{HgX)z<)%TOr|dX9~CA+0WrLLDikhDm4nCqO(&28PflD;0zsEfDDc5`uV8&x`_6-mqswy'
'xfHz~pf#)@hwHea-L%!zB_~`L#5XPD?dsOn0%Cp1+H0pWBI-0+W48#1TZXHMrxVl@ZzHl*8Bd>5?-&0jRNO|pYCu&w&%Ts>Uv>Y#$k$F-{m_R5'
'T*x=H#tsNzKIi>d5Z|(pWlVmZQ8~Zn_&SmGHJrYqH*z2Iy8sXYX1#Pdc&lyVuWxHbzVKEk);aYuae(;tbIXSqpFX;$=O>fd&MymetTH=wzsz@&'
'&t~J(v3oh0|2UbqiS6y8V%~JMH~l*$|1V2I(iDQd$~>_0qmnITx18`zkzL^&*d3Ff>6DKWNN-`<A@0TGr^)QH{vifBTEZU4E!lL<nEo{WF#fqc'
'C6Cf1B<Zqa&X=>xi`n?{lRG{>o?Kqm=LMW%K8?Dx`Efp(T%Lb9J83MZ*IQ-)fCx@QM1=+bSy7VWK{*lO@{>U)g@eQo0NaR74meW74#|BMJD9L}'
'9JYv+Yxs(6&C;BNPAN%~9Q?+?eWE|CU~vcBhcmh+ZO$am;uVPMIaGY<=B(eg*|CTs1$ce;Ue>}|9ei(*I^L3ccOihH)WP0u2$S100FQbU!(8a3'
'!9#0!m8Y<tpL=(W9;FzKg%~hZSY)9MPBAsNh*=tkh+rL`lK^D8g(L!}5=!;A$ip~V6ldZs^Qcn*2tB^ooWETsDP55WwI&?NQR%Bq2Vpv79?O*e'
'8SEqxpBi>(8L%b<QD2ZcVEt|JK?gpK(4Kh40^((D&Ep6f4<{??AbISBWCOfAtS*884(qLAVX6yGk|p!ngd}1sp<@qta^isf3HEKuhX8Vd1E@HX'
'Z!xwlTG+Pqh^~F0Oly77B#*qM!#Y?J9}j6wJ58-=OfmB)ejM3hlVQf3HGw0FGDkv#7EwmMrKljCU%@6>w%KNu-~ekHA+Q1pkFuoWq~ug3SinLc'
'#iXqh*1G2M7=%_KHknY%s(y_x{-z-uwwou_wU1w@ly@Nwvg(er#Zuggm$ITAN!Ju2NNvdljaLv`Fml6sd<A>{7hsNu!_DD6^RwmQk|G~KAmB1A'
'B3a!gO!;9J=H%|A_T%(qat?PVlVcPCqgAbiB2WP4%dH&#em?)~o=oRBz5Hx9Etxz$1Bc{Cv0H(V9R7bnpsHcm?CYz}A+`$(8Ea}2!DHC`-1)g#'
'Wr%WBh#EG(kA4XUS1n$ArM+X$bEkt&QPrx4k9Z$JIM`jbB2~R^E#`g2GPd~UhW3Lz3|IIX$>OF8FYfig`P0qg(bGTstK_{|3HZ9NHMK8Er|K(B'
'{WY)RS$>of$Jji!@V^Oc!I;~e$u<s)Iu`#6Kpd`VQI+HWK@hKKko#e^ktUte2a@+$3aW#w4KdrzVlmY&K4}#M;%!7~)gQtvak8(|(fiI%r;w50'
'90;gbcR~ICG+4NFPrdTZHVGY>BBvhK&uvuhdw=ggP)h>@6aWAK2mk;8VqLP&f)Q^90090D000#L003cWUt@1=aBp*EUt@1=ZEtRKE^v8`SN~7k'
'HWdFIiT~k=O>#1h+`-sXT}vy<Dz$3Wq0(-e5Iy-4-^FVZd$z-s0ss5nvz<%s=tL1p@$>us<@4OVy}h^M=3rrjfIsCIS&Kq)I8$}48Vg1^BgHkZ'
'U?p7%+Nf*M@Fo|)n*zjFUCCTJz^`bmu-Wd;?#_kaCNH5Dju+f<kQQ>}mUs{#u4N%megutjq)#mXe-}J=Em|?6LTh-R!N*cSrE*^F?p#Y7F0Z9n'
'2?N&2sykY6$e62ID=P|E7+#|v@LpUaBX^FYE5u=gSO2ua-0bc+HSnnrHaAi`NfG0XY!(&K*Uo^=mGRN-?d|REER3pQHe0l=HDWdciMEZyj9^II'
'YoRnmYU%1~eJ~<Qx2CG(Jk#7*(a9stpGkOkrw|L!6_-td*>6QU-Q5BF!Gg4=@mjOCLC3iki4mD?=ZRs@E<ZoN+JDYQzyLqinJAs686Kog(q6l-'
'^sG>;rc%6^q4@+OS0nf$o1(La0iPSOpj<+0`MeUC&9<I5Tvj7cm|3%u7=tr6KFug9w7^f}svF2lk$=IeC?l7$;g#<XQhN$o>6Wrw1bt|o8ikso'
'5xKCI8;J^d13b@#cHVO5E1&_bWaF}+?X3{nTlXOyl*&@7Dd3#S@f1~3H#pN;DwFpbJXbBS$Tu(-Id3WRq>U7P`tT08*4Q)Xplsmu{LQ+dGdMTO'
'sa#bDD2xTaj)qCfVwMEF{`m3yB3oxL{E?Rm$Hc-3TvDF1rEm%Bf@kRnda5T}fn4}nz{v?PS@We}ASzUUxy>02ER+FXZt^%<i)ExA)JEx*RyNjA'
'TZi88ZIWV%{H-T-y)qE?0jS2wrWHLkSUcI2+OXK&aN{gNNW#Wrwyn5R!%FA0oif-Dq@D}gYfHwD3Z`Z~HnGz;xp-Bnm1j$WKxVNgYvEJstuc>e'
'@sfI?;qkiF)N9$Q1I0|4CEHX#wA5?C4lrH|HuQVdh~b^))~=K(y1Vbu=g|R6nvlQ<j*ov%2Vsh@G#~H>MgPheWge{>mL=w8i#3AFAor0HJYB;Q'
'xiwqkr^VFXP^vbO0|&6JoNxC*e`@SviY1DKCib9=cy)W91_E)}Nbug+jb^0w4<%iSAmFEi$jo>0<?a1ddO5l3MQC_Yq#qpd)pY|M4E4NtkE|81'
'=LH8U<TN5N33zED4&3!F66msFVd7Gpg@yvp`7Y2dN-XJnq_YA45Ab_~ShV1^tRy#>K!TEPwT{9DUa#E2D#jwAci1QoW*rwp&0SvNYWSR-olWk}'
'&fw)=$&2a1#cBEi|Ao6B)4TI_jye%tYgChB97E4Xq~y)xP*6)BWuIUogq&=+wFWeL2&gc9Ue;q%1GnK~yb|-8#KTD~m;IsE$L#)lx_lcvAKLO4'
'<G;1=>cdGlg9Eyuzt317xt0G?<dYPh!;{HRhm)gY|Dn#Tv5k7pl`LH8S%>If3b`zuC#UcXj*cgf+Nbq-^2kPi&>eXHsNy)q87;%6NNyS8mvA##'
'uqn&KBXP#|>7hZxc<Cn>FRw5lW>sr)fsAViWur}d%t==vM;NbuoC7%w6n+N}G&c1NZ>TuG4an)X=3y4t6hCW+0;Ny)Kzm%oEDivhZcM_Kh3?L)'
'8&3D9s0Cl2M$X%T+o#4!>sY$ao&wwVq%(YBNcYfxDjA#77d1J)v{A(wBJL93Q;3gg2U-!OW376i4sYJX!)s^)n(bP;lPBFkuOFt!WJxv_U1MQs'
'p1X_KnSMWy$FcI8yK%f4b1gGn>S38n!3$yR$u0YYYw_T8i7!KJiHp~h!*}$$e}&O2Gd{{DkaX&oh;;2dLo4H#++eIPF+!)Bu`MTu2MdnR2`vv$'
'9lq*Pc0aI&tBKcSjOQWk-ELvHd=Lte@V7<!h@l_Ca5Rmikf-hD^krCD6K|(p(>{ZRtc1JK!oO3J4#&O$ZT_vs=F-(%KPYggojS{EPIL0>hzPzw'
'Y)U|JG#U9*I${X-8a?gz+_%(88QHKU?+EDb2#W4DRvj|oZ2+F>;n9=H{m3DwYTZe4_{->clBSzYkQt5-{2AtNhIx0b?Ou3sh5IIntd!gC8WX|<'
'y+_z!-w*J{tA!73L+1U%1)Dqmw=<-C3LUa|*~_TzD)eHm3WNyOHeVVXbqIH9x;Ya6`&5aD_#aS90|XQR000O8003fL>#Mwj`Xm4VfPer16aWAK'
'VQF7uZ*6dIb7fy;WNCD7axQRr)jesG+cuWpQ<eXKakh$*o6SCACoVbkuH!fp?<BF;?PO<5qhe7MBr#2i8XkR2Z~ymw2LJ**)N<@E+f|7r5_ouc'
'csCxPM~@zTpV!-kT!^@?@=aW&iI9u5%8QARSLs4-6Df{gh;3XH@bZejEM%Fkw)iP7OIelpsesn<I{5O7FTeP$l%=TFQplUMtkUgDYye~|<aV*k'
'(`|L~&sx^f7cVc~i|f3&l!b`53z6iTU0&kQfp~o-i(9cM;$`*a7wfnbDU6a|Z^a@lWCBfYpNm*5<1_=zap*G5w<3wRVk@Dm$l-t2MOsP1<2v1L'
'5sjp-0KrF(9)0=6vdA|gik5X%7cz=O3S$=);IYjsm}kB%aVGj{dAm*0+<c$MrTpfJ`T9qhZ_SsyG#?9TK9=jcN;5U=F0R&DI@e>qgReLky^GU`'
'=^&#rS<6kVTfIa|zn4I|vhv0EfWd`aZR8fdyi$Y%tlwqv))#**;@vt;_{+ue2B^8Mx&!R;EKP2O!M4n^D;e?Xr*Wp<cl)J<?t@QY38+uBgW<Z3'
'|5?XbTA5MOZMlO5Y<l?SQ33oRi!Ovsjx*B$Ht{Y3rdiPXeEG#fE=3t%$q^FO7gups%kkvPF9iI-Lcfzm2|U=aatutKCUKVCiZX|xD_Iovjt0mi'
'z+uMC;*uwqNnoKROfev?L8#ScCqi1o;5x0=QCTmS>CMOs;Dt9fZ31GysFF9;2pJ?;)SF#7A_Q^HfKC;j`ydo0a$sC0X&U|*XQdqD!z5n-$A(^A'
'EsvksU<(OUOL)HHh6YKzi{~j&I+f+9U^dZoC?qiXR^0J7{PVILXb`dVdJ~~K1oR*(^SVf68E_-u)FK99L=m0md8VNU77WuU2hl5o6;N$^HS%7*'
'iC(?^<=xwh*U`(1i`O5bpWl4{{^k4s@y27{LS`p#WHyWjzi!6226*@S@2`Kph+e*X6aDRV2W)y1&*R%+xct5!@SDHC{Ql+t3_z}dV}V81gP?VT'
'Uf93=dhzb(mlr=pFJHZSeR0tP|2n-)No5WWU|Kj6F7iY!>H=1!T;5oHS)>E^z1_1SE+9l%Qw7(cKh`6l;Z7FbnC4*?{whxsi5iwa4KMnK4;Rrd'
'Z-03GGtAGK;HECqL^F*ZS%c@7b&(AN{^hreH?N|1FF*X$q8u0k&^HkqBMrD8#_w+~-n{)a`sqiFA`Z8|?QidYR<tmr;PE#3LS_b&drLKel%w7$'
'N2IeX&GZ4*udL$D4h$IRKbfq>T*g)TT<q#?Qq^RHPFC@*guym(x-~k?S2ixm;&f9siU|~Woq;(>>H<`01xji<^Wi^!yM);x4xoBK^A(j`FiO6-'
'l(%6PZ{~|wTu;RH)C07N%A09q@h@E(u*3@3)ie~x`sMWaY&P{kkHf<kBKYPh(lnFXk$OKCFG6wt#D?5p9>lvHbTw_CJ2Po|sHeI)qB3?cEo4y2'
'xB&hwylL>4f1mxE=Zmr`s3|V*$iq*b?*aD&@FLxf#(y5ChBg+6{_c)G<|YlVEYodTuE{8BS?~p$NEUU`=Al>Mrc?#;1~DKxZm^)`Hr}DH1GX&J'
'yoIX<b}SRCLiu~jiA!LdhKJg=w~&C9L^ng*-)Nf`8!+kUKaj!i>n67==w`YAhbfb-=oN2oM`dscR#5yU6dr3ek9f3121UpLdRbA)PEKhr56rlf'
';&-&9uZtotM&1v~D-@tRz}b=+d=m^6#3bE;R4E4*?0lK!K$^IaVh6?){sk3%OJ<)v5L9>uQ)BG1O@yCQY>vbPKT%DC$ALL7#0%4&L<6u-^6i3b'
'm&;wI7qt`LiA`!2Jpf3iC(a2Mf9j#)M*O{*-+40r45VYKNt_wFQRzYxkenoV@*F0&1l<gdbO|02u*3p1WwAlF#jpY3pi#Kh)I&@_Oj-=%vUWg>'
'3w*P?9WA8Xg&&F<9UzC{JBy+0S+k_bMEP>*Su-W*#!{6euR?r<vs=lngpQ5k1qitO^Lzn(P{he4gZWDQN{SX&c9SlyW8@MLpB0Lej$Rgl_#r25'
'*vMifPYMaopc4W#K~vD^V6`g!OJ(7o6nROs4nf_64>`JK6^<Q=EJ%<>Spy3xAT~dKhb>4Q15JO;x6)OVW(*3y)RzY2i#AAR)*}&|x0Gei&|w!K'
'_t9JnEy{Z1$j@<0EcP)q1KXg2#lcKipsm_q7qYTAxG~KPL~|_%Md4sP)G%;AlX5FPk+@!@xd=n_&SVL4ByA@B7>2OvOi-g0YW`511*h%7Ib@1f'
'90!(t^6`~;3D#f<k<c6%UZ4buv=slWa}*3x=td*3m}tKP<8E|NG9VEJIY9#zmBFUo5cwQgg%<{$rnwm*<Yk#oCY$(Xbmo)OK##^JC+d-cJs!`d'
'6ZLuKk`kk;8=pWSIpwzAkf4wBIBg>j6vKgt?ixGtQ55`Eb7G%D#+UA6jPYOwp74S{a?s=CYE0S3m>CD75nse_IypO^ja@h~067g1DVhj!&8dLU'
'2O`ick1L|b^mNuZXm0npAiPIfz<dCBB#?OL!BeIvVmdxKdo~_-=s=U8aOF5g->og;BpWuJv~Oc28%Ai+LvNkumtNmAJmRh>iP8D$f+vo^sf96y'
'z>9G16Ivnuk0d%FS0wbReAhofmS4*v%H*;d8X$&+CGvz`2}Vvvi~M@q4{QgcpyPO($<T|F1cn}h3QSQVNEV++^IOD{15mW{jviGRPL~+00VOw+'
'yLb|uFYk}<=JYSD>ev?2ABud3eiVpxTEQBwVo*-BR@Z9@EPkWj7*VI3n~$yyCVf!3FY<bxfs^N`eso<{?17<j;w3{4T^U7W1fs<W^b$Knit*8~'
'IEn?#E~oP(A%N@6@jE9wfG>ewix{vHS-ONq8(W9uNmB1pQVP?SfP=GY#V-z)P8_qEh!wkFJrWN<_#Fzm7M31v2L}w$uciA2d$WCWWew#_z4NUh'
'j#T&~1D3t>(;nGF@4?KPr`cerUtny7(0ee|O`34SHuMgegavSN{$$oMTlH?2k(5DX9&vcv*bn3R5=>4>C`Dhk%$iohtk=&blXF(vO_(3w=;Tp+'
'3m%#;z8=q7lY^GUt*u&3C^0gRjHY!JEI2$*+oVPXJD^IR_o>gLuY;%E)xrfO#K2OBb6aOV8Pb_9rSvk{DP4vb(*OOo7VOtKtZY%Q<esA)cWFXd'
'Z7{j&BfP%}gVSgJZ5TXx=3l3aY8?j8p7?8-uGUo;oPT|&x~x$r&cTCTrZCi-EUPgDtz_F<fL0(tOV)ZJ7@&UmhwDY3f;)&Ns)*u@aOx#7Wp2wx'
'X<CfDycu9ycoUIwEFE{~?KZm==r~<bXuzxw3jni2FEJ1pZDSxryC6x-`Q+<0ghD!w0PwfAo<+q*t|sfNyEth4+Suq>mpSgm|BAbn)v&9Oah9sZ'
'i_?3HK`Ct=isPq&D~lwEgiKc(R>#x}6wT-?c-kT$nud6_0?)OCNqcM;JLwz4<_%>vi|fq?<r!Rq*(iZEVzBA{Ri1%=7TP)i00jKl56;g)gtc8&'
'c{aToUR$!Q*;h`IrL{Mr;!TQg(J{1QP|VMd##Tzbar`-**iOo5wWR`%C8!h-wKAkQt4Y%Q1lCWe)B*YQYzZ}RaUB=9IfZ!}%toLnk@v0l05}*='
'tJ@|^la?hQ0@?vX__CGg`>gSmQg=MnF~pNSZoOI?Zey_qW2ZS`>AlzKle>q2+-vFMug`TX23;*|?t9WS?O<)do*`eWC25qy0~7Er02Bx^Es26-'
'GD7pZ!puSeI`AsSBq_=?ENy8}KLpvKu#Hgc2~>KQL%$xL$eS(gStTikoyHT>{Y?5si4F4CT^Z=FEv7L!7|d8v^Ly?No+-P)4jeB93C_KV{&4T}'
'hbXU|=f1}e?(O70x11O0VkNsB_<7Uwo-(~dDI*`&XJlE89(0cz5)9Mwmy|OoR(es<VIsUEFoHTJWE0RG=Gc51Tj+QubLAnaTeHw?7ipbicz86h'
'DCkwTGGgQ5Nf}9PW`%q3z~E&9v211Wax-WX5ukN5;YAqk4=$Wt2GVN=jbHO7zEKJRK|R+y#v+Gyl#XPNW87rfRG{Dt<!)y<ZaFw_XU|SsX01gO'
'73Yerln~97Yh2KR^Cmc=kt0qb16s%cdW7lZ+39S&2Z0j`Ac8Yr2N=gD2p~7J?OF~Xw@6ET6c?EL&%u9AlS^=Gs+{l}!U!{oXvC~B{zPJp(9-VQ'
'w%*J+?OKu)220?DILk(b_m2h1#UB5iGitVtKVVJOL<WX4lpcdzx3TQ;X9nm?%4Si6KPdDJ<2hy$RmP0{Kn>re_t~k#0-XCxxVB+Xn;twt)!|bn'
'`KyukkDsPJd}peFwc;nOr+T;OxDD?iGD{3?&b1W9kLeb4gSCqapl)^A{#Yk^pQ{=rO3m-$TePfWo#}tImdT|sc}=DNw0`vo=_IeweM`4XAe>H;'
'5-?c1Ek-_6u@e}fGU_=k2>^PAc9_hsqh%_y1)Il4?N6G|r7De)xMfK@`}lwRA#It%W8OM%Ubs|)cnR~Ns-Om%tM9bs;S7Rawz5jl%=$JA+qe%C'
'^U*=u@xxScgXegCc5@TiP0YvU^}Q=Zij0TY$*wfvvgo(uFu02{RbOZRX<yVcWH&WRw$L#8C@bo1OBZZmF?{2P;B2O&K>}HB@Yo;qQM~Iejt4?;'
'>u=OzECZy8>}b4R)-;VRAET4ev7~~?O|w2lS60vbK{7ABJM48oX^3^FKi&I!6Bi!#4P=4_g^cDH!)0=dR-82z&8GJa@4(rWAd2Nw>xi<(-2{Y#'
'QV(mlnXHGwU@$;JcU3H=XQfNMJn*<CI3NQ6bITvRiFu&oPH&>Z$(BHuNe<Bw?Z=$@;_Ns$fBG1gm;sEBgVS&5JNE{*5Y2}4M1QhMVYo)K6H6}('
'SQx+u+;_S3#baM>JF8l_CXRkf6ULwr>(0bmpvcQN$Czb?k<jt_{pVjMT7y&NsMw`gImXJn()Bha)u<xS*Qy>B)iZ6)6~~maTEPk}0`Z<`3AEl7'
'c`31xvL99<4q1hT&R&zl3SJ>%xr@_<nzMNM?hST_ps?O>hSwy^Ow}!xBnD=(Du&Vsh68m_R$USnvuJb4z1K^wOM0lcsetN&PzusOqS_x~D~%Z{'
'F0$u6^-z;uNmGL2RBD;higpnWl`FglvkR-I3Ko9nNI<QH>R5{5c_EIyG!sX{#@E&iMviu}0Obs;=1qtpeM}r+P6{m3&%}sojhR5q5SoXyrP6kW'
'SiMt?;Z6^DXd#iO7OK67ui`YL&;rfGl|6mWP1A4(j2E8SZU%!1#gsoysRC$}J|0<pj?cy?-;UV_Psx<V+m#$W8|xhctTd_<l~@8D{DZ_;a83o*'
'FbP#FqQ;^G=T;@T0`Wb{_C(x~jFWeGI=c^qXT|m`Y~nvJ-u^1?9+B>OG&v1U{YR`{9!;h*|B<frel&ShS@m=H5b0Ca^FKhWK;mQep8X~)Gy9um'
'y;`vjE#sx6N^!8H&~Sx49<}hItnfn5^?DtXwon5G;zNPnCPce7fhaqIe&I6Dt9S)&Kq+92i<&iej_zm`(4M0r8|GE51p{gjb5m|#lvwUQ&oPY{'
'<CcF}v9kPRf<D41I$c6oqX|f_9=U{OQW6dAIkpMK4m43Y15#2g@~w}h0`f!|E8Ka?>XP<WkWqByiDK!i7-m!5ntGaadqwqzR6&u?Rqh>z4lK6U'
's>$JZw-`+cs47O&YZ)=CHr>F-y@r53Y_}L$v)1NHjgQM_D>+?A&zbz@g<5hH!8@<I-AQz+fl}yc<3AH?i2K%-Ck==UrG$6#tnE^1hEh&K{|HMe'
'I0c^wp-erL9&biFE3lH|!vLlmU}h}L1*pyE$(=|4c#F7-C-nT}kDXjS=Tj2Ccf=op`8Q7}r!7Z#;$fsP!8&%Vl0ZR?=C>6Pd^`?V1&+q|_l{C7'
'cen@BRF_a%vKnb?<kL}%@Xlj>KIyYc5G={E1SG-nfz>GT45wBDcVXDUGxsLy!~N`GpSkk~6bG$TkfY~KPGP|KCaUsFiQ&w*r>FNmQa`V&aCG+6'
'KYQlv2<tfpWE&#j0L;}_bvYXMu&QZ|^B#?-Ub05tyYyzXxv6Ff-^|Q(HSpeoO)_k{FpxXYxdj-}c@;#vxPaa?IwRHgLoNG>rG36}E7R@h%xCA4'
'w`6Gq_KUd<c<bX%(BiT7@+MYu^~j%Mm7Awkmp8#W9IKgmL3O{NHJ_r^WCxW__y}}U*%gn-fgG`VHkDrcH-M;%Z|&#QJKSTkT-f&nTf}%!X+axM'
'DA<EXn=pb*=tPoGr#{SSCRq&E?7C~I=94u0U|1ZIgL+ew;vF(}Kl%ZhB(egj%4W^;X6tnz`P>sH?XBI6uj8~*<QtV}LPO62d%H!SBW-pnbQy1`'
'daCnUL6t|<8Wqc^jq>1HwieMUWz}u)@Q)fW;OFKmH_+QBpHfE%+F@6tBSR(KJ*E6;EMu-3WKZqPOex^b5v8k+&YV0yK*P$+_OY!8wl*1IgQyxi'
'RR3CvX@x8ub`N=t@RLPzy(z|t4aPLIa~e+MfC7M!gZ#2LKCn4>L=BrAUDi!Dz&_?OIeX8-jZa4npk=ZQ!#vj~c?T(!k1{!~@=;4-Yau4R;Zbs@'
'dL}aUlx#$phDB1$Y))4Xt=a~ethX4p?xBEtvd^cJj-zHkK0u}%07f)0)i<q$HiWWOm*KBa^A}Ov?8G-iwca?jlPYBpe(Z+-VbGga>LjE-oHV0^'
'uOXv1X-Mi+Bw+4FBj)J@hm`_VszQ5fdc3hK_2xZMxmEGNc#_3-6;H=F)?1~=8ruAk&)cSMzeO6w3B~M@76xZcDcQ{ace1XxmvVtgQA|-+${{~U'
'J{&*A@3n$jF|(ZeBC2=bh%6!|fom8wkszRhWg|ZB&#C<P+xLHqet7d9lkzyB2*?A53OD%B9f5CC5B?XW3&SM{G}v5%`5&RIZ9(Udt3;KYQGUs1'
'BDifM*6Wnw!j@@B@Vr>qe;Rb4{$L;zV^Wp~a@smrH|r!Z939*w!ZV#BX>u0e0IBT|D^W6tT&Fg~+Ct)nOIamrwTKU()dhwqfNcX4i)VR-4J%w?'
'zQ@9>T}#k~M%%!Dz7Pev_wJJ%va>OE4kgi=U2-<(61$Jw6TiMbbL*Q^vVL=_)wd3yj=QIa29~<)mdD^$Iry?0H1CEPxHQH(Fx;>KY68r86j;l<'
'hw_c{$aw<HcY_D%dzel2?un2eS3ogtu{$1!J($TYbf`Aq9Uht`fUy<;Iw$L78I7Fj9r6bdRPvo;1>7P{nu;3Q5VQ;Ea2R4N+IiT^ld8mzh-c+;'
')OKQ<-|^{TfZe7x=w@Tjmf_AGO9yysW2Fytvjcf>fmI)v-kr)Jh4UvxzDpHnV|Eat(~clvasbxKIm~Tu-nOXNyrbSp?VM=%N_@amYZ8kvN?A}d'
';<!9Ros9=1(1*ZOR=(tm68J!ZPNHs_y%2`?UwG38Zmz&2(toj#Y`|0C@3PbYz`2O)3{OAPzN?_aQdnP`mNqyTvXG$EB#ElI)S<$(ETV(ED0pB`'
'RSjjgs8QDGJmno#oReLjYj3v?(%iH$D|c^XQP~bt;d{v%*S|uY<H~9PIBd$%81>l|DThm~n9{~w!{Z4W?4jz%R2k@=Y|BjP6rikEu~n32h8pj+'
'zeZfTar^<Yb%0GaeOl(Y1v0cMHu|DCb&S}j5^`CVVQ~7yw_TwSmAe9<GVdmMaePwa-J)-%j~db)Hv3WxjTg&M^hj95MWy`Ku{h^^u>YyYf6eOA'
'q0f+qe1Pz;H@oUqli?a<JHOsXT<w>W7Pe`8re+&ye8M68c+=cDI8RMEJ-3caJ}v2Nf`3KlY;`tFIgUo~K?~7QWUN8r!-@KAulW5fX(il-f+%8('
'2{x*8vBY2FGqj>Tp@<4ucuA?Enm(CR;_-G>1G#Lz=_96c8{=)9-C`okBIo(Jk9b)*5mgPRLo2nJH-vO6R#FICr~+pG**Q<as+er;Gqm>)^_GD+'
'QryMwtK!ybX>HQ%!O$rLuyP=}5<nAERRpMV04oTRH;LR;;x+w)W5syvE%{8_%|p|1a@nH28BEa~ATgd!&N;2D@UVbPEeN)AspLhWXzz4UlE8sr'
'lGwA}EhrryHbv`l6L0Z!z7lk`8CGx2uJUu0c&5j6bC;2HQurHL#kjQ-fa$PM(K&URK4sKDHt)%*($0E*CElq+(xTkqh6$h+XZflIB-t~xgM|+t'
'E<}#ov&aFU7q-W$0+6SwLDD(N=5jO83u&pj!-TnwwbULB7)>nSCm#-v)+ur?mq@BBYY(db3WFhgx;O8gE?Z8+)bdc;+kSmI{SCP|PDs)+sZBw('
'25ehZXNnz^qj|kp$>vsz&^258TQ_nI+QWMOIC%23cXYRzMeo*&<HIrT8>YF;h~p;VE$b}voWhN!+RKU#xk9Yxus_A4YQu(VGb;%(>0aNd@iCo>'
'F*}G=k*>>hT%-uovyJ_MbCX$f)~h@IVes1B4b>WoJ$M}gABP$m!=cCULic%_UkT+RgLVL7ff*;qK&{LPu8Y=Icu$VnRr(i~gfw>-vQh~Sqc5T`'
'EK2=RMY~r=hyT^wSJL#tLVt>s0d-Bv24}4{_vF<~gibk=-sN_i3QynHMS6+^!M(?8#^(b^)<tC0R^=1ya7Uf6w^R>0uLC28_SI(lQ0-<Qgl8^$'
'Q52@j0?`RN@w6LByXT&N=c!<y4HK21Lj=v)eviEqZ=$@E9&7DQtS?l;YyuB&g7oq|M$Vg<kf}+--y0kFOT0S)64hX>vm6<a?|j3{%=qReAO`sE'
'7%dT_(p8z3U1VQgF)V-S-p4^D>%b}zH*htixoQzFro;#2jM8C{Y8@mp%|;{RQI1a-gdM?R>lYoCi*KUVun)wzOYefc7S5~P5#W2FfvCxqcpxZ2'
'Peh8oF`9ZyPz1^iqCHd1dS*A?4o=K{BFz>n-$O!3IBC(?$Ft)^j3esum`C=}JmP%-{Axt+j?Tu99fA>4)^kjX!svas{KVgd%|$=o&8-3<eSRdg'
'M8R@LIOquPJSW=pCcxZQkm*h4c9qSZ&G##5i0gzb&yIq#^T#y36PrkWYbioafo3$`Sp(y?H>h!Tb4F^`W8QvO5jRoLcgnLIWr~HlpiF5b>wUBK'
'THPX0x2&>hGk$SqU(tj$H8-lPOCN^vCezXpZ0o?Jdu(b3;X{>HSbww&=)*m=-BI$$jA<=5YXyv69$aw>o|>!{eLeF2vAxiOkmQ(4137JO_SNTE'
'N@*3S7#TBGW+P)HDaHfV#ioiFSXXmS<%plS$jw}xb!Kh+QOn#4k<_(h4S}bpv*AdcZ^?^Hh-CBq6V1q~(Vtxxab%?=|FQjnFXoH8J1U&P;3NHO'
'Dq7c9wwa5@#NQwU5e}b0u$Qb+fi2vMO=>w%tz%uBW_ma0o%HefMS}sA{<I20b1a(&zDQ89<H=2*ua`Q(G;Lm2CkfV!X5h1&U~(bDJz}~m#%~{_'
'*5!U&z)MAE;^N1@y927SQFSSmV1-Uvoa#vRrB%qKfp}$E=Sho`F-hny$%Q>lY~H#zKvkIl@1NPc?`B7sOK?B?<0PTsE?>BcI3!(Vi-#K`_FEJC'
'(n2x}j(KtJGu(GUz^qjaYfsT_ih)9>e@7s7-vfpEV{$<D<&FDOJ{{j%3F8j<&`p}~MTq7$O|1FMkmKkuq{#)HePa$n&SmLdmc+N0M|=)a>Z7un'
'b@Ga8QuGv>i#Q`-ZWh_{=*;p}^wY*)AsDu!!lz<FsyH}Vdh|8AGd(A#=ZkyqIX6=k34KKfL%OmlRF8Nu9c6ib1d;w@GuR1X=Uek4Qii6%xWgg#'
'1*t~M`fcN2_j@oG)qv@T&^JweUrPnL8#JNk^8>uP2lp7vF*x^L7rS^&7ZPp2kS;0r`H)6v)vB~+&&6LYehORPKf)=(%W$L#H{y(rG*$8#>%axW'
'Bc7AAg}|p2_%_?tEn_`&Fn3urcZ#sf*p2?$^%dh*5@4Y6gVs&staZ<xB%S_S;(YZoV#1e34fqZ!oGyJw@lv;x3=G+xFt~?tbLG!qGgQrLFqP|Z'
'5Sq1-EtGBC1GDESB{fj@z6q}4rx@LS9tw4<l#0^%%fM_SRTD)#c;YqBRjJ9oZV%`f{M+P(G#7-?m(L(H1`FTC=OaYu5HZ5Px`h7e!r~`hqI7Wx'
'OC@OC4g+8jisBZxsp2S&O=}$Prl-LmKLR*%poM9B1=V_K@VF1IE!M^W*RALxMtCvxG<a{eX8_FcZNN_jKW_(KDu(x%6TV)f>)9mvdux+0d7_Jb'
'Ew!c3LV?)szlLijj>I&xnq<vF&RpYVou5XyWy`U$-YN$zo|oXH#no{&{x10XX>*I0!fxzDg>+b~r+&3%CX^VpZJe=WJr^z(66r=Yy4J;u>>w1Y'
'ET7{Rkm??BW6q}a+^4RZX+9v28w)P6LeY%E=YZ%}Gde$lVVh7&<0+lf_#Gybhx3K7x_!Hs@9CesFj~aZzZIbNBb)i)BlXb^X7`rn<Dc3zV0cqJ'
'i9PE(0f7f?lS=!geYgF#^^1n6j&j>o#;q6^|J>kX5PMdheWza7Bih45%I`2Wsj3Lr9rR&G-J)BMx65SeX--V#dhZq_^0xc0-LU^u6>W3Wjhs+#'
'V!Tja7k3lU`Am^=?Vq-?9QL1;Wma3xdL-sh^=+VcCklD5f8mIcQK17~Xi~-L)?5-tAvrCI8-<*^*Rj}tJ=Yl%VQuCOdAEDGm1al_UqthSPY-ZZ'
'!YH?*qXEPYS`}An6<>RkY44i+`SoCX#KRt);XR2IxO^N#(yl$#dj7er-H+*wO3Wz+Q1?QpI|?YBr+%zRC1UJ^2G4kmfZpNfye=O|Wb~2n&n59_'
'-o1GXza0Q~ysZQBe){haR}BC9@ATDttVCU%Cq%}d<~`>&&@j?AuVT8EYu*<#wqUd{D`Q-|eP@YURXuzFfk&s4p6*c^`;ApMrp%?n!GAVB#9*k*'
'<Dum|jk!>h@*QUW9LC{eDtfH$ZHv%3NN}jp!M_7El#5^2e6*9~?Y~bmA^10JMw|l~pV6A&-+)EIzX9!NSghx|4O3v%jQ@30U$u9(zqd}ESn0b5'
'C!{q3qeiNLKf29Uc}`NBWuqQNF||&2%`8W-zio5KOSf&^tupL4jyo!;HJz_O#8ll+=Wa9lB^-{1K*F#1*KqKL_AI`^(245p=~QL!-02@W>i|U1'
'=TFqlg(a384PJ}oG1^pg_%KC`)BDg|KS5Kpr^FqdvOVj0>#RDu2En(znu`vw0?;9+Vc3mQgmdL}p#pkJO5=XaY4GhS{xpp&*Lk++SQo^QlAt^c'
'V~o-3sDC2qvUlc`3_9&n+mLIdRV=AjjBsh>7o>X4xPAAisoeOTz8!%6@rR>0jzYlM%eq^wb`-wns9;YdXnaT=Bzfk5DPEjKKd#jc7h8(o(oCP?'
'#;OnZ?U47Q3_{Cy{jS$_ZFaFOm+{UHgQs_oF>k-|@t}&{HKl26^gU+gDTczJQlHN5og9yWhMV3kKM$6h(+X|>@1TR89TKYFT8jpl=icx6#m%nr'
'{}gYX%&b&9L``<ZlpBH9&bK+J->)@0KvbcL8>R($@-hbpO5N-a5qr*;8`1T;YOB^d=%P|w&5xOxt=XoYHy|?*r^W$L_xN*O-M)#xxr7ItxLc9x'
'YyPoJ0Jto<6breds!|-vT$`ydpj=ka6!Yq>S9IcmCeQU@<DRfZaU<=V^*BiVKI;Gb(Bz~1;I@|iOi7brZ}Ic^hn8R1n#2PKRq#eM+PExXnaU;p'
'MoRNzeJ9OD`dm7lRL%X+J!-PG*RNjjowM5W(8(QrI)ZMO4g0R&@RbkZ{@N6qJFUuRzH<e~LSl)gK2)G~VimaPPJN)A*kH4JVr`$OA7(`rbgp7Q'
'H}3N-(C&>wbw)chz!IvX-}DP{;q0FM_hag^!FrzI9;}*N|8Hih(051<Tj%Vpv3+wyUJc8~<b8e7h<#%!E=qIf0gh`9)+Q3qE}%Pszb?Ma@@u>?'
'cYS(xVvYeE^L+l}+|ynLFB*NGp<T8wC@J-8^{vORP}>#Sna~}SlQFYg3H{3QE%3G`-!Osgbab2c4BhgaRVU5Oc$%Aw@Ra+ZUegzRYwiKXN}LgL'
'Nr06;m-4n8nR{@0jB#&;tF^;k<oLtc1Jj-ATG*N2@@Bk=`p`3C`-+kYu>pWnE^O;H-3Zl%(m7n+IVRg_(F?`B^kXv3^_dv=g@1Q%1ucZJj$bX8'
'&bpc+QY@_eI`wVfI9XMfp{LEL=c{~E^T1SKoCP{B6cP@U&kq&wK37Yxy>2sYSY@V>YE;Mrvlye9yu>;>(;CJ3{{T=+0|XQR000O8003fLJeJax'
'HEaL?&z=DQ6aWAKVQF7uZ*6dIb7fy;Zf9w3WiD`e?Y(Jt+sL&j`aNsqKOo4>C24~cCCiS(4m(G-)rnp_9n0P69JVe4k)VW#Bp3kH&^r9@Z%<XI'
'Dilb`>CVl0>oIAI1Zu8bd#XKzVfZ$i#PjYf&5JTPIteD(VwDwXnP$r%Sze^er1#5T{PGu{FViAeWYhIL38u+BJx}tuOy+k%k*s3;S_I`~o+MCz'
'IZ5(mF~C|+g6zCV^6MCz8w9i2D$mZ7C@A8^YMxAkSspKvA`0T=G`Pv~X?LDnC-b1p<7F|)(-l_x3Y(dxc{1Tf;-FZ>^La4GapRDn0yGPQhB3rJ'
'dAEXwUstpEIFGrF<TjbCxsB484Nf?n)A*C%I-aLfn9Vd;LA9XFU}ypKmfdvcX#vlF`HLX<=9|STxd`I?;yS>mvOr1j_U*d>8eYOA^57!9PL@<X'
'KTlysIjq%X0s}9zY)&)gaxk$3iWb2t$rou+zyN>w3+VVJPs;@BOLwz$UShL(yt<V2nBPIyVHp1M7qdKD1mp2+U9R(FJPuNreqKWDWmeMS7dS@s'
'w76SN(oFxpnCP#Uad8QotAD4fcsk9KqR>zNUf}rT$2_|L2)WR|7jb!6{YV!H?nYHNtG?!m{#snZuFR{S+2krItDoz0*xiXagyOFHUM9EY4J=9Z'
'0D6`6SixvuGHNkC(%|R~3=4*>UK}m&qTn^`a(q5lb=UbEs`dc3i$s-!XMfMqWrSbo1+FQTj?*!pn`B%}E|W#9s<aVa0ytL;kjbayAM2zjBj+)~'
'sCrfuNof~*6&J~Aa<NF3>WKs1^7M}g8tKG%@+m2@`E@dV8{Z|lJUgA;9$!~YeV3(^Bm+Q{YMuI&(p*jFc$$_}E||c<&ExSb%g2*>x{A~>8YfU>'
'jEAS*&MKRylRMQG{~E*QE><O!S>jek02nl6YA4IncrMlI&HZByhqcu0rORRkXQCQFq#xzUa+>70TL`dlh=vMsdlyOhmY%i4s_C$UWxqRlbNpd^'
'{Qc+S_opWx-k(CP`>?lI9fna5_HL5%1$|j$*ZlWN{vPt*_&TM(Wx<aR;LGNhzc_h!^vB~`tAAfzP^rJeNvAKX<%Q~IMPDw`8T~z9tgxA*Z{M8M'
'2X_<87*@|@EHV9s8Pea`Jf862;4idt`uW4hzkK-ghN?3>gz<T{F0oKN1AIeYF5+}aU*<(df0tAGlPCD+JekDnLcS(T{+r_XrN#54ypb1^Y(1yC'
'xS9C(GQOso>9VAz7a2FY$O;-nww&ML-|{jpb%TJZ)103s3;D~{;7cK2(t`hH>Or2ZFD~)x8leHd0My^<MsKJG{L4eVQElG9nYxrObAFPR{6Nj-'
'E?cYd(B!gQHw&+5p4kmIamNkJ=QL{uj<jsaVB@7UqyAEHX`S}vm%rE`aQ^ZalX+Yef(er2JkRp>r}YxiAN}YIcmS)oD7eBjnFZrzcGK<z-LHe9'
'%%w!00MS?y;_SiG8GdPpzxiwTH;e9Xrk{WFr@?RD4SsX_UoZqJbFrWjosQA$BEB6LFw99ZKEDHIfct<=wflf#?PmwUlVHDpa43K9n3nCVz@wNf'
'uiN3#$@uk$cOO5T9*^H0eLp_^@a?D9$KzLj{d{~1Lx2&s_g_feCr_SrI@Un)^|B48cmW_fAmTiN3THfWSib>w76pLmllcJK;<eB;ribe}hk;B#'
'@&g%vncvAb{17Hu0HTS<h^}zjg+fCxfHA+4Vc^hIMJT<VmvGo>rPQGHNM(>6uAJOXl2sX;P#?6v15c~QJdAa~HJd_<L*w<wlaI&rJjwI=a~Z-='
'4+Uu2<ZAd0oc}}yl+L8v9&hag_^{wt!=P`BYnqgCI>*^^N$##UO|Wn~T$i)%OW4r_r&A2WJOQjd0pvUC9`^hF!I?3`I0Z6n!TKz`rv=&sGw39N'
'-M;7kM&11vSbGz8EZE~g7@(qfz42;&4ish#RJ{%VRbcTxTPFGx0lA;TR21oNBOJny4j!3wMW;d5h0FGc5l8}HzyRg|IbW6~VRHNnlpx4*dMBj;'
'O_fWase`SP4nF`o40yc3pFV&7*gow9-+p=vjBgIS;sk*tnW~9nn=oXjhaNDqK<+xdoJSt^!fNbPg@#M)^`&58{i8?Urr821!e+-{->O>G5OK!t'
'FzvG?{3aYIAW6UhFGh!FqyAX{I5ohZI67t#5n*8IbUDk~*iEFS%FJdS93DQ4NFW$WjnfaW|2#hZ{OS1UU1wvREY)P35;y{njBmD2a2CEs0zi`z'
'wgvt!GGyD4i=)*9xWqpIjf^&+Rc;g~B$LHC>3JJ{rYl$k&4TyR)mXl^<!{F*JA;zFv;c-G1$isM`SXK$maWd?$(8XWU0#h@e$vmNIS|wU#`Tj0'
'9G7$w7iINmO$=2!OX0j#T{rFj$7&?YY}p0+TOx-oi_-`b`hXVkEE$8?lOWjYqZt8SnZ+=+aj~Aw(p!26>jbP3z@$UlxCp#O=D)2EFCAPI58(Jx'
'33Vv+L6C}adTXrUe+M-_?!N<<Co`5vkHa3jg`m`<YW|n;!Sfew9Or;0;7o3sUL<tFMQZCw1(^kMDI^2+vQ@Hdhxs|I4!|`uki-j<oWTZ7F4xN|'
'oGP5LygiQ>=hJu~MFGXf)3!mfJ5g{Rh8+h4csRZF3Xy7?8niL66u3-o`CD5yo8-}eraaAVmh&v0j_H}T=S7(=MUQ}M5d}{o<R_TlAi54)fG8Ul'
'1pl}w+(t7>-UVcQ+eBg?gainm-=f%`gU|+I`IOev!UJknMGXl|);S0#B@-5YG*1^PKG1W+^W$N#xHPXFZ8Z&~cZ3gK&r@i%t?2mv%YLsP!MIS7'
'0Vkcc2Za!HoaTfoI506rc*x`BMbduZZnl?t%4|AzQDcbQO^?B<9scq7Gf)?)6$O_7W(jZt_u;pIi@HY_@H89*;pw0H``x3HuB`jsey<;HXxM<X'
'R@t&hoWaQ%R&SxdxLg-w;!%*eJ?rmB!L$AW{__m~`7QqQa^q=+b_6gb>BELqlQLGMLG4f#2s{1;U@VA8RI!F`ngHyR2})}7@XSM_!CN&p9#GQ='
'EI30YMq97C8dXzG5X{r%I(hgCe?1I<e`}bemy9|yL}UTS2>F}f-q*#(H%(wG=W$L<ntzu_#jq^9Ku<0}e)BW|Gt#Y~f(``}^`H}c%}4X$6SL=q'
'_$8ULnh}OFOE1>g51wXj*f*jwD`VuK`@Z>VjEHI~JXeb=0FE~QD2AW&bpqV=EeJv5?23MRdZ2S@w#CHaMu-KVkJY+q&U~^h@}ZYLMnWRE$|N0I'
'fS)J(bQr)6S@gsAfxY7L+qXr1exB`7A6}&%scaiZ&VVfx(KDRSSq4add8f~K5s=0~lMhr`(qJtvNt4Vkp_s$q`Mx8MKZtJTte`CZInt?G`3b4#'
'HoEmdnx#OP5D5>E5Qx2_3V+&9A4S5Px=a=M*g6!{N4K3aiq^LefYJlO?-=iGG2x#ji(gC*cay=Nn*2SU?A6Jgo+Ay8G~T>$ZKLWxL40-*o$AEy'
'mbktgqj?C;KS^M|gs}KY2~iSRz+JRNV2m1RAPLPJ&4UP#L3o<B=I7ZW_1Q{nekLNL??qW{AfiZHBodZTmD&txjZ*t&8Hl<`Sz@chz)<qOJAQL='
'G=BTx_0e0U;eB)R33W}OR;g)WoD?iZQxDNQv3?fd2-b=&Wuc8^iTa1>R{^el!w5Rn$d%f_?7GuSZdY(j*9Gg)SnY$_z;sfYM}U_`HDN5&gIh2r'
'j=C2-c^#!S<n~x^Gl%1Kos7$j2I?t-DwK9-$D$b{3>NF649)>a@zNP-Ly-cB*uG$>GvOW)JGOBX$3^-hhyy<1^+7u*OQ;UXwtE3v(ZCgJ%7rqS'
'Cl`6V&@59Mr#DWgO&pWU3e}YcWp<S;ZOHz7{Py^dpN`&*Uw!yI{`}$3$M3g5HPzp-81)7fJXa_~Ex?1+0f+rw-!JtWtWSIc9J~_Pk%Y}yb3NGG'
'i&tq+tz|FEFZRx}@}6e0*@Keb@GTjdg)ER0l0^nKNWN`6%+^-wx{B{`yG-JWW(Z~xYMOv|sERY;Wwk$y=YG@SC%ULZg;^VKRL!nJeVTNo2T<Y9'
'vNpjqRmx!kOF2$v*SyNE+H1>FV+-bb-|K;GLJNXxy0v&h47Nhu!rBh%wh4rBj00fcbK)MURL?~HF8OJs7oN(p5XQ)Hn-Tl&1%i}FvD-8N6nKo='
'1Mhm)tGG*rb(6W#y2cd>Mb!GJW0R)3{?3YBcRlYjp<#jti}h*+$8qW)JEQH2V3VB{-!Ka8F*bAxgl0q-?`U~<CZiDBEWAY|dz>C!d&6nr=b??a'
'<mrwXs5TApt{S}DS<;Pd)yzoSh=pCUWuWUA{L(EhK?nkDl&47{x9p3!98T9cO+G?uGY}yvT~3qR2ql}z7%zaLn{)~i@MV%-T$WKjCd0Ea+O0_;'
'ykdyhEKMF?#>*w_5_O%C?FHf-xq;?2?VK4x);<Vc)vu7Zj@M3}d+)p8l*F(M*mjSk+;(+a3dq{IE$x^xhfEx5&!8qq4bYUJI9eJL>8Zku(U1UG'
'qI>(hc)sRsi-OOvVDgoQ_5YlHc>jim!IlLN6YkaKYkgk2B&;Any}-*m7;sIf=N-iBziK}6j0ZoGi{WcvN-`+6Om53|o9+g!OG0S}FUH`J62lZk'
'KwHCM5K<#yhgM2Z@_L&7)6G&NR<kJPsSW!*$XSZ8&;WKU<tZZok$Kkz-J<dIc{mPAxHsTXZ~Q>_Z-OZv8fMDE0gIknfCvi<0GaBAkPesBr(tn%'
'e(WjF$FN>Oo_lIgA<|S#pqf#oI=!J)VYVvlg65gGi^8&wdA|n1d6v!F^usDbPjvV6tFik{g@plv@_15m>lPkHEE4BHW$ENbdTXi&yLi#rjI2WI'
'=o%l_ur)s1d0weO53a7GXe^0%@s1HQBFR~J&r|9>n{DWE$xrpv^o!I+qJ-+n-l@yhY5V-0us+&y^y&9xK(=hF$zojWzaSP98!<_%iH_0K2<ozR'
'kv@4(f|s4(DdOt-{;KW5!%|^vwU^2!8OSQh+{x74P-ZT6?CU&UUaaGbgm10+!m`G3k=>F_r4^YDL)0vi!XQt`1~kj>TBt$I=47`#i;FVJy8sTw'
'D#_8R`U?I4;yTaJ?rqAdk)lUt!`LF)T^9pyp8#D7*-|YKf<fe%7JNI|eg><%NSE#Xm;G%<T%SK3y+3{Z>Ez?*lMnC5umAMz`#+CQk6!~#Ii(#w'
'=!a~?Wny2&0a~WuWZ%f;SP9Zx5GO@KLRjgAv)K#<f2|zulRCV;&x1$X<=!Mo=NejIP?0@xQ*PxK1`)=kuu50kA9SU2TjFq<K(f%=cLa=c!0%Vp'
'BOgk!VGqEY*#YS>a=M9ljg66rLTSB3yO1{7pf4{x8oef-jgUfdSM=Dz9;p)^ShCDyBe&}C$f*?+OW(QsPIWdA{fBhsyKH)&ELP=Rc;;xlaqoVo'
'PPns_PvD5tLAMp*bb|Cz`HP$p3v@!i-W<IQlm(d$0R(C`2PX@PW)owyYHlZJ9|wy%^ssRVnb-!xrO~1T2E=|0iFI$|dsnrNhEgHze|V-D!iquR'
'HxV&@xGoOBFsS&NOqNq`g9(`cKcYE4k@0E;HK`q&Wuez~%E=BQ4yLUk)e3#miB1>mg}UB4eCfr^sOd%}$aZAQ9K$rt4Lrjm3Y`KE){IX83yXC}'
'`;gJ6jJOF4L8pYJ(1OE$hP65s>O}Pp0^$f}M%abB8V36$Z3b-v8$prh9XKT^wAAK;-C)1NoNk37=}qU%nCxy!@y{_P3=P~>g}GUr;~AD9q~QpY'
'<$6I*NTk=YP_yWPs_okYMudOeQ&WTf;dDbr_r?u~C6?jGgXZv4<9{xJ%N(DhxX6O)Ji8!2CP#c#T#|iO^qjWxN3!<U6?=3|B55&+=gIK!Wq;5;'
'pvV8<GKbOxX;@suiJ#x~JrB_Y7wrI-C8?Bo0X=My`qc0D$lQ$J9!<dUI*XpzDq+uU3C~ZVhx-s$1vu>-d6u?bq9}NVt3e91K{Y>9pD6O|G5tYU'
'ugR+TVK)$9C-Nm=<+fqRDU+3{1ETm%?_dT%|C=H-;1AFC#71BWWc8KFWS+#!F{$a=*C<^Z_U5kbeV+7+^?5rFNB<}8{@CySu66d5w8l|DRh4<7'
'QX+?~(8HRpN~ShjIJz2s1OnObJA=YT(4Z3iQSi<b7rMgy*g9*pwMyF(keMFhb-=D__5xzBqk#<X%vi{)<c`{GV+m2!JF|FMVRAJ0mwC|A6^0vt'
'iKWoWAim*y#@-CD0<apm0Y}~cf-N|czql3Mv-^VxPbT)Scfx=ubqQc?IW%rFdt3^S1b-9L(#i#=_DaTYgDyW*Cxyz}{md1-6&*CiQY+QSRm$>n'
'6=T_H&By6-%%or|P}xFHg1#5@eHVe1zDLCWn2eAHBIY~rXPxi(n){`NbIaKb&+l<eCQRMHhG#e9Vv^<m%+%<cGBr--$)qF`0PosKQ`Ck{1y#*D'
'9fZ*v_e@0GEg-`qR?XQsVjXKWRR?EoV+?Lsl)t4*^-=-snNrU$3-q<)t8p&VF*y^M6mSqULz&!ay_}S5B^(v&Me}HtD?+CS%@lZh)ozD>z?Z+m'
'e+FU0d^AtBoB;`50##)w#;H{RF$z%;8K0-<Ef}8pn<9N4$UQIvXvY|)Sr>>;oBW3u0u*CZ^)C~x4G5QtYzpcxh*5LLPyAZH_uJ<-l^|+}EaP)x'
'26<1A|2Eh92~<3M4iBqlqu|gW7c;<g=np_nlH2ZqNOl`I*9Flwg#jH3$2nDb(6IC8u+hO&jYRsDe$;I`9dwql^=%G5t7L5iMKW`y>Z3~Wd8YVX'
'rSe`*rKMWQNZ<cJA+7BIaH*A4>Wb?)CAVQ==|O(?odx&`z~qddA6664ZB^KzYX0{_)yPwr%z^1zz@EVb^oFN!+vGEcf_mf`3~-pBV;E2YkXnc8'
'@}IF{sNmYUqbe@RB;vO$U8q?Q!&r(gJ;$rQ%M36tgYf=-&oC&O0HUT_K?ObKdNof*f?CZ#mOw#O*Ljhl7)BGRp0n+Wy#2kbJgm=ihu-O~$JOrB'
'3P{zb^aN)Z$vSsB4J~Ne+vd8F-FW09E~+1=Y^T<VjD~^f0HK*y9nbXh=SJaWl1z!zcokRzDSK4e>ICS~71e`o0PD4@Sw#WxSwnXG08Dd(%=gXc'
'fR4m<k{5}q4boKWv3qGUi##4z*D*tjN8%kv`)3IH89Ml)7noGVuV1Uj`|KP>KP>9tku%ggk0Qe89Z|Jz1v}#9_O$0{=b8z+**UC7weF(<H-1)M'
'*&c3E+qW(KGjmVZ%O%@a_07{};ss~*;ni4S1o%o&OVDaApKIulpwA!oBk<K#UDp<{D(uy1dR7iUG19QPaYC!XsR6@5)opj&)#drBt!>7I#_TJ6'
'SamLIF{FPh94Q8fV<Eetdv?_}z~MC}(yhks0TUi^4NSfWp=*?;H4BW#su^g)Z{T@Tn{Fzzk&PkYt?B{hVm{$+tgBHOxGc<UJ)e(7mh_6KQ9!~c'
'msV09vd>!LQ$*%GPl0iHiXthhw^hSY@Iu=?0TCK2b8SIMW+)94389{XSdJc<R3td)IKLZTrv+?Vfo{xUHk*YcSg0YO4IB3|$6y)-y;K7=b`VPE'
'cc%2L%8Ulj%u{QRSOKjK3FZVmiYV%8NdJ12ISeNV6$icN27Bc0bvM{Y8OhUvf-0v5w{?h(x@{qydggD8e7kX_hv5pCMhsH}@{xRX6-#uU6mJ51'
'1bA=^=zWYP#CSJRW>C>r={#8`vlRMx7hmx$nN;DpL@TnHn*cwZkdYjWKVY|WEP};XdAh(*J<B-HNeR~TbyKm{!+!6@fp6`ug`>}tSuq@idgRqX'
'kW=W>&Y=P37*ND`f{|IEda(j*ndIXcI>=)jjhM7sMXk00VYXVp-KwvH{R4twBz6i$QKo+8preNXDt>kVD}ESY>nG#c;Ki8_X7D3mjJLTy=p||A'
'WmDy?5d6EQ_Fla7LNI;|C_c~15@VS73*HX#u=(HJ!Mo?r8eWpo#Bjf{GTTT%3x~gLDB(Le$t?<du}_JsKKke!O{YN|ke;inG{9UtgM)z=d^ho('
'aNbu*vQkG!V3dGL!Dj$CH3j*_dy4R5ouub4n)cB-O^bB80thM}c%1N4J5r5D2tS?Lsp1EhS#~A<#r)ucV=gI@V}xLGD2^stmgt5o#e@|WOj|t)'
'Ra6eI=4si^!}cG3KOFUb^@p?04+ScSb)y|^k0dRa93O>t8&<s`_l){k-8;!@x-HvK*_bsoG#czb=la|()veY;W=l0l{m+DUk>Og8)oN)H>);yo'
'z8T9H3*3NYt^f{(Ar7Dk<`|_kpU+0DvEOUNG`Mg>>~uHc0OHj&vUmy92m6B4JI7lkv@LC(#@##_h8Q|wSvD1_@qWgM*|tXRi8HDRDSP0#QD3!K'
'B|wLyW}iT$Mwy1e%U*vQ^l&_tB&Uc`EUy5i4qZnz;2NhFi3R!lX8^x1`)9s_V^hiG<&vUGEceo(16L;a>EL;SmJsX4ZDx=*=kZwBw$|WjaL_y0'
'3hG>ZH?R{K8PAhh<IoM5M@>UK-vRFuLE}pRo(KH}AkUE#)~iRrJXw=Bz%MGx2z(8I{dI5@i0O@Okiy<LBrjqplyZI-h&O@oWXUw(4!UM*x--<X'
'qJAiY8n7{FWs+yBUZBj#&=09q1b}D)7PDfIZOoQ1cFT-K+MrBJ(zhjFN%YEeiLnHgiTVWEUlhHnHnlkBTElIW%#!r<1tCx>rZ*NIF&cGFybvm*'
'!2hS{i!rZ%cQRuP^`Ykt0g>xH;+Pe)+i`MDS`_BMRCvn<fq!=E<2@)AQz6EIjxNDb&glLW45<#=3>Y^683jfMjLRXl)Du<Bbv9owlA*r6nBDgH'
'QKg{ao{?CQ^{stB?p&)F&`GO2w3Q`rYz|mo8tS0FfGAiwZvq(t@>p{(57z9H-D=f}I#@&X#dvofV^~zt4Gq;7qkz5OL;I&uu3GzH_0@Q<IQp^p'
'dFr)xUl?`R2{y}7G)Mpn?cEN)5qRcsz-wGW33rK>_#8q@HG$`vq>|tUQK-7Qsfj+b@Kh;+#fr5uXuEfC!k~9B+Z6Q{j@9Lv05;SV2v=%KbVk_h'
'si#N;Q2mhNOH^N0ue>_2joGC-jm8}!?@^J&`Q#GSb!`&u#W({GLS^j27X$Rdu+0d=@Z@5NDre*Fa&H2l5?}yMj2(zv9T3Aj0lAVbn=#H`z?~+B'
'1+yHGwpJ`y79Bs_Abp|n^{{=Q<<mBq_To4@W-v{1JAEVDse){wxd5zNT8_tfDKOKt%6Db@B&sw=biX=q&#fJ#Dw0`^)vSgTy2_Ff=d+&lSVg38'
'-jAJDi03Wa?o)u#w#lOYI{1#Z8@&zEizV^^g+9uAw8zEX={Mg;9HVWH2Ca7i;JrM<NO3aLY-zTL*9*c7tN}>|r^|RncUV<uZMozw-~h!K(t5F$'
'7}a<!0zH|{p&Qmk!U4MhkzOQd>rOu1*pXt0By^;yhJbRF4$oaKdyinX99r+}S&yQOYp2GxU0vQz*NasfzYf_^%#{UD>2G%{Uc3Yj9x+-g9S8&!'
'av>OW47SWeV8TmpK3|}>Bd(R-HvjG0cjIqAy=6NLr`X%KNAHftr^lbZJN{G^($-r#e&k+gkCc|<(Zl3G?pp`??!%kow>lhUSK(b_Ollnefd+@^'
'WRaAY+0+;s2U0=Yk^=(<>_R#LLPHv%I}TZk4nW3m0mmF+9pIhWdCjWJRDtkSY{qZkxg?9UEQ0k?{I$d=Tw^U`HT+XPdi(anU&e2aKOcX3ck=$^'
'^z+H<@u%b0A7DHFO8$=+1t#n$i!_po9r8t{qGi4&o$;METgK1E+B2L8OG@bVmE=X+J~9S1;DFtX>J8cp57A)Tc<W#~p&`u{BC_5eh*kqqN|#*;'
'OB?+0<F^5lVZ4PxUhU8E#l@Tx6R4Ue6y<^F8`lT4kgdutnly3fXET7npT1#e>IKr4Az&;3%M=>hAw7-BE(&lct}#W=P@BzoE3&nr24{w7OU25M'
'f4qamW3GiT)e;@-<`Un(i;=Daiouz)N91ZfpIfcP&_)i(@ZDS3G~_*jd%k1QZjmgq{H|v$q%n+|X|oEJkCSqE3yP0#K!T2vzb><P2-x3b`D@^S'
';`!Tm5k38k=3wcM7-3zt@{ov7bO3zr^JeF2*DG1~d@bYNvqy<XO*36JuTuOlo=&HByao?c_yRW2s`0(8YUcrS@idMjWrr3k{EjRkjLllo9!>*J'
'RRWVum&vrPI*(wJP^HbGvdA~l3eb!na)H0P0iY>dGcJl3KLrsC<gZ1%!WdOC^c!xh+0r`qChKX8aYXd$wq4T-qoYwsmBv%@hQ)rk5wZo@e_?>g'
'ycW0dq0KxyutwRqnQuQFy%`_AI{B>8pp7l-_|a7%)=$pXGx7>B{?T&!3jJwc6N)fVly{o{SjT{{ek4RPX0$@ntvA)6kD%7l<@oSCEq%em)aQJ8'
'PGu;`!5Tn@mUd|E&elC=9>v;*U0u=SNE8`{%h_S$)srd@a@Ba6t)V-v+B6?eR3u|jd#eoC5)*zRjtFbldYG#Kw@vN>6lgj;?7~O`6)MTVIbVT%'
'h>w=+JOO8Rk`^S@k@G<1F@PUt5r$r@y5-D2d1B17V*(eF@rz_h9x*yt7O7MWlI&r~XfQxyiRcZ3QGNFJ{!I(>Y>qw}VQ~j|x$z=Y+08)71^}i~'
'f6dk$!(7%)`M0x`XfOCAoYFF>Zt%@iMd6mVnb{W8Ufg92Xb_14Gp1j`DWz@cH8kVX%DLhDXsPs~;L3Bt3pIOVFWjombp)E4ER&nDw$sL2hv)kT'
'+nj8Xx*UB!et-NI4iUtzwJ-bRTU!%EF5@CD%e<}04rcRMt>$-Qkhjo8XR(@-pS<TVkS`+{vFX}9Pu+&iR-o3TnVe3?0>y|AdAYV>HRhnlsMVk('
'0xPeI(jUf6t%A&mvXG$iCB;ya4I$mC*~_<HneqATF*R)p0jyfCYL1NV&iu8smoxXE&i9ss^yG<}O>-F=zJ~VCE%m0^m?NZk15m1uX8}B<UVS|4'
'^1u~+{MzWg{+-rFjhnbqni5`m=%Hdv_f<0L?w<`jFrr+i^g)zpF}l$xD6Z1gn4<Q_b2Tfm<aNkl{al;$Bk~=!&W^om+M^0ecPzNOuCC2x!~Cm-'
'^qn<xO&_>Gbv{sqnj?LZCcLC1V8;rV7)OCxNLyC6f|!xq>h$z$Ab#@%HKVA<CIv%w{&x=DC^l@cxJjw(aHQi5Rbc1o64?~<N)A7Y`>+*GG*Lqk'
'slUCL^d{LHmDlY$QF*Aaz}@JwBnyovF%aOktw{2%YNJ}hUG9O-=r)oRK=kjwPLFX^T#>Fq$KA+s$zfPHm#6C5S9`>uqp9p#)g*90`CWkO1J*dn'
'NcYfgM<&DhH1Jjp6L-u5$^ewHgBM-iRIL$rRdjm6DN5G}HlkkFApnowzX5<u<|-+pXg`tJnkC5;1#@HFVCm8P9y3^p#$KC+p#_<(HmfK+@XxH_'
'oi^z%Du)T4BgXfmFNP#Z355K9p7^2Ly%HG{BlyQYIz`n^8tEQTl#BR`bcN676qTrk!exRpSfwr@_E9}`&Y8d>jHkw$yYq6+lV9+x0wIbXn>0um'
'IFS^1@|u;aV@h>0ra+0(smlrC2M%s%-Oz=~{Zv|eL~L8D9W#>9S5*i^iy4=Tg@BVLHiYE&s_YHjc~fwDbxkq!w$P-t_cw+hM27(v#+fK`+WIO4'
'-MQGT5cA8a+=|Woq_L|b>HvXP4PBmi;DB!}c3e|PyW3XPxgl!WYjp(&=PK^8g*zQ80u$`nzO!wfsxg4t6{55ZN3BOUbK;CrA7ulxOjp-D&_`z2'
'_?QFplC6vRox*}_44^O0bi3q#fKd|K>QMv0G<=>0;a9-~O#}cZrn6Z>;mNf`Tdak!U*tkHNPV4Du0xe+P_#|v^?WF6p|bKNbY>Y~^n%xjh{aT1'
'`LoI~iw)}4+mYqO0}s@jb>kyqJx~O>aaLXxUk6Q@Lz*`4lS&*Q_Rcs1<;j7rRq)SEwVw9Cd{aqoSISGPN%H|>I2vz(xX<4<kvtm%GiJTvtc_G;'
'TU%n8=Buu3*0wdYAy;MJ0&Bb!(S@*&04VNbSk1f}+zT4(X$1@WjnPAW>g!l<<Ha@7#Um)mI~i1^_OJKFhcS$Fm<o1jl+JupZTQfkOn?E^6I@cj'
'X+3vJ)|@Rl4i%YVHIS-!dDmvoE?zy;oce()*hDCNt8|R_mY$X22p(iNsIISosX+&S^0)8}Ug$F??loVi0HJ2jQQ!?1>In>Ltc}*T+)h+noMy79'
'I7}5+3(0hWReM~qoh~Qy^%Q-GK&%s)J5UeT`CP=sEfRbE1Ck%QCB?-gO;wX(Z8}8d>7x`I#taIcwXZ%+Mp=^0LdB)t%a;wH0@aW=>5&`AMj5YI'
'+&Yo8NVCAdDfXU;PFfyCdbFM{JPdf-mXh$+UYJ-oOAQK`@Q3B=51&3Azs4wl!KdT@{PyJ2@ta=o*KD2BwI~wcO8O8wTur0bMDD^*inW|b?p8&t'
'R2Fwp_Yxgj#U-3`7_h1JZEX6ZWJR=Tux2Dp2SJ+vvF*%P!8DTq;0xgA8oQYRCqXjB0{8*6@6Cc3ct?Bs-=KvV9e9VhlAj1Nj2NW9sS9DPC}tK&'
'U{&B0^#lMASpecpU=Yc5yevfymHRGp37WdmOEU=i==;n4uwy=kFu*p}iy)hkRiN@1)dp}y^HzMu%+4fqz54LxuS&C|LNuHwGqfess1PNLBrl7|'
'NQIyZH}~<H;Re0WNti^VGh+JXoc^%<t~ypq>Lia;IX<HShATY_o;IF=k2OedFXB5)wov9E3lm2<_+c4x{0<b_MuUA!hwEW_Ex-Ocb<ns#@wAxF'
'J9uWI208zVlzyd1DbKC5)1Ia-Vpi8o-YDbvA@eWvf#(|DYj%!$MbN|XV8Xxs?Qb<|p<2@6v;Gg1YX4z?1M2+YhvJDDLkq(Z#23X73ZHa#v;$S4'
'rT=ACH>y<5`zR36TVr<?gkiWbk{__W``(iuHlC53+SBH`cr58#582jLyWc^ZWsV`{=oe~1DeQWd4wE;Fwsv6LdRp2or!)vWuz9t2l%_{@O>rx$'
'$Q}<<V@Nh-BW9*<!b5^Qu>yoG@x>=^=0c1WAa-|RE|1RKj6#n`udu}w07n7%conC4l^_j&(b4dk0Pt(c7o|(KT}a!uV5*5PzSZ{Nj0Uvz*0ytY'
'(ps=#9B#s(3A0jgXbO+1*B_}F@I<4*bHHsIPd(Fu$J!-QPgS9scx@uWH!gx`&Ml)Sqr=n3#NxANo%jZm-%{)`^|cEtHfZIPKFz_5+mU4$0#!;l'
'`g^{ZqcL0ojnBKR!si(6#BwNg0y=aGIW*V@OQ)*v9YvF^v_w^bUk9Jj7oPKsBc7(1h!}sKF5o4r8Y;GkH95r*vBI<nSmz8}4B9`MP5y2FZ@BA!'
'3-`mn2{9^?M5&3TF_kG<^d8Js_MD-QLRJ`#r4kv;gvDile=knE;_UDr`=LFh|GdFiXT&y<(J<R*qVh4G#}tg8A`c4$++4z9GQ&cB!8y7raJ|hY'
'W3AmscA*zo{Gw9+s!N(1v`b#9`J!ElSc>*6@HRl3sUYWXu>2H44xQ|RJ56Dz*2t-HRwuN^g1Uu~vf<Y=IdNj4Y^v4C!|kD?^_|WwT^mAQL#fCk'
'u2lg|czce}ETOuDz<lBX>IOhafTLb!UEDR;CpcdSdcby*ydYOx#E;rUZ<`9((JY~7s&-Owl)>;zS=l!zkKLteT4@?k!}3;3tQE^6^S8rMHgBiu'
'!~=1@K78#ujYT6sdww$8(4xH$$~E2qx!UqW7xhJSXa}~_ym<aZf%YmH5wT5eYQq7g`W#o=Dt)E`0MuTbJ*;y(y1~xgo#ICK?i;6nKNEy^NR`q!'
'lr-{g%u#y?MtM8Fxc$n{-L#DqHI*u=dA?3(6er8LI_9%D#)#5IUF{p<Z3HQ0+38bhV#yH!y_?I+ltWqmV@;;GG5^vo+wird+!a7bCRd1a2s@ON'
'T!lg%&(kHTc+X?h^|~tFP<wUKD9Euh2-m>yE6-|Zn!E<9F{$SgrKHxRb^)a(f%8e&rC>^b*(-Fts=l*_C=7)=7f<b4v*avFO=>soK)a3xq3Egf'
'TB)@$+u7_q>IUxoxemET_7NxF<(n7_P%tg&qVQTbn7M%%d8#{AX>8lc&}gsW_JP(}Kyp`D04i%jkA{;i<xS8Kh}cqE?f5Rg{EhcA^IwikYb-s%'
'mF){dJZH0|NwfQ(p#<=TPOaW=3ps4S|CWgRMzJS#`xVY)1PftWqj{`<cSmJ4Z<V~ES!sm;tbiL@FDP~vEWeK6PE#21ud_(fCsf8kPy%hOHhkbH'
'*ifossgDK+ha4W4ro;^B;Y*VwiMOnlP->2Rq0erZRKLvTE48qt4gQ?1#OCEHxno0*(8Y6>{1LGWc2qf2Q-F{>lkXO<7&-de+C8qpH=@_L;u<5*'
'MNh={!6`q)lNf3L&8X%N%|<LXBVAL5){QunuAcQ$IyG!{A%?@c>jlSngGT6@3c2m-n<z7UAYBaW2!kAy!f3Q$Q>RS1glR>&+qg2u;aHlz;62$_'
'iIVt#!LcsL{0=B)nPRL{Wtc)2Jm}oZtLZii;CzLY4?0G<I%2Iw6%32+_l!z)je!e~m7O~oWOY%1_nW$Zc>ngVlIKZXFlY$Y*BbThQypNDC0Q<U'
'S_90?gR*CFnSiX0LV8TP@b{iD9y-qfrRk>Rzc>&&{JQ`KUE||(KqhOnXK)nn&dmju<vr>uE~rsSuc`jyY9T;b9`15A6Of96pbMsUhmlM2bSA<g'
'-<86&VYY8Fq207VDh`SSI;nN{7i`dDh(KE|j0D1bT<s+n0T{<R>m^26=L8xNX($MFVFc>TgNB^-BnuCHT;<9~!cUOF#}ad2<L|x1ILqn^h%Q##'
'E}V;&0uWF!yTiHpE_f)+yFSld^Q?F3t3_8@w`AUn15_b65QU(wiJiXDRySyi&CDI%7UF@_1|2lZJ-(>w)D)1dd%pP!2cf>yk_8uAn=q1G0RxQd'
'MEL~<7+zgiEJ+X<wVZZMuF4nCa?=Z(D&f(*$kY|Jvcrk2DD4)z5@~!}$zu%w$KijfkUs{r2@oN)p(?bB3dDI-8Dwy(W_kinbx}u$SPm%ylDR!1'
'aC?RXv{#8gd%~CQ0dv2nrJg+$dG^$q-6LVg4-{1ueZuk7|2Kdkrj;&LB$EJ`2w)5Naui!<<bX#TfchK_iX@>8Wek$#t@o)nGpb_a@cI)`?FD~M'
'l2x^Mn1TV*$?1@X7?!iJ0w3zDUmQ>)DvL?~xIQm{e?WXWtD+|Ws-0&SYtLX2rNvY1=Q;^azxf_U0k~^<F~1w|)Mpg1K1S?6ThA4|<BrEn#VV$O'
'N!lvD!<1e1u1;p?_^wzGMTrmuWLX8gDlUZq3Jx`JV>KjAxPpkwR~)YU24NBw6DDFbK@31`FthO9VfQvaEbC>MZ51Y-t!;11Hb3q7YP`a5IUuoG'
';d3^0Qw!M0c){^-;LT?ae{Ew%qo5tw-`D4NdPz|MY*f{&K(?Gv75CREgOP*eZLQ%Gy7o5BrQF)CVc6!Ypv+3dhpQw325vQ9!|q~)n<6-0!wD_7'
'R338)J=G?PnAV4dUeJmy^+Nzh02I3hulbjQlR@fGCR>1%#eyy*bFD$h_(Gw8uBIQX({Uul!AMdf4Eo(?;n^l~k$KDNO4;azbgQ9wp>WW5de@!8'
'YAtyh;jXqt*VNEZ3FrsIv4)$oI>9*+T0=;IZd0V$!gQy<BMA8_K=<zeZ4oj&(Q6za`s$emW(~LqJ>mf?Jnz1gygjQ#Te>0pDa;dv@R?vtn97zz'
'1IDZ4&wn{Sey^|}>iP1OPdH-F31QLb2-OhgH)NVD(I44l1qX>fG=P3?G7~vk%xisqN5Cs{Vu8t8_+nVQ8G1Tow4}r{$#q*v(-O;*UhvAWa4DGC'
'x<ojqIAN06zK-OyIjeA*O5~}ZLM6kDRgq17{9@4KN{JYoXK$JoB=heTH|Z+b6WaqcJzwQ1UhK;hIV7J4*}St7dBXsWTN_an2)#ysv34tJ0jv*N'
'9WNUGtYi+$$k}dmbl)Nx(i(6o9Le!BlBZ{#=%5p|fPtwMz-RIr4^nGzLwcqb!Yhoc)nN_9jm|ygzh4qtoSIhRfXse;)B5Des8`stvXmzeU$Un`'
'JN)kC^xLDi!JFgLlRv%>J|3N(4qRn}$2t`8s)20r9L}@JmDj8*6d<UgH&xbeB!!Z2l~$JO3@u7*%9EQsE$L`1e0Epi8;(3xDJ{Ih8jct#A!Vw%'
'j7ye~*x6pic&aT>@F4FI-aw3g^wfWLV!m(}h7!5H%k<*1YZ%GvGtMaifK$ZsqQDT&+tCtK>PGTv<G5mv&$kq9RS_ED*BnogsPdIEjSX}FJLNK?'
'gH9QW4bkCm@wd91jg^Dp7spvqZ9L4LNExPz;}a`zn>Cke*?iWnwq@(1o7CQ&Zn_-xvHV8sVV4?wL?gZI-utizSo^4>V8-0aHLY1rRpJ^L)%GY^'
'54TUBiLH$l&}V8}B?2mh@ov2FdR&NWj@pW!2Xyw8<A<mwMFAVvWbU@zfQc=Z4h|bEmKt~4PPw*4rS98jxzSQ*v3}q6v+e7Sh(;1T&UEx+jhZ=z'
'A;;aZ8+Ug!eBT^Qvx=*>bNgzpNk7H%{{+aTG5%KNFlWEEg1K;nr<%j)wty}I7|o&dULC#u^YQyPbeRo15T_Ly))A60@M|`o$BQ@&j18%BG=_We'
'D&51VAB}z-+R9l+uEgj(jR`{Vqc<KVu)R;oTM*u>Vw>vamMcqSy@vN8krzxOOxwyq3p;c#pC|DGUWf-s$ToIiEt03WTaxi7-v?2B72yObjJCyt'
'arbj`*5o5bAO!3&h=RlRvwjo+x7p;`(%|)^_Mt~ddJI!SrEIw_Tyrq>sXUi;RkBd+h`k9<5@vM2kw<D~MAPeyZgG|Q8Gr;~U{Cx(*r3V-EZ6(>'
'ay#Ls`8|FbplU}YG{VQTj*8;}=1Y6NtA-!QR3s)7%)*a(HeF9d3sBkW_E7WiIQZN(D7BAcePZrI3cm&I8>a-jF@KT)N=&!Kp%K!FkTvHxmtqe)'
'e~88=rMg(&cBcLZowvy)%4=lta=$slxcFas4i*0i+>D9};8Gv)E^iKPQg78ER#mK%pOSs{6ZaL#Gj+QCFxL2A%s$Or^Ye#s!MizNf4g7C@a|-G'
'e?l%xXXLKX4aeZA_#Lmoe34kG-hC<Sbpv!Ru97>BUrb5__{C<tlA!?2mvqJrvdW7u$ToL|A*K;IInnsho7&DykJ$`6Mu>FI|60w<3X#5ja%Nib'
'n|N7D5DT+Mx!m1<p~huqBHpr?cII(zd`sSOgqC@{lDywCmj7*}Dwf37{HAK$UVBldhs;wFqi<$kP@Oy_%gmS7I#)TbDgTZX?bsQu#~7-2jCKl>'
'7`;r&%R%rF{%v-xpo=yxPjG;BddNGB+5dnah%drZgenRS5(Tw_>XnHA5UFtEJ-*7Xp$ESnL-}PGUOuWFnju|fQA8^^Ji~BZ&blvaamHl;t|fP~'
'1*EasVhm@#Q5Lr#Ik3I0&*jS3>?$q_Yb)<J=HcMD9|Q~Sd3v75cs6j;Bv!vSmw->pHLxIBn$qizl8gB~EyjQ&t^JVUDGe5^F{Vh!DDvWWCvT2F'
'jE}$neEk0O<iq<@?1pZ^IV%R51~6lJ^6u!5$3{Ul^I|<O8`N`#)@(+XuqDNpRwK<X=Gl2Wd=fg4MKLxI3SlaUop_E0ylMeXY)-2Qx!!{{(^sX-'
'M;uZ+Q<iZP&dXye%?2qc*{0Ioj?uC7GA=G*S{Qfj;Q5O-yMn=Z#ptlz>GU{G8-BY?Zl~!5CXqzy(|5T}g-cWiF2^q2V*4Qhj@y-69__m*sOmaL'
'#@$J5LPazY81Opdcee}^l7U`AMtq9A@b+M`>00*&-EnI>G%2iACMs3ylS$X8mQpjE0}lzDWvzyKu0rdN1qSOKVg!fueC-{r`L0$`43^2#q<Xb+'
'K?*&)fNGKrub+TMQnCh(nmMYym(+7oj1qc3@4I1U3?EZP;yIp(RI~D<lXrs{AIpGidD2}}Cu$roa@&<t1em9bWri%ABna7o42|;S=2!iKwXlA='
'7a<BGO=cIBI4<FQ_~Z#vH_{zgR<~`a*;^&I=n9DZGuk1WF>5~_e}DY;bbR#jWc=sjzuGqdCMm;QCDc@U8=uE_{<coOKYDfa*M~Q@UKYTd7VKiJ'
'@i;~N54}Bn`~LLf+oRJzjgMZxK0ZC&*$Z}ZlU}97@A4a)<EoaenF86&K}}VN4<8a=->1E;f<_oR+ZnmRB6M1DuWHiBL#f$3MGJx<Ee6K(th~d='
'=(0j5pyCwG6*ER(=a6VHaXJ?N&M`Zh>e%Z%XuAq_m6Dr24!Mm(rr_FW*WDfoK4}Fl78+{Y^|Mar%w?}E-*=u15?SfV!w1p5x}@ZxZ~#v2NX+kO'
')jYgh&DmZWY~bW{(NT*qQ)?S4-R5+!uqv?+5?H`{Wv1d5G!Dse-+!23ZeS^e*zZHJ2Fj88YoG~LIH9p9oA3`wuf6dYN%eRPB`(WyB{~==0_U58'
'5vji%lt`&wRJ?fc(pndkUO_$GIBVd<;A#Cu4NNv%%lN-3T4oq{0e%BlJ`hcAJtf8+F3q_Z?4ed%y6M58+2h!sdveMfS=z?x`>Lv|2B@oXkugvs'
'ih@gEDS<=|@57^Yd70&s_5k+dfivGACzZzwY(ngMm6>yhsAYwSe85KuuQuRFE=%ppglGbgVu_kht=~H!{hoWuAFRKtB62!<yE@iTo@`QunI_(g'
'Et{~q%*xE~>Lx(s)N6`I1u#fRh`E}|)rxA7B%!!Qlhfg-BZ2AMO&Q9eXulKuKG^TSgazz+UpIwdGdXTbD?O*2)QT5fUWFabFFjYesq2x5^#PF7'
'qR<otXQ(FDf=6gQ(*l`7L;y1LZMV8DLi!D(GuT)hT=*S1XPn;!_eRs0GquvWwh+r;qwD3>5+;Y{L{R%9of(gUZOdJVvo_T>75T)}@@iNn>94UE'
'B@8;aZ3@|_R<Eib&!x#($ioi^0L{s#Q<z`FJ-NGhAE~mvc+?;n{LmskbkUG=H5RA&F3FSy?oEkw7BUta*CrF0VkH+MouDlmnm?mT;uGI5N0PIb'
'l%+Q<u#M0au9C)NsmWK?60nS4ku5_7B6kcLY5V;@H^7ZW=YArZT`ldu5y4h~UZd8bk6Ihz$U`NON7HNRgs3gvXbZu90I7CshAy>2;z_xm0LMFY'
'ciR-uVS78R#o!vk77Qe(k`oM64gOc#NEMH8#4_1fCx2ox_-KYhm<9gx(-D1>ntQM7F4Kk(6y`eYB-!j1XkusK77PTBjOKdbQh|CovSH+6_nsiF'
'_;E-NwZWAU{XOnVzmbd)K{~<B#f*~9uy2xczB;CNUrGWLQ3L8DhxH!Ud-G5lwu6Llo4aTTm-^|5h`Nt5k#xgbldi}Dyq4SLx0RyLCN$wGyE_CQ'
'tAYPq<mRDM`FSW%HNSr=5+ob-AR4q&i}e@<)>h)wgkdewPz{HO4o_fDOD)YJ-z<DRNAawC@+P>iM%?m@nAvzO1cv{9V&G1SAV{T|C=k7Ku_gSr'
'uy~$atO0{I;6~j>17b88Z@U{d{?XmgT?dAkVa~isdf1LA6<F-Rl&Y8qRqFW|c+%op$0mw#Xi4o=_^moN%u8%%i0e&<VbsXKe``#_My#d?WUfES'
'I%c&m#kS3(Rw%Y<mcFt-FZ*xq^}iGL)sV3r{XXViNVqQA#t6+CI?WeE+WWXo2WSu+;UBsGF`PKaEF=m%e<X;jaTiW4^>oWbAuv-=Sig4YqhSU8'
'voHT_Z3V+m?QEpqJN)g|RwOP}V?)oMJ+vi?bhT|j(89~z)5_C}OMg4h4<6MJfKuAfkZ$L>(TvKBLrvXEbeP_q`a<e7_2V+ZQC6U%qV}+xRC~)M'
'BHCotEGPjav#eBR7f+(VNwiYkH8KFwz0W7^bNUt&cAP~Gen~a|BDw!kH;S{QoLtJdeK%o|{Ku~(&IJq>IT8G`C$<JkQ&g4cEw0eQ0~iGK$>Q9R'
'>@POXu1vE+x#Wh~asbupx|1)#xk)M2oSFXU^z``i`0dH7Pe-2|-irHSJ3D2tCYD@k43zamZq)3RH7?W^@iLv!#_qUz+wPl%W`K+Jf@umS<x^Hw'
'm|v72ll>+A^L%&mUgmoG`opJVoN2$ekM>83YjU6mMKI?HYIWU5?hUlDI(dS;THzSV0EJ9$ohV9V=R$EzDDhD}1<h!ncNecDWp8QqEq)9Q1Bag6'
'L}p_;Pm%SIau%78Ot+G#YEdkzw-H77>y3FWjzen`(FVi(O;HEcWvk*-fWHOKTvVHs4lvxf<1~yX@TYA&A;yy1b478*k?xJPHafAAox^F2Nf%ei'
')RDmJ21!O>@m;VlgY}v$^enR$l98O#-yB$7?mumdHN=>0r-_aW5zS!I@ea8GCvG2)n3X)RXKiR=+m!I;^`Nvw@oQ_oZ7i8uNN_O+ZVU-IqIQi9'
'GVYkPU_eQ|))cS1TZR0rGfP%a$vnMC*|K#s;rnnq=sXF=EMcggTH5?=O%c26Cn+E>;6X@bzA%sa#P9dXP{=x7XQRQhGxz9>sv`$X=XzNZ=%c|='
'HNYBP7=>z5P962w%nDkcaKqU%&i<KKDx0|5VMtaJ(_+o%?x2N|iL+lFQ=n5X#p2i3PM*XCi{kMdtqkul@B1{YcF_YbL>v6u*bdJVw9IEw2{ts#'
'sV#GBjls&^{@x*ntVdVNA6ha)Wq#qygrJk?u#<*Oiu8Uy$N;#qU4)(p$B-dAwL5B6Z(DE_X}RbP1F!K`oSh7=X|E|Q&rD9tJyF#`|6sZq>+D0V'
';3-I;@hllD3rJ%;+^Z?Fw`gr!ttMZYR&TL7Y*o4cOvHq&bBk2YEgpQU_xIIBtHHRrc?1orkFhRQ+$|^8s#^8WI#IQb{=UhYSeLP_aGiH3infs('
'92e=2joxo{Gp!a0fOG+l{E^DghEA9m7{_;W-MpxzMQJrz1)xpU$CJ0ph>A4x{-PAviQf0nWwdRq6RHCw@Y%LBmt3L;LU1~}^EpCoxljOdbe@dK'
'tfswnhs?9eTU;;GoKNH-X*24yQiBkupSDP*#>4)3S=hOPY8yrG(3QCA4g9zsL$lUvy)b$5ch2I3*z4ZIIs7We#AAjJ5C%7C*-VRJlID%$?&SxU'
'nPChVYMaQ4kTVvBV(TI%;;3FZoZ_#8*O)Ncac`p2vB&saW#s*eBjHI197;ANyM@9g+Uz+wdzHJrVmg!)nCv@SUpUNBN0-73fXzhAL&|*^%i*+p'
'@pOvp)ea~6S?hfk@VZqQyx5>~%SL|{$a*2&4DKzWIOsi_ZCq@#g-O?vrI23B3CQKHs`})*k8`xVwnh~%XBlDEDJKYn!0d{mgjFPH^XRD6)XZI7'
'Aj*JH+!DS$#8M4rXb-a5Wi*_sFcTFs5LF~$s_g0(uVgCRu`O0F51qnF>)eS<>kzk;ZHv#BBSWu@K+&jtf&#@<a^6%SBApi$H1c+cJ26yyF|>AM'
'$bXCgi%t_(;6?eW0?RcORqvx9Tg8*Kbmu5fBZl#z1E?qrAt&lo;)f0o<AWM~^5|FX>d1rtL|^C1y-L5cg;qRyq6hC|scTLzCg}NCdGf@e3kLSd'
'*->66DL`j!iI*C9VS^*61~u^TIa`Mp2`o)X%8~o<7XAeQ3h@^zM%a*WpbmD(00a;Ci?=~Dc6}3gd8$b?)><VMrPWi#NGajgGD5m~Dmsw{-IcOJ'
'z<R1X2|C>R_G_ns$6x8xXiUhgX#+7KzhcE+*et{Wx)DqNZKc9x3VaQaC)AaFDSH$&jNKiLKxv|?ibw;vEQbf)7;X5_@sP9>?R<12edxU2$S}Pz'
'!hBmt_au_$gj?ajnM2l};;_m#`>U#G$xnj)eq{%H33#4jk1xn6=X#M%+tfr9Ji`ETiY5XLUYChmGRG_yFRD_|2fE9{LZpRLYF`&puG!%|b-o$g'
'OXDQz4KU3!f9>6KOY|djRw5}T@k;ziePL%z?@?`L8~*SEkhC2CbgGB<{OsO*w&7<R^Vtt{CmDq1J()_sL2ywGM##+|U@N+V7o^Z0Bf2Wo!$W7p'
'VHo}eT2pc7fT>xbjuS6xC9>wx>1jYs1Q@lMm{j6b=jn>#+)=y&POHXEVo>0v>x0e-{d#mKq_b~xvqN%Pzkvbt@Sir-cSt7*D&u*!ycm;XipomN'
'&w>73-mNa-3<&7k&C}n}K(ECa@?9h9W;CD<R4N7KYaRaZUdC>2J62||d5YpoE~-n?Y1MW{T`sjX?^S^dUm=lMp8Dsk+6|q=my&eh;>Synhw{A`'
'`0yoNu34$b<7(^H);3fzPA%_LzP1e7&kym6@;UtT(j;_dHXSJSJ9vNa0{;1}RY;PjsAhhLC7xR)jO07g4D^NnTBX$Gry9<0aXbgA$i~_fJs)E1'
'j6l`;#Ym0o%zIeki=sWjqjI#otDgc2mo`T+E|!#ZYF#zKte*>P(_C|Az^k1adLRO~NsnG27283j?hKGbyu8;9AVR=rP`Q17yNT}aHl6TH-y>pE'
'=IDTTP>G9FULw<~DK_Ga{qO^nZ0{fRqx~2Cj-m{dqReiUZgN!?q=wdTh?v(|#uz*ee%FUDJm9Dw9lU&C7nEG-ft_?8-^nVgPeE=GMz|uy<o4xr'
'Ts``Jr@og!)AnEXBQ60ng<m6ak;1T)@E%sPqZ4%r)B>KzunB@qut!V{9$J(|fe1yUZ30mhJHR+SD$&p&2>-}Uys4)iQc0((LPwH?$|$4KQB{Th'
'q7qaYr8rYnRpviAbCrn~%Gh}S8k2p=QKmyXp;X3{oW&ind14pw72V{E@VPrhkcvAhg)rXmDL+yrR9<aiU*xIZ7C>qw7*|y)kzIg*)-)~0T=5vb'
'VhuZUFqEUMGgLtf1mWq<i#rTGpXgc8K+V?^;9DqPuMSE=!8;^#I3|=)%TVozG^x%IH(@IDxy`3TRk>Fst$-8eJS54c>KZpj#|bt%<?!}O`GIdm'
'yeDx<?zIdC9QCuc!YDn7><tH|-`h8<7!!|VjyRAX(^b0)_$zbw<VE9FzphQ%NW~-BJnF0Bb!^4$t?rqWSSSN~Dy{jKQwxe{grLAQs_Hgx1S&*v'
'y+qKhmX02Bj2~?(x;>e+u!sbAJ`*dlU$H?kC!Xd3;XITzs2l=YcPa^Vamgj;Aurw;%voNJy8CA%TBEbzYXF4bK1cSAt`)feT~Gd2J=9$1?T8oR'
'j7C9;bX5haad_ZD2CahHM%DVLf2Ic&dBzk459-m?9m>S*TYGt`hJXuV7(C0GSH=GY<E^5xjSMtDIwgC_;AA<=&Jfc0H|C-o1lNbXe*Y;x1>AV|'
'DJi}mkaMK*<gf7LcL($Yg^yDt34`E(TZ-q&^c4&%$&Z%P)5{omf<bVXur0hvY)0Q5J`JdDL4){)JSGOgdkjhvd;=q2>Th`M6a^pi6hn)@&gO88'
'qTrN-Q{%yhb%|@Le!hw)SNeN~DG*OUMolMC@GWqFITnRR;DB8vP!qo5*uR&*e}%uObi>3`N0@0B+rU|!G69snpcG9J3Mr;93f{%}MY?>ezI;+&'
'zKeom5$Ua!kOdT48of#W9)GtEPJy9~4qrz9`%nG;H}rA8?$iI^>+kyfiS~h!!vF2}`9Jy|J%j%aBH(b4$@wk(=j;7XS$1W0{o?Q^==u=;d#SoU'
'Z0Py}riQTke*RPSt*8C6p>Lay`FRHMTn*x2`yiOJvp3-d^~PPhUmrS#=Z_eJFpl;@Fc2Od9vw-~#@C-?5X70<LwK=eFP_l{zP^6FPoGE)!nYS7'
'%>Hio5P<o$H=_R&0K;<=BA?R$T%>qR`4TOJNH;l%sfgV;#v2MjLX0Z6p_|!^l0$yRq$|>oLEGI-Sz0GzlX>J}zUIQ7$Xf=*n}u&AdC4H?NB4&I'
'kr<wh`N-y%5;P{;MCusI>PdOg-{1&4TK9_S3)2ZO^EwHY1V_;veowt26BGToIaZKv5-*NO?}8E!86ePw=bCL|+>STZf;JWLaM#*a)m{j>aU@9#'
'*R0NIW5tTE?musCfflt>+e)47P7FP;C?$Zbds3-AcJ4{-UAK#yFtEQ12yle>pe91YQw0O{)L1p0P-k;*D^=B$EZ@kA7qei2;DAyzA<XE^r}T(3'
'Wmf^{O!ycyE{M4V&!hgJkN*d*su}$Ed%s2SCmO6_|K8F}Y}`58!>ta+_W4!mTzP_=mdQUuPO~b51Prj>>u&>OU9uY-^bR`@T%PR!qS}O~%q6(1'
'LwLM#d8*k~>tkpJ%=;PC)H7eL!U?0Iht&6;dl;?kNQ*I9d%4j%F#wmX`te0Jge9*<X51bG(T&d-{C9Bl?VFPiR>;OG7%ioK8FNrVY(?v@N#3hD'
'H&p4gno^$da9@9TbNrf|#MWh254o=av1*UBo0kVK4#${73uAkPm&x^{`ZZ6_Z{fA2@>KeK)k+k!M`w1s=!CDT3N{bBoZ*fP5$0tc&tlG0-^Eg0'
'_PAsv>uk6`IOmAX!~P(jbmwrMV)XTdpJ-ucJy@KVd6EoC@adLoPBs5yl4q!QUQX_`@~|;w!ME_=`|pn5zvd+MbF_ajf#E(RmVlxT;0-c}P3C&c'
'W{XvV4j#qjx=iQ28-Vu9cF2!I8`JZ{2U%<CD450ie3fUIF0GfZm+cX^fwR3t_3nARTyk{vu0T|Q0%=iB(e>1*{PE=DF+ET6+<gA#_`CPtzI|J{'
't71@c%8_YZ{Fon@$+-zP;+P8K?Qy@`1HsxLXMx7*kY+OZ2mU#D7CIT--FxMSM26Ln82w*`LNjLL?3Rons?fA>R7K(G2Bk&ZC9eibF{?a^c6tw;'
'e)Ih{-ytzZt2-&k%2n3L;vB_B!{U%&wL!j7o^ZsqmN<FmFc^^?vKg9|nyBK^V#|&VEYV9p!gkTE_Fgv&={TwCVRh?6!>W&)9*_F=O(su9@20lT'
'oB;$Bc#(x0+&05+8h&sdC#r^hrr_E(%Kzrm@$u=0Z{NT1$5F9%gsN}bi5-C;&6<8cef#WbuYJ7?y)3^#AO7nD<+zVDeb0@lyUmIJGp)fr%>_O4'
'?V2?=eerJwaN(QOP+w7u`oZT9!Lu`tx}MCZXdDx!(}>hr<Vlp=6p`4!Md-6mZ_rm9(GvQKqrMAjJ$HxUy7?QgJM~rE>g_LXzfFl6EI)sXu&@m&'
'N6@rKSKD{0QIhe8)X)HqA2&*XS{o}q7(eS?{-e8CJ#(Gox7C<SjoRSrfd#w+H~2)uSrg7%2!Cf`bp+kG06pxsYR56zr!W9U+ic`<epdqWf8Hlc'
'n}b97V~o9Sz=U}OB_Ir6D@@r6{u?#0uW(*5iSz_&Ra9)kh3j)b3zvxA+r|u}tR2s+6LxB++n9nvx4R6n>T}Rz4UodES7I^DcD^(>%lK;lrD1$k'
'+qjD8-7PVNDV(8kl^*h+bSiE_rpBEM7xdh`RG!2B^vIl{YBuvYTPxLsk>*m=93<aVk8O->X2o1io4Wm#blXq{QK&H@YX~Eq`5BjrR5L?=3(5eO'
'M8xZCzFs7rlKjG;lJ+c`7iN{N$n4qXi0P>U6qUoV?dujDof?^%&!11t1do2|$0HD3iks`PmLRUmn9U}JI(_k4wS=4D8$$qmL_(dMXF$Mp>I%<!'
'&^OYQ+1tJuY=lz3cXE|9-Es%3wgz1ZO*YJnHfN!(V0vivZsKbLQ+ng}_$3osDrA7WoTw_|m8U0?T^%_&x`NKz#+W*1Xt4$Fhc=k1!2L?Ze~ehe'
'&lPRPtA;H?4PgP2UwE@7auWLrgiQT}nf<|hZaZWl{&2&<)9TedvMSLD=Fnp|A2sJFU@wqiDAtQ%XVk?2^JmT!1g!McrA5uEWfkqRd&psr^dl?j'
'$Cq02R!2%9{-R1p)L^+Q2SsLL(2nzUI-ic?^)$`K1SkIvgk$-|r#QJ5;}RklC=JpRN5QX6-)Fat1PMQNRj{~tR&u7<l<lle!P(8Y6j(=IMm$hA'
'_JENk#G+BQPrT;DoYAVWHAqcWkGx^kRy|+@lDa%uA_kuNUJ-IYp@1g3BLqD2Kq>&dW?#JXsa`Ad^A>c_Lf6=8O2N~hh1j*#F_$(9de5HQvm>>o'
'gQ1qz>0S}2aU%$3{7s)_WQc7i_nPlx{mAMxsmR1~(Di9>Uk!Bgl|ZhFub@czx=1IDy2*Fno}RoOe?0p9C!erNbvgMN={L<NF3xshng!<-_Y8Qt'
'1?QsW+3*zIPBLJh@0_wVZ79E|7-`syHLgMJXeqbnkjjO1Pv%;yFn1C`wmX|SVu_~}k7KGAG>Ega>uc!XLH`t3{v9o<_Icce;K_uPHv$)#)JgQS'
'QxVl|tMAOzsQ;J=(<mN}GX%T=a4PK4+B(cfBNBZ{gFMxuT^Ocu-NH7cqZ?vPJI-TlUaa#vjSJ<&PUxdX$Owc=nn<g<ETZLXT1_dfR#UptfkU>5'
'd6DE(bTGS3{<%Rjf?1kOk{eW-TqSp;ksT+utGum#DI&}X?Mb^MinlVyjDdVd?)9+=!|)aI7D22{(7*YH)cM#0u#vba<z<Z2x=ivFsdGVm3nnr0'
'+}xS%AOHo8Nch#3wA)S~BognDKZP-wBkiaHDi|FWlzDX|!=SSN(@S6~o&2eUbgnB$P@)0XMY*sa>ic)K5E%9Lo-2^?+y2?0_k5;qbIFGznZd?V'
'v@^aFpqO;iHVMaQNy@b&>2|a~z_2f-`8dj>=+@GwkR`Lyc(pID_7MSds514gSdn*(tZ*MnmixG>ed|o99aM8^-*Z<qi>LAn>lu$bPugYozLXz('
'_t+DE4^8xGtx-B_&&zfh-Ix1ZG`hDs-$eIS4c(AdIjc8T{z?S5@pW3PfugZ~v&|35uHE!fQa-WrW42;Hr8?x>UNHjq8nCeo<ShXqp*jk^k|dRJ'
'qQQ>Awx@vq#|+b}!W3}*=%5s6wa0pU)4wK(CzrzJv#V2ZbBW<7u99R0or!)LdXzF#I3<!b4~_^R>R#{}dnWM?j$=+f>9DrfFr<uXGSVcEs-B*)'
'IZ$%qZon+O2t9qeBE*+4la@B*-~wU|QYC*VG5f=;l`Z$~Vcj>~mhp~2f?mTnRU%TH$n7J?bNe%n<?fMVx%(N%BBug)4R}1{ceEG7djk2*;9fj-'
'2LM?>CV}a@0J#?<?5&bXIVOrWOqUb%yiV$dE0a81)!K>#l!reKTle^V(;D0kTkZ6wo8Ii<1HcQt=TFgZrqgK+YOUU3;mdA%X%)Vs!qegYpio`j'
'R>N2GQ>F}Z_Bpl$IiwS7Y2Oc4H-}q%TGE0%!|}?4eVL4x#ZAgdw8>kDoXgDDBnQyz1rY5^^F8~Yh;&#5jYKovv5{(9XkejYfLCrGoR!PTp&F@a'
'G|BfuXt%hTY>9g)C3d^N7}^3+_QoJhECs7dD|IcxBOF?27H|pxdrb{qi5v5~zxcDwKE(b&sC~`;z?a5N8Z?r)sa6!H37ys1PcGuHvO<xcM-TUf'
'=`^|ER8!HN{2m>5_IJ$3020nVjvA9VU82RI3Tc8f(aml%+A?K95-j-|jRK{*GH1p1S{bShzCpJ>B;pD*aIBxXOtW$=mz3t*g>G8Fb0~j1rmopi'
'`SSbmy@F>_`7Tz6RjAY5#i3BB)cAy$D9jugXKreOwJl+75tFzrB`2{Jn6$5LI(&V)B{3q45sHvkPcN&z3yxx>a+8x)t~{SAyFhM*I?=6mw4-*m'
'Sj5Yz6C<^<*`eeBHOrlOb}<KDd9Gtu(^F=`#>{rzj>}7oXGi8K2VwIHOPHg5Wjn(uxJatvHIra~?y;f*bPa&+a1~F#-5HKj-18yULBW=B&THic'
'29w1>*SJxuJ8MO)d3-%f;cplInYAX_Jj;iZVgGi&zkhx(K{BbMt?&hcqMFWQMMU$e0+LdowfA+z)Aga9LGQ(E(=sl?JvuXU{9(5!$dlOZKw5h?'
'+psZ^w@@?st7@ilQNnLKW+YN$uV5_u2HC@>KuT=Iv_%8iIo{PtbEKkFMX;!=yf`Q+zU+BJq#0||#>FWCSQq1YmaS+q`_`1GE*TBlwwfJnEvPE`'
'>KVOSbJqHfxtgb5rjDx|sjVKWUF+1?gZ9m92xOF-FDDw*9bU}CIu_KvtDwVG#Av!j8)FoS_7o%}Gt9KNYV9VQb)Ju!Laot;$plY*%O;oye^=d%'
'NNZuLFRXE3jtNv$JUioXjjFNM*$$FI@}=2V-K^;*zBbftT$V7r<1A?0v5FoQmFrsdMHbi{&!ww$>{{tlR732WgBofG2AL^8Z&74Js_wZa<Yo<S'
'>RN^GTe95M?(FGD^n^xJzt@b*r%qd6Yg+acIJ1XcQMUe=EUAZSps_n?jT2bI6n0O=a0-ffR(=J%DC)nKEQ*sOqZ{eX*v?5zO6Ov1ZYVLB-Zu^u'
'NEvoh&<g&>B{kx_@6B2UD99c7rnRl6#DDiid^bNNv%$2_)v3UMEc)4aa=Bh!B~u2meq`@Yq%Y*+<;6O_NTRCy=Afaq9v=}9)VH_5Da<+O;CW&;'
'F8Q1#xwG0IU(is~oBg4qGqyt^bCh|A8;jUv1;373^u2ZV6pkmxi{ppzjUGAfHAbUOx4#amYid#@mGNR#1QIwHLkD7L&<i+S^g0UG%XxaGl}ksb'
'pPcZH3KoT@Cm1%c_j-jP)`s^jOK@rp%;*YjlNlKHql#e|YW4h?>w5k;RvN9_R=gl~Gw0d#PUA&g#^r@=&#j)^uW=Mc9xUPYDoFNyYq2g9;POQ0'
')7scBUh4Bm#@<|Pi@_J&do`P<1_)RwRc;118kQ8)D56wnqZ;n_{R~w_J{0vJ{@9^<mcP;t2I|aA?kBe}Y9Wyhk6b>fPVj9u$v?!$^G#%9g<`0r'
'?}`hCfUx7A9t?Cw-X+NVn@qAc*R^=yFkY_b^CF)NCzmKP%`vqCSpk7`lw^Y<M)?R8eaZ0fWxwAyy#@-rktG7nf{ihWT{CkyG`A&X-$)xyl3GJ%'
'3@cJ4DdY{_&J=4R_o=F4h&0?J;SUz<=J7cU)Za4|<GtTjOl}uQnO}@qgGaaVh7_1arXX0cKN=~R)@H$@L!VT`Kvf8j2Ti0=f8REacIw6nrYZ<m'
'u3p+f_14u1^1NHFR3guoxv?hvOI87o2Y`ITn1x@U%$_Eq%wEOQdd+r*O6tzu0enLZG=5{j`W%q&RaEJj4ZfG*x4PIL-KU$u_1V2O-c5wDU*deY'
'k6%`0G5q-Xw0rvb=+kHTGP;KCg=%M}?PCp4jV^rUAEi#Hjanm$oDsobG8ZHf#bur@GzA^rPf6a7V&6n!&%oGpVHc8XUd#9lL?p&kJhmw2Y@`lS'
'E7X9+DpxBG_!wWrDPYG#964<MPcW+*gnEW=VLqBlz1wGx>9&TKjNaVx)%AxDH5r#9ctSk>=0^R|Kn2sZ^k0CO_O()B5--QMcu9h`$%dZ6s_?aW'
'$|>Iqymp7uLEx=*WY>%5^R{6Ppcy<0%G0MJwfi3`$%3dUOV_&Ro+?$gJro$5sW&%{C``O|YAp{DW8+j=b1!vS4j8A_)9CEi<PPy;qON!^GQ43X'
'0}QlDgjG;C&gl}^2K<3^pj7~H=7n4tn~~Lj9em4-IdbxIfF-PA+yG;C=F7AQ$oVh7#!&E3@J@>C36F2Kr=<bSW{TsrI7y=!&m<xNA0`mQ$btWu'
'DBIPbjMKR>KDDo;Za^n<-t)3&7~Jq9wLzow%%iY0AM;&F7w=a5vJ{_Yr`4zX4ec5)s+K)c9k)shfx`*Z0MJgIgrJTQEBaKM7-79EE(!(IHl<VU'
'N(jP<Fzi{q*Fd%*Id&TSo=!qz-}V?Zn9o|HYj}j1$)g~ku{nsdbx(CSB~t04%vNKR1c#G3suStAfk4%0V1o&l-kaUmqU76SR6Uwi9-05bYRA`r'
'#IVxK!8I(jz<tE4BB?^pY53~<!LIKBooBaM8&?6>cA&#xsGdm*KW%O%)fAPFZ%gajX{~|Up%JxkwjmrhYyyo|HO}FE4Wn1T8Lw{!UdX_Xy88?t'
'gT|gX$ldT>wt}I_oWB>E-2F8x_&|xb`DtyRc{Thf#(L%3Z~S=4z0FqR$EH`-=pVz~s2HMk>;dxh48yQbfJ1|Au!tvc#>4$VJnWCIvErGV!q^^M'
's7f`m+tw&dQo3^#eHg|5D6WmdCefiP;gxHD_9gk*XS;K~8<#^fM-NhG8LF(H(q%O-<t)56dfo^}-ydNU(;a)(GD@|yn!&i9kUK7Q0S2elxtOr7'
'#vHuJ?~L1p%1*<Mfx{Pm7!e4VS8c$`PsPg1qVi6|WS>#HW2k$qsG5?4){F+_)Mbl8LvY4H9dLMjl7xdN(iHvbbTBG?RYcl<Vs(dw9xiYHL>=xB'
')BmF%Zbf8YunygW?GnEMN+kPJ!<$IN)coQ+j%a`)urAWuBp=4{kSnTmC@cy(0-_BTT*N0VX4i@$n<oR{436U%*#Q1_M~XamqPHFIlAD>0YT=Pg'
'NUn;l0X`}9f^`;Dd@rt3OA}+%za>-2D7d4)s0}OgwwgsJGJoEA&d&oPh2uxJZde@C$Z35$(Y@+o^Yrpbd%uf@%$12jXYal`^i5N$wPaa6&$kZP'
'jXVChA$R|84B5^0UG3%WA>VGE-uyH}zJ1J)YnjC#H|FlYcFe>rZUG7dWcLrA6ma|wo_suc;*RgY?99{EuBiP60z^j!?7I4;Et3H!8u$n=D5~E;'
'8E$HPUmd5~<w-;+hv^c;KVW8%mcR@fRycBKDX;h-p06&E3-V*iEh9Lq@a<D$G50>df-<P?m_Dq*dxP5=?GIqVHfL@NtcJN8-VR&$42zrA;0}J='
'sUJx;8^;qO4j8a$!1e;PEPT~JY@H|NO@ey0$QW*P-^8CVXn<e`k9;$+nM71Sn*|kzrD8*xsz%D3TaRDK6=2}&`E-1~&gqIWp!D9hH=ChG!~k?{'
'>zb1pP0~OmMKL~)^QvzvK`F3)O|{iUD0>mRwc5KaDZDSKH@21Z45M*DRImPUhi$6tL`l?*y5RvkH6^CA+@@I;te$el1@a0%J5IKsaL>#EG~m2I'
'1M^vnpS8B00P_HBpc?>d)%OO#Pu6m~fT9a2xC3TNq#)FK8KQtFSf=aA)mS;&ZZU(x43VZ4mE02hjIKy7kU{eOcmcx%<}PJDB`QhqF+L@=muTNq'
'JC&qiwHWPpcA9dTa?H5t7FxHO+;+D0=SnvEfI~6ke5(S}1kQcnNvjo?oKdUqhG$h5QQ}6r`@X~&x$ojf-R$sRxjJ!gR4i6G7RrH<^#DgQd5%<>'
'ZttZKhv~52d+EJ~5nbft?|!~yF0{lj2mG`7Z~41@ik|V2oxg^970=B9s2vkfI@p+UWDO4(qT#S9z%W5vRDljrH+6<i<<^4+%SUk6N?V!&JzRC&'
'V2)UyQXiGGamlEE#v}K%<0ueyt6ZVhiBnvgRaH2{mul9Yr7rul)nXlx+eYVUlUg%|p%2++hBZU=s{5P45@pqS`eTCZm4Y<Pa0cjd6wl{WscEM@'
'%TTMbYZpZ~tee^BZ)lo@Bz}ly{6NO$(_ycF@O;23hL?l5EJ5ra9`py8UKh^r@ZfoWgh17_&jw_g#f&jlkl@qp%bj3VsCFy314<oSFwfRg+<@W9'
'u)BXa`1}$6^qvpiemWZN?~h_YUT0MxfQP6c&5@yMum%H00Yhg_mxP09o~@uuL)kv;M_)K(xf;%3D<l3hp3OPESS^~bURx?h7(fD$nmFJg{xSIB'
'aHUpojg}Ey<9iIG`tj58>G9`Q)WSPW3~1^xwUu_XL7onN3e}4BD#_)$ZX#;oAHcBu1^%G%!+L?`=h;om6{qmyvn@SYUc-U3f@hh7cCVSNwH>u2'
'Tx%=hzfr3zPB1WxL(^HqtjOLalYG{SkQ3BQZa!02l8!B!<z|D7;#UHI0+*N+sJn>K4$t7^s)wOK3Pr9RT!)bYl_$oq@Z}2Nb+({@vv=$3!-GDX'
'_T^%=hh|3HMoeaJF~wl+p3_e_d^*w&9m4apL~dwvMTKH=8DjIdtrm5Q?3%`dKc3>&sti&%0>1W9%QVKQUb0pc5wZQdzOYPM!}|myUc%{o0elno'
'cytmNzMsl1El~br8^<A$JG4S-$)oMjw0wSk;=vBG6re*iX3nIoD!!BiE`uWH$(XEp)Ra|;n)9p-JRGRG*clm}%*fK3<zS1kFesEHrmyl%&Md8c'
'5C;LXr99|S_2MiFo>?8JRK}_#|2{K|=<LR-j6ARk5wIF%7}oizRZx6<XGsjJMbmO}N{2Nu8p_n3Id%)7-=+>~`d_%!#Z-;`m(1!&DXhy7Hwks<'
'yL5z#zQ^@JuWI!kET`{s;;V$>$BXYQxx2&Fwsfdwstufj&v*>)s<1QqLe6cOU4^WsbVC#h3!4~6@_(N#bgjwCpQy3rN)q^j_Swg&scA!WepIth'
'o>*INyj6jNH->lrdPU)$7%v$43bfJhI91VXOb_&CcW`f5XF_#jKqzQ+8W;TA;J?s);QU~GnanZRrBp&iEIb74A73Q784>y%*iGU}5ap>bHR6aI'
'zB+pS=T~o!{_A-B>f6_UKK?v@fAsE{V}Buf4?V$uxjldUU*%CubCZzRY=<d3H>Rg#Ees2`M!jvEcEuB7a9=IsW>BL-G<o#8Mmr4FKy%x#4v<6n'
'=H&a$o<yHEe5COss{(cGWimam<WW!dn4*}owu2`eRefXpMMSrI3yOfh?-)xqEcM54U%fqfO~Jrl9i8GL8!_1GFDY7T&hgh!8CH)`dLPnc5%=L='
'r~)(7gT#>B?tn2W8HMV;$8m1<XlQ%tOsT~qcV(Ig;WbnT%C_!~JO+X*E~0oGw!>Ek0WYYELpKP1gWL?p8}m#^7Qf2ITE!-N*6*LmS&Y%1FQ5Px'
'>!lUFVoRXOzh~#Q=oJC`_zZ&AAUh&99`$-Xe&=Q|{SqH}E~ulm!&v!AZQ}7woSFqHpHvvN8l|ikkcz9Z8RFj`$?q>8KK*(8=HwGX5+f97ssJzw'
'_GnSz*C>R4;Hc@O8@g$6oI^)ZK#Jryhg(*6qw@sSFbn~CT0DA8O>>b9L$=a@Q7J7thBa1Uy2T1~C^h2=d9F#A%4$*cI8|IioL^N7p(D+DWhPGe'
'ZG(f1qz~1hx?)b3RoqQ{R$c5=S*EbQ$=U^7vX{`2>NvL)io$Nm$|)ql(3#vc;@4V9BbbGeXKDv8eVWXHKLA!j*6#)gUv?8*#%9nAH(rKsbCl;F'
'9d3J%_7Bw3RhE#Rd0PH(628`Oic55G(k($<3<$F6qjGj@+UL5{rs_O061Aag3;(G3Zq^{{OCqv=HWx8iFLjvduf(BbuT05IdIm`690Oym3pWnq'
'p%I7Cw^71A1Ie1-6=`8_)xk>+o54KI5IJh=W{54ax2$7wi`;uTRxzX08d{5|mnsSlU&{Kb4gj(_QW$VwH3uJhFbeSwGCU*aUX+~JG|_t3QHg_5'
'hT{7FTojiCi6Ht$$kkJYO80_Ly`&rtp{IhG>Z+OzmhMg0i&f#I*l?sQ{4gTMOvf}fPXRe9e1K6@)2Wp0RLStKo7jTtSKBV4lvl9OMNt}LKTA)6'
'JRv>KvZPRL0u0DKFjRPI7@Lu26g+$osDQ5ISBvY^Cd2BGR@ohxJ110<0t6p8kCSBqqU<;>CTYqM!p$rm#&<XEm>pEnI#^XDIb!w67!4bz?b%`E'
'z_xMj>hT_H%?=;5LR)YFP->PAZ7gtzu`B(<k;b?pB~*Wls*yMHTb&_w=E#>1|6KzAuJGL!^r71AaZU3e%8&|&eNX{O-A<&~uY=d~c(LLD^<Awp'
'0z7*GcuZmlLf3QUBO;C+z!N3&Y?X*}#{?TmrrMQTsE^^!-ITez9zV<04wILU;XG;Phcw^mwCnXW>UA?q>1u46!3h>XCWDbN+x^t~Fr@|6PQ{-j'
'N<(OO@zm2ACBU@iYBu+^JxF&-&Uz~n+2}uSHD>V*MyE)&V>lW^8YX{-vKS$kAJNy20lfCkp#{1#FiXj~b+08<fTG+^=r0e+nWbg<V35m`MVu}<'
'U6{#{ik+2aNm29Rf?U@FXxl-!(|J6gS0nTr3!lS&Tx|nAXV@Ol1u)dTG((<mx?lT7<ZOVMS>6}yA*{xRM7Qd3<0i1LmZCe{HMA|T=!}nYS2jfJ'
'6E_bhXx~o&2;4kf{fsFwfDY1(KSYAwiW@ndfJ!*#L5_kI`xQo1sL+|!PP8bQ3w$|%^$&NYL{N*2uFdV_CMzIysz-VXHau3<y4I+?aA>aXZXgt<'
'&uCS%e!AC<h2!iUzz{@6hvo(w$(|dx)01@hr{BmbWARV8m8#K4Z>CwX>x^y7hN_i%@RnLF>rFL!x3|^UExSWavo!liYV4|fJ^zT?;ONY4q*$+3'
'^8}GXoZm^b1#eSmQDUMCdueJZn@*wKET=@6+^Eh?$SIismuA<P!s(7!xrB1AV|-}MEvpflx?n%t*Jo8FU}{<vJ26*%S!TD5RjY(BI}~4Ppk<z7'
'$ZeZhe|{Lr!N%-}X;Kio?wOzh4i#T?5iipjF&|v+0qnUFoq6kOu{T{6h|8nf{cMp-Sgnj|Tx+T?UPelKSKu?$mwtMFcHXUK1(6DQieoQK{cgRA'
'x|>i{B=O<HDj|wg*W(x~A4QM$ek}$JhwOZ6TREx@w+E1QxeGIXvI%j{BLM{G0D5G42|;T=^eG~o;I#e(nK%+zL2~b553Huy)K(yR$ZR!~@O^M4'
'#tNwTM`p0`GQeJT<=$4+m8f-f#*AVSQ!+&yH}n3IU6GI<6C@Kd^p({h2z?FSo1>}({xPQw-8gkRbr2#jM+j}J(7|U!8Jy*9QYO?JltWN!@;=0N'
'P=DhcvDNCP-YoABn&Z<Khj8#>eQjB<IWGa-9{8dHy6unZW$?qE20l8hT{m=mb&pR9^A!aKjA*m#PAI`noq}#?H@(&MAg%R+HFtb-afZROf5H68'
'!>7rWZJ8~T@MoR2;kVI5tz4Ag2CR2*hN!A{W4Q+-SbNDIfM_elxVIs<v=GvUUVRv|<1S!@h#$6!3SDz;r{|BHgN`?5WExGKx$4?F-qBq4gBIJk'
'mUE7RYH+xla#+?)Lp5-fg4woxYRvYmX8RA|P1EvD7L>0i%q9LO5$i}s^d$}JnF*KW7^7_6w(7>WGO(cMYGkH6ke!-RHED{=k74#mt9Wn_Jkh9a'
'k0iYY-l?M`zG5hUs~b^OK2K)plBAR|h+8UH&)1g8p?eiy)v1!F;}a8=m_{>p6#t6LmO({<F^C$^U2odY-3|Tc0yb)h_x|=%!QlMIi}i-h$ZzO7'
'WrVzjJ`~1E`l0uxN29l46QZMo+sLX8y@*t;@K7(c{oVnJ1|&N4U{eEdT~WrfhuVJp*TJU>WESyVa2a2tpi0hXQwd3>>6gHE4=Pc3Zn=?|B<ZSD'
'?NzzM5jL^FO}ZonB5`$cX;A=k3h#1EWVV#_;}jsP#1I10Jf7(WshXIBB|v&PMAtkn5CTEYMmkQHt38jb)GQGF|5DHYK&j_{ihv+)EDE<HpQ|h-'
'?PXAun6kU+7)QlOfnzt=KVu^A(CoT91&V`Ka~{Up?am_);^$%fmDT??Ud#NjEthAO+j%ha^I(3={P>~J{511v=I7x&o87z3zcp|3HeSyBctn%V'
'Zg5v0{|0#2LJMI0=YBT}=rkfomtxxd4B3X1oho;*3zs>=F!Y4GFqGAe#pUY668@8GT|_8ys4gD7HAI%QA0u|Ec0}Q++N>X7lcM}diprrx&BQF_'
'o2l;6eq0%qGxQUO00Y+csvm7S;^U?4_4hDT+t?vi-)>{KzR*HNRb?FDK+gl}+Iu*(f0`4>%XAh`(5uiscA;ghrEG`IXLX+i7gL!VO}}f76MU`O'
'WXPoqm^#+}dhN{&>0m77Ow%*7s%d#<*qf<ZO>0}3PkAh+Fsr#^9koT5uab$H(%n8@&!KPBy08kktUAN+&cT`!9<nQ7^`H~!e5TWJTn@*}?55q3'
'%Zh25Q-$j00%r360#Hi>1QY-O00;m80AgKY6}YYK4gdfSH~;_;0001CX<uV+ZE$aMWnXD>E^v9p8~tzF#`Sjt{vWO!21zZ&SW4VwLYXxS+M#Gy'
'bZyXI41+n5Cy6yhviz{EW%=Lt-hJ`zNJ?_nU`A_`$9M1U-uK7F^SobKRi{PH5}#BoPgupOdCuCVq8VXH+K{p;wiVrzO;M58-;p=PzAS1MTwYvW'
'yxlR9u`P`c<lQ^*A#DKJPZ`bMy#suEn&vfW0Bm0*ESqm=oaS2+7dhaMn}CqFyR^Q%fUj-FV0>Su8LJ5_o90bHXinH;nWeF@k0#x-EX^6I3%%o-'
'Bn8QfhV1A)yS#vz+m_We*%c1})>bj&(|ikwPcnLdxju>Mo>dg~h@1Ce)r8PCNedDKmdlHZWwZed1pzw2*8&by0RcSEyS&&`#hygbrfu4aMG;AX'
'3RMG807emCjg;b0&7s88ONiI`!6!eY4Xm7Lkn)hwyh&pP^mkgbe<JC8@@>j8U=k-}bWgK1p-oZgnKX@J!J4QnfMN%Q;%8PD**#0X0cI9eO0%E%'
'=l--h-T}k(_P&#x{+ay_RA_wSFxc0ae?T(gMlLVni~`wz;Us^PrR7u~V79uvAn<dS=E+K(`^FR8@qFUtw5mA4yjv0y^+8PH+=#XyO)Ij=3fce!'
'r^hs58`@?~xCpL%vSr~F{>@mp46bHku0Ryk4Xqk?wwPc8GTfg7))-(?gw8;yAWvbeOU8N-_Q4Q*!4LLAPpcTM3m{0{RJ~bV%`z4@EQ!{KC`l`L'
'E-Ik%o<2qy%eT!gyuMP1*nOI?9FAVGO}!#nS~oX<^wz~1f%*cPLx>O-uElbO+(%?zI|4ez_t2voV7Mn21bjpSu;>8<qIjsMu;a~)%wG|&6CY0s'
'4gRDXf*?W1Ch}Ei4GE5PGdPiGS~K!1&06+NRl#;VWB$Ia8?t6(3#5eYfuRC}V<J-<HrF?}syATE#Hk&a7zVdsp)>bTF|aZ@U(?jLH{5=G&kh@y'
'#risj*6z6I$299fxT>{~s<jB7U(0!qV{@+LocUF-yymm;LC!ind6s+Bx8ObQ`sX~YlBZPhPSQ}Rflb*?Blc;oW#7_-ml6^Bl6!8>Mif*6fQGVt'
'xubPj+v@pTt7qM=e*^s$c>Y6vL?`5;!$C(QuRdNSE1GYap^ygAhcwbs#j`X(I$R6_?#Q?%o+<vnrgcj*?uC9SS=^Tk0Afv)CY*$y%8K`N!SF^4'
'4b_wVL8yl`X?DY_IgmS+Zg=_sN2OzM{=XwIzYO*c@fOTmUP+4O>E4oFF9#gb8y6*nam}a}yRLX~uE*N*&?l<f5+jgL?yZ#g{G+1fE{)%`1D4Z?'
'R(3^khp|cl@o|9<S@FQCsM)1eg0H`~1*lI<#Tyva3?TVqL;0T_i|=qh_yVHG|Eho%pVu_mGVk_}GG`5YY;1Cm8m^AOWkzGRXL(~ByeUfD9&iSZ'
'o3V{J6u_{mBZC@W7fn;_MHnYxdQzn^bGd>^8=COHHP|2)wPp7V(Ls)HrPxEo?itMlc0NfF>wqpy<CRi?GiZ#vz^4aV<q(2vRw>qSWNOA4vZ^Ag'
'C}DAm>C>3){{Mznt{v|r{f{x@TY(SLr2pNDzs4B<93_+iS^}v|0cjrc5Ef2je7G;tm=*UzD(lUoQYCs^kCL{Ma(T8Ll`}ssC&}oVWlwYH%d4}j'
'sK}H=1~?>?9lgPw=OUXMahk_T<}f7|b}mV4o)dENKcuDL4sGI{GzYEHFPG$Fv0&##uFvu!GJ0oQv|!P?X!C?tCrE**F{ZSAB0!4&1`_@}1zV0T'
'l2y4|7l|RoZ}%)A>w{5`z#788UI)BHpki!HucLcWZ!zV0ceQv%(>Y@OdMUGCd+a#L>||09AcpKo8JsSkIwOGn@`XesVzh|IXm&Wv1HmVlFpct_'
'2WpWhyGWEy@b)oir5REfJ1h4jGpizZp+Q*rk@%EhdIGk;eyJR=Rx*XnR=bZmZcIy#5F%2#&aiT@E{Y5gy{%eiIKv1ebd#b}lp$Q4F`7qA)q$i`'
'-BWl$XD=c^$ApQ@PiWwBX3c=u8RU^|4<{B7#%M*7H0I9F$I{pM!4XV5GvQq$F!YjqHj<|>SnIrSk>UuFL{nv1c&TgEFMio487ZmxeS*RecfCzp'
'W=!lG-G66YeiqJ<!dbx_fL6l3!K12Rm?#=^gQDw{2_q1Qsg+DClj^gfFJG*BZnwuIUmh!v3NNalJ!~N)2LsrWLr(V)w3lrj?;w}pPC$%#LpYuf'
';E(}71ZI)65SyZU5JB|#-sJ3!r+E2ZK71IRrnn!Owr$!EwtY9c_El1!2yK$@UFtN<ddFKSVKA(ybpI<&D$d(56xjjLkQ5aUt>$8b<SVia2KazT'
'`ZyF*YBSs`8NCCwf#Bpz@&%cK=UA9LdPNq5Wf15YSq8igAy`}~1lK1p$SEM+(?zg+>GJ_qDM^#E5Vl7g5pWtOo{`T1(e+YV1Ri(;GXZ*zE5F2*'
'IUmTM$x^SocxefVSvBeo!tsavx8IGz+9CXJjOe0Jcw<MzPTdU?If+hG)I<dMC}Nlcchdf842mEM3DOF&=(9fo7apYB%hEzao{{AMs$1i^t73&h'
'j^#JQlFFV9Hz-kwLDZSH6hK$Ko`MRGjR3*wGN{cx6jf3WyM!m%YKYrX>bfTR?ehVpM!8pNtfEFER1~z}4KLXj!(+-`7AI8(X5JrfYFPoJH2tbS'
'6$!oW`L0SeXvC3y1#oUacyoJpz^6dZ32QXkwyFyZ4?xbu2zC;=%7SQ$I2~R=?^;}|VN;}8u%@*R<LjAWepD2U_Jls7`MK73%;vz{9TeuILxdCo'
'Bu2B!)+fwxLWKig9DVDWQW~loa$wCZghTrphG)hOx*-SFDrJU(+#?`}Px9jnO-Kh;D7s$O(tGt|l{LbfjhlA8=j{F9nYP5ByeX0aElb$e6kiMw'
'Q5p_Ka?C3f9UsuQO?10(B8<N6(<K#1|FEVq`AQ6>#&mKdPHlI8IA*6N0>e`M0ufOKDk!(%xV(TfE+91o)p+1K@PJ~^Etz>jQ!2g)UVz7cxS0Tj'
'Cb#g|7lwK8qQ8{2YJ!$z`VeecGo9da6Mr(9&4LPsLFi{#_<>x#*-fXDVB(X>XZY)H_-h4lw|(@mTX~p)W8x2ds~Kvj1iL}cFNr+o*+f2-pJwyL'
'&=Oh(0_}Zo7PkS~NkFUFDi)ITnGcch80yZw0nSOetI+1ClEG>M_>ppI4Nkh-+06=431`Qa=~NQU-)s>Q0i!cl(aI5|&c=>QnGpR6ZJT1!6HJ9T'
'lmG*~(ra^*+pQS!1QKJ<=|q)uCWuqU)wl}`*RdjkQx1HU7#tYZcVnrNleldrvojTj3+lvhEECiY2Nqu;d`Y4Wqj=v!)LD#!;)=)4*`3A)GS*e>'
'>tk^$X2X(2M@+E#;a)}TX9Pl-st}bXSU|L_MxH(Q$$)pf;bJuh^)SrB8DPR8K6^ehq~$Y7L>)<wY*KoO@p@Z-nj;COCzk3Q>XWJ%$f_8j<?JT8'
'T|dV;%a&irrv~cvsOuD<L+;r0p8~4yh9BjZM`!#X#}Cdp@A1iR{Mq>~I(rg8yf_fDcNZ=(4-=luA%g0+C@QFT2TnHhB$Rv`4hsZGFDNjWX+(D%'
'*pBT4Cnfl;V)M}oo(d#8^CR;4NX<f_^@`2L%yR!keq;|s^=WFb(EM5j#x28<xJD*i1Nok@(nM#(^w<S_##?}7gT*JXLhv71!#u27)N@1+bOy6V'
'HZ;rdgHP6NW8#w+){++2$LnR3@H#o(5YwBATfAteCaK{FZ3t?^`?O(u5fX8#!y8hYtG+tV2Zu95Fpk9(X}AureV(^V9i0;qaPyUg0de#iWAD(I'
'!mDsomD(G#0nnWbELL5pfd&-*;MKu2S9MPHqYg4|yNxFA5xcDEIb!vMov6^rO=Y70+)cv`I!H*Kgpa~uk5(*<Pa{COCo3TJ>j-$K_#Avp@4I7x'
'w+|&t;qkc5=Wv5*fj+vG9tdWeLr-!JsP-tVsC5yNi2Hoeisven=8k5pK8<k5?*B2#826Vb4W>>kg1;OU2>Zw<r9w35PYHw7P(LOLx@CUc`4>P>'
'3<JjqbDF8#iW*!M7b}L@Dq=lhhF)R)@>B6y-tIWYXl)v`SzHD$j#+}EF=5B?jvl93F0Waf=DN6Aag^Wue#&uX$5~M$vqwPtycPpT+i8Yt=ao2~'
'qk219M|_$$<2;vMUxu3EX10`&_ujr0*ElN1N#lJmqyiltukxh1T?XstN-*5j76j}>2poRt>N+5KuH4zDRMMxfGYFaLUs}LK0dZR~=&6*znt@^d'
'01AF5A$0cbYd)5cHnHj+Vs?X`p7O*<GIlRZa$F?q&11wzOtPIm@8(gQkxvb%+Lm~|FS1r#z+baWj&ZlEb}5Er6~@?_zTcs@r$?T&J@cLjY*N1l'
'CJDPyd%xW{D(cN@@$%}nqp!Xj)UBaTU_`KZVPz<8J-Ksr)rHC{5}jyB&%(E^@@5VlsqL#Ricxt<WTsVBwOgG;>%FIMIjz*ad?__};|A{o!G{4Q'
'<ywg=;=D@M;fd}%$|>Fle{3zQeP=Fa)$oi(R11nLZh^&#=JYDP_G3MCE&$57;f3s0!C7nb6jOrp4M-l4O-8r%k@ekny19DS129&yTgB?kvK^oo'
'(#e(Vf$2*w{ZhJ&Q<Vf+p{i*&L-WOevC3be^=3d>xqN878BofLhw8x}QjV~DG`<^EoX~i-c52_92=VHKD5<!gOdj?=S`;!bTZBqp%t4a~f?J`g'
'IQPD2F$gv9I~ajH)M+qY2Ta<1DQ^P)h5}-4&AdPnYV{sy;t7_4h2^}R>So?bi;d4cp>bRIcq`lUyl>H+Qexfpj*AKmL0RMp=DEBpCjWgF_tWcx'
'?udzh7UEoO=(a#!g}jQacN&<oIXl)Nmwlh%nK#`Mx0d?kVCoIkaRNwjI`mND34>RS-%^NP=t0-DC>0Ce1EFN!r_ab&ysPb&cM@Z^=y|yTLVu<y'
'AN-b~mWKZVP)h>@6aWAK2mk;8VqI^-2**JT0018$000yK003cWUt@1=aBp*EUu$7!XKZCIaCyyHZExeo5&mw#|AD<ASh7da_N8eMxh&w=GUCQ}'
'Hf$wn>ly?_t}I3r$#R!EyRQFxpV=2_CDB*+TYX5xUCz$T&if4Cy?b|6@xgEP#H(t_Cv`pd^0rv=*6T!3J$Q1)J^ok-NsF@1($YImg*>^w@Y1F!'
'i!7BzT}AtQ`+GNCCA^~YQg7OFG4q<X&bSbv*YY$MSQg7vaxbf^Y4Omt>L5Vt1)V5L9(rk&dle_*0IsCi-<$I%5qV=UA$UtKTAp#R2SQQ+)uJgm'
'aPn$dv~{)MmGmZjTDROQ7ERqMuG8wtThw`1asfWyy?eL6r>G{$w3A)Slf(c65<*aXfwbBCP1MzDQ44dnN#(38CidVpmMb8^vzE&>22p2oF4xO$'
'VkEQPdRjlrV!>C7t}Q_&;%!^E+nX)_XUD}F@4t$Mq)3|B-^=;bGXM$Qz@MZ70w%UY4??BKr?O7-_)}f+V7R~M;V(~R>c#5HQQj>Y;TxzR;#F4X'
'+z-5BYL9w_AfxdJgU3t3)sx0#Xam_Mx6zIMRs6iy>=4zM{ug>PSPE}N{LKD3fQb!$8vTt8iCuPDR}$JCjGr3LhRm_O17erG1e&mHpUh?G?{gt%'
'+Z<q7H@x!AIwXPDos7Rf47^l;bJ0Nm+`*zR1XHQBXz233g)?;dW5%0Jg+)J($JZlzBD?@_Q082wMHyo=BJc2nDCbJR>ZFKS%bPOII1BEEN8cVE'
'-mj0gX#r8g*@$t4-k&le<vkNR!t$cs>Y3#GGW4D`wHFrXeSod4ByF=994cu$h@`G%=s_4wCxtZ%ILzo#S=h9w(o!fdZTYg`pATTjOJ1c_#yzb-'
'pf0<dyv}vWWm-bik#apk3IE99zwzC@wH7#aBV(d}z{sm>QIKvE^CnU{*1*No!bgQj$mIRNk?d57TX0FCXr(UU+cPBYMDj=J2SN0}rOy;}vPi39'
'3VUGT-F>j7e8X=9tK(Jfvwu`9`mHW1U&z*1#}%e|;DI<JG2?d*1~$>qpCwo1EM&5fCC6_nWm`--x-nKTY!1U(KP9j#tVG?lnO+2eF*{$2?Bw=#'
'G)_)$MrRk}+eki27DS)hLUMgmFuY>f+(oncsfNZRZ=qK$G=Z?s^9ay}fH4oHny5)zsPjM(jjD9P-?M?XmI%?u9#KwbqEYnsqB$oO89X8T%)-^X'
'i|gcUbbfg<9-S%Q)3~;jNpwh;{_BPjGMHnI5XJ3U+h;OE;;P#xySpnUNEHFU6YFz2NZ6(JGQHfk3i2*(vc%%%I~676tk**ApOKu>ldnP{;R!Zx'
'kA2dQ7C_#nQ<?n!KwWB5r)_Q+^;WMCr`m$U;UFP$S_804-Ttjkrlm3j*o?M>S5R%=#4tG{4h5iiuFd12tS7$VKIpstj)-*nM?$gN;%QM9)x1}D'
'{Q}fSTV5hFAg0M$@e*vw6yOVD*95vgwNgU&3GR-mq9>tUb|3r~EvqZ2E$u2`MbzaIZ({91tM!}gL1A)?!XzQ%OY0cENXfBPU8NYc*!5QIL_u{g'
'7jh#W8vm`^dF%k*G1^J;qJ;pC)$3?6$I{oA6!Ez2c&Mtxq@JslL8U|1CNdMoj6fs)*g$BAI;|h9#Dln6WYWdtq?5Q7ky#?!HBbH;M5)0grz5<5'
'!*tX;s8FX6w?Jlt7F21W`qK&}&KxCjwX{W_UL`uYtsxywqpm?(<o@$~xYRizT$15!ob)Z-AR>bky2e_%)z!<c5LT!u1<ivY{VW5Q$d&-?K^8v0'
'tSDK;D6wJ}xjOU_Ve%w7y+o8oEtXI%79zPtq@2cAqBNy$B<ne^C^Lz4%KdLXyw2|~(wOQyy?uFgdUBcEUj2M?I!ewiZeqHXwin}2ffLE}nUl-Q'
't6!6!KV4pYyg(jFZbth2ZOkOhhlM8>$?4U{>#N&Q^6S;jPb<7+$1XYtPV;dN)IXe@{?vm=6FnP^M>iiYK3&|7FHS-D^y-(<&EH}+MU}*@VSVNO'
'P#`GORr%z}8ZA`aOww$A4CQ)FH$jT`WLH(Z45?wtyG*r38N?QngtM~lat}$V<B^kIbx#rLA73R;dUgz!meB;HeV-kD_%1rczathN25@0A-&g@a'
'(ez6(#t-g}Og%=$c|2t^DVyQJ!RqwznGU{~I;^a92#HCM2Tb>82s01s_pybERK?9wD-_cJ#p_-ux>g`}XoV0Gc-F~uf|igC7BISQS!8uf2{|ni'
'T?Oz&$7PgcL;UAvI5rr}YAWR1dt;NBXCbMco&wq;*3oYeD7$x1WS?^#MH1ziNfH?Q3t`f(rvP0}%>yn<kEq_=JKc`Yu6`byJDY5SqW9QCznxG%'
'P~F!tTEdLrr(GGJlkN1iGCATjvLTOEL9(b38|tda{NVlfhfc#Y1H0Zf$e--t;cositTuI7Qsh|&ot_NU(|WK=H4~tDz!W_O&&kM9Q?>FKK}a+_'
'lOB6dLbtXMY$egnYpyhqHtD1&QS5OM?{>26V1Y6=JxGgzv4??sf`tew!7kPbvfX@h+ca)I+?p;Fy0`1g(n_|X<h=2Z917I`Znum|T4UW_?nq0?'
'sB09yoQ&fw@`A$Hf~TTG5kIKXRu54I2*YY<!IbhS_B*oB!F<Y7nstc2_NGs*w|d+{RRh=qvn5S#az6A{WIan31qt`MDqX?=$ZMIyu^lty$mgpP'
'z-+?hoMONU_CeTHWwAh#$`dLHXwOs`uN1P@LW(?Qu-%N8<p5&bNKZ5V=pTJ>#R)fjlWi$;3hgXp<2+Hs&mGgx?EZM=+bS;V?bW6!ocd}d`uhr='
'r>ujzG=!Hz_1lmxqL1|IsT#-G@e4J&tt_S~OvaRi(HfO+NgixAH5-lL?xZI~#-JK&Lr$vr(NzykHx^u`YE)zjAsC7%C|_M8v850EQ!UTyuF6Mx'
'Y-O<+FcCk$yw>5NugPZA7`JI9@M@uh^8^hmip0s2Bm+i%Md{WK5bNMCEjF5`JcY0_^Db?Qoi>G{%!-3MqIe&hrN7m7DR<UUq3=RFXxO3<G%IEO'
'!ry2iUlIN@t{`Us7*hVT@`Rr#@`4R5$u0Y53%F#7_oy(p>$mxO3<1eH#mkT9$;Yd+(Iqv+>)l^2ZZEDrz4CHAzBSvy%S*3`sDZ7|{!&kdEYwCH'
'vVWV2F)SLp%DJXxCuZAb|9WmCq;4T=hII)%yRLE=!*?GJ4>$Qh=U;ciWY!YqVA|LC|0GSKd2bq?zcoGT`NZ2&AmlC%*n1=X-4uwTACgGhr+96j'
'X8s`;4Le@Ff5l8StfPPDgc(@+JD9cMZ!&^3!OPg-)0)Ne12RYG?zMahZIDF+CH!yeHI(vz^dQ)_%%Ctu;dik$2MU(k>K-`8@#pvt-}Y$ISheCU'
'{#tm73wp4m@yxzfwE8W3$1D}>v1^vwZ0%fSOG<cVK}=l+m0Rp>_UGbIBWBYIkMaUB*OYGCT6&$f{qWL9-PXD8QM0FtkKRcTc~k#5<_ni%r}p~~'
'eZSJf5l$l{^*KpwUx)sjN}Duw+*1%b1>QIRdx2*JO8J9R<z&BhQDLkBk3BVU^X$ly7*7jg9D!QL0_#JrUE4vdJFmBJp&i*f?O(T8kfT4_!enTJ'
'vem`VJfNx4D47}}m_<OdP&H!J+iSOebqX6eX&owc!}c7~O#0F?fGS-@nl@4;i~&Fg=6j(#`rkeEYCfe9sYc~P9W2e4C1lF2PfIUIvm~BXgfH@!'
'?RetQ%VtQq{$c$`q@ArmttEcPz<jbkp$AX*s=8h8I%j=zaokl%J5K*%wO}ZHDGyuEh&-CU3uIp!66-GjyJOA1sc6t2&<y@_-A+I<a7aS$uxAk5'
'9p2k04nEst-GMwqz+c3EqHPdp?Pr4>X!R|sbUVl0J`zS~&35BkrJXacgF#Qsd8420@?j_w0e?+F5$$)0zAn>RBkJ5mK|}fNykKv<g8ZBo(zo9-'
'kG|~zH#My@pRl3pu_+LsUJ1MwcwSKHbKFmF+Zj;JL|KWNg!ra^kVukHy__V-A4x(Z>m;%HV^eo(oK1yr+CD7rj%>;>{EyTR^`ARmRkis{Z4fq<'
'*3f&FPYp-Yl?9Pyf<B94<AwDbq6+Q%d;bAYO9KQH000080001DUA}qY-m?M#06zr)02crN0AXogV{dJ6Z*yf|Y;R*>Y+qq;a4v9pZB*NC+CUI}'
'N8%qw^ueYgZqhzfsuZa-7nMR9l(ed<s<IY)3{#d}XLb$Y?>lSbU`3b*JeM<b=H|g*FjdQ?#xra59#mOs%9JN-U}c>rvjo3Fz-MZN@dPrvDXl|b'
'TY2Q9-|P1tDg!(6E3gJ(XL*f0f`zS24&K7W`;W=F{4Nrhv(2ll4lk*aQZkAUc?VAwRygYSGF8^*I=UNy^On^TA=89L{DQyBR+v{B*#V<hWCBOx'
'Bg-AJ{0X(eV9@UsY&XK%czpJn7C^$EJt(&~*+IQgMv`#9cd%J^fR)OxI+(eLq2St9DCbp4t?gFC@qF?ULs5EY)xrO$NVUzBPD5?j!oORO3v1Wk'
'?BX%#Lz;f?^GTMq5^^j+Q-Q8jCYE*%KlOVc4;5f+s3BRRN3)HiyPNCDIK7%p=BY4Dr#HW&5k%22ah|D+TVQxtk<3;G>S=>F%$85VVN*E|3k+>7'
'Xjw7lA{C(==?VL)jD}52VIdTVt@#s;C+XGw-TeOXCY{}1jc4iYcsBbozM4jV;SwUx6-EuJ?vLnRHzf`Oh&z>aLjJmcn5Ng0hfvxEb5iL)yKvEP'
'2=5>=hd^gRm$)ZQ9zV1k>ukwJ@j4}GG;>vY%;ST#WZaUioX|N=FmcU{g2QXAIMxvxtLJ^Ey?a8PH)26XY27uxfuCE-EE{mE%I9`xnm&hVYE)YU'
'uiV<fj;UUARR~#Vax&JJ5^g6sM(nOS*`vORBRu6`;tv0Gp|bUo!{w&6&+v$m>DD+1_ZDm&=55R*Qzk<lcVM!Or@Ylp(F?I;DPu*B-`1X^aMcx?'
'6rVG!zXtB|hwMft=!urMV`}b3&OvmClZ-uijhE5m*YgW#Hvwct1S;Fo-q2<HGOY0lnM_wXrp<@0+onb%ZvO#LO9KQH000080001DUF@E59xem`'
'0O1J$02u%P0AXogV{dJ6Z*yf|Y;R*>Y+rM9Z*pZWaCxOxVQ<?u5d9t?|KZ@lkTf{LhGAb47+IF8t6Rs;Y-MX!6apokY}OK~lGKu*=zrgll3m4i'
'iw)xkS)`7~d+(l}JQxg?T&zkwdyr~_3WPM6O1Xg=mFC)@$ne6h0t1D0$ug`^7$~HITFJc043`2vHrSvJ2ZO=sNugu~Nm4YXQJ5sat6D1BFDMJM'
'(fah{^rTz;prz=xWKtM>G$l{BTe_8s3ARGD+oM00+~7~Ux#1OdIcjFsy#%*3wTGRaWF^xYW*xfMGF=*}@ci@yXi(X25NiUMA+L=FmMKZ$8r9##'
'i9&6dqUA;tS0Dw#^7ko~068-(Wg0D8JL*~%+RQl?L<JYzB#E!FEJm)xpgY(OyBgX>7`u9X7jWb`T@+CtR-2qFzn$oDtQs5vK2om|xpC9p<_AuB'
'U513n6qp%50Hz^Ndlk4s(-0pqYYh6{^&+~8BZ#kloklQu3A1?&(c8%~Ucv{NYVdguar8EZ+r{MOYH<&5qWck~GJk59ZT#$RIt{#F%fm1Gt=y5b'
'Ez1_!rzKB`kFL{UKge-0a!r?w8UOfm53W&9d$=3{2Wahh;#lVN=*yFW3s#o<8Hkyzlp6h@-_*J>36atDl<pFqk6^<ET}dsb(YDi1rDXYdMv56('
'6*Z;lCa9ED?!J#%$p-?Rb3$q}jr-}w`&ql+UE=m<o7!DRhp1^ZHOUt9gqg7=G1NvC_Qv;y_h-ZEY?#NxSLef<^WpL@uO&KMRW2e3{;#$svt_i1'
'VKR&7t(g9}n%+fA@GnPj8Nhr7*YnxS>Et>ll?Cu(4tKXNtQwb5>?AhE$E<8}%tJf#jzhG=-!{csOqe%St$iZ+83cvoQNr@x@9hc?d)sf*zXsm*'
'vj=|}O{41;2)ta(Z(0uid=)Js;Q9EH_}<nX1Yv<DTa!lo!;&2++kXe@-4E{r;DtL7OY$SDtUe^?&j||m+X4HYsg>^I`VQhXwpL<&PSNW{w$^zc'
'vv!}M)@P;6SP6#d4VX0rt=7OPprCgrW!Wa|)yg>4vS@@APh=J8ds_=`k*h`k7nFyS+qw_Y_82cXte+b5)uwd3>;tcu+K~Se5Y^PA`U9UKv%VsX'
'vaT_Q-)`@==R@W`Snlnrker1H-Fj~MbYVY*?#r`g25g5kt+u}ZpG9K7q+6*vT@W8fB(r<Gla1F2gn0LZBL>&1uO}3{JvfYOb@R?^4aR$K3;5(w'
'X9P8b1nD{QbMNz!GrDtno5$c4jF%Mum1t*E6h_-Z+fc7v@d6+sK6xz_-g(E#=Wk^RTPyQBhuj|QEOUipuh;i)7)5Wi-uXqN&;I~megV2kD{lV`'
'34E|6f|2xC<P^nLj0^7w!8?*Vq?V-i_IPZ+`<?r_mt|*xm-~+K+~Ibw2i5WqP)h>@6aWAK2mk;8VqGzbelf!g000#t000yK003cWUt@1=aBp*E'
'Uu|!6X>V>WaCxm+|8nED5&qvZeg~A9hSCYvQL-fa)F?B~`8>%qy=3m({21G#fhb7gnIc&bvaP6khQ3hWsjtv)0g|F9+P!o_{}2fRi`~WUw;v$K'
'ajukD$`c(d<&-^0U&o5gl}HwWpE40m*?ka69r(-_i4Ni@<xI(FDiykiu`j}K%_eIW1{0B{?c<~4qxUhJsQ59JinF<lq!Jo^<&*SRT1-O8{CFxk'
'!#p!HbsGGJ_rX%kB}?T$Rx<Ks+GZDFczoo?v6==EBtRJ@Srn_K2+aT;Grf?I8>Bjj=B(qLdrzG=U5{OS{K%#<4dxM)jvfT5<v7mq(M-ilHXhGb'
'dZpxej5(87X;>M>S{UO<|EyKX;#SKi{Rp3A5)z>ous0w6BOYno@zGSym`vd)*`OiK;qei}^#%KTq}3`VH)tiI)K@{GUoYW?5Y9-}Y>AgFk`Gd`'
'Ao_$wFgC<^R>71Uwp?N<$PSXHB=nD1s+FCltz;4kUpAcETZc2}KmY!hQ)=8$Tcg~vrIu#7<u;-wH0})XvAXw-2NAAhZb-yNY$=`^9%3L~MN|8h'
'y@Kz(W^+%unX`%5>(}gb;NjX1?Grt)KjUNQb@R4Psi}>oKT``mI6GaIA{jp-64HkG&k-l{rHpi%Yo=tn3N>f`N~H*_5xcqL%xio0?F>!X2>7=c'
'oQ<U`m?2aV18`~fmNOTTKXvh$#w+DZx5-8$=0-v(6Dw|MI9=J)tx+z&ZO4s~Ehm+xR12jm!{Z2Hypp?C(wfXAm5tez2?b{pSUL-1p^aZ|93$7c'
'Yx2ETD1f&8!7x~`uMc!J0!-Qq_En7`6&2`0D}_Z6H72}fMDuV2=B7bkGf^XMx0-E{B<R<e2EMff;z_-c^>j<rdX?9UthpdN7xml{HA~w6pX-|^'
'E097?8+n}G%k?M}%gI!*$02*%t=ul=A61~`co|cyje)HVf}Np38-ifqWC}sZ--t6g4fNRFnn=mh0;u=FfHi>7<c;usN&+Q<ke(u-E}>GQ;yF~a'
'fS*WpV&YCNlZ8kFi|QoqgRvz=tgetg%zz<4i|&Q5SEd%oWuPGvDne0AtQR6OTCz|^bG^tx>}^&~%#=mJS%?QJ0*RoaL15|M+aB^M2?IaSES}9+'
'`WR?`A#+g%+vx@qE+v9-<B&sQ6^PIoGUv;0J@4}D${P#_<R)=A&09~uxxBi%qE^H&d{K12GMBTx$sFCb0^4SWJ3Cv5!k7}&I5!^+qO<R!S$s#*'
'?DO?Hkwf;-YkOXcT7L}1`h!e=8Jb#EYWWo{C|uO?Gd%H8uR{#%I@MTqZhq>un0cL=d9J7&>LGiNI?CA<AS5d8sH}4Kqk>s#{WcB}nVfx;z?o^O'
';fIwbJMzbG;PB!ej^gY)je6^IlrV%5b%hYEf_sUPxCH@_zhs}k;xiFGiZN#w(L9tSLt&WFxR?vQ7GfGdqEm<(ZNc_$Ld}Ed`~2oZe)AJ&?;;Qe'
'2t9O1=BP_K6O>h#@{{;!#Xh3K@<EsH*0FlqF+aF{<JpUMxMzMkp8YfT{1pHD{M@78P8WYQEAr07|K4hRFV@6;J{mZJ`cuv_57>2j$=2Bn)t|F#'
')O9C|vn<<2%m87YD&w4mjZlp<ZZ@{bNG<6CETr_DXZUy`RC#PO&uzn9sCy%{>RU7=l;li{IpCtqY3SPZ*zp1jo7~ic0+rg+m93ekb91lk+Dc%C'
'RjPr>ue=D%oJ(YLJYQkGH%7xP-&l%brz{?6hAq!yB3oC!*=)#&?RL8%Gq%p2;R;S~lns93Yz-{s3|Q+UAK(v+Qexy_q0RK8e%TezdTs_zCZe<3'
'+ayjKn<r?w-ZpP<C&&x$_7<x|A02;VCfTA}$7z|F4(iy<SuzhlA=^A|@k#$T3v;(H<`jj>#0yGFDFmE5DqNl1=3tgrkakY%QO1azPi+aHMXP;N'
'!NWZ8Qi^Z(rxA02&rZoPOIkl09~BB`=D?-_)iNh@yJ2(i%LJXwXGZ64?&`Ubonm;vZ|qo7x`EQuFpo{*JU!ZbP!kAm`%{caB`W=R<B~-8u4$!{'
'XgdajShx2QugJ71%J_*HcyrRZ%h(qrwD$sL-_ShD?R<YJnY};4wC9~^#{FC}uZJ~5DiiGc(N30ITV<JFu&dw+E10b`n+OVPKa@hD7Mm?tx=Lsx'
'M4PaOAO#IDo!Yf*)jGzzD_oVWamiQv<7{M&UJSg-(nkPFQ<<^Xn#<DSH_1u6P)!yJ;pZ?JrjkLYeKF*kfZUSFt64nDmPg>qh1^k9uC*@M8T=fB'
'p41C5TA*QP#YtyRa;x)_>?DxZ7u8V8P5i`*M3T{JX?E+`M9;8uP`%)RmgLZrCfPobx&bM^v*sRzF22utd}3drL(Rrur*Iyx!2Qj}0Hk4FfN#fa'
'U91Ym4UT!gtMI@=*2=fxCM)y2ZX?Zuq|vOI+2VKC+Xruh-F=JQ{T}LnzvBVCXX&78x62kitjcv_b12@g&lNDHRq(NmrWGhEW?tq?Ye1Pu)Y@=)'
'Q;df!jFEJ7%jkT<Qss1^g6JOLX%7_A0HLG7%(YrYKAohLGSdtl<=%o8ZByj;P=#8R6lqMUz&36WX%&xm`(W-)R4|>(y{34z-)VaQE0-|VH@goV'
'XmcvGgr5UT6F~46sIzL0O~5*djW^a=0m3Q@P=jezV0L)5j31=gi-Eeb*!Ndho|FY(Q1N4wdE<oY4lA@{X&CDYUM(qUU~0p7h4y&#X}QKLP(-h?'
'+$m-!QKK3U4c)gtyuW_;%XOWvhVJ#Z?-)9~fA{wKyC2@W&+lLlylTf@U;Opm#m^VNy4#!GT2<2BFr2~yp4pUfUZS_ye{h$wsQb3_O~-S&WjUSZ'
'XURDmc>IJYau0vq4xUWD)|oPMk+0&*Z{J-alhkh*b$hEp9XHE#%Wc!3>GE##8926}KBz;dXOuIf9y+yxSEmDh4r{y5t5;U>@|=?0f9iSrDx!%_'
'ybhIoL4p36$KPNlk<E-5W@6efyNWJ}>Q_WrL>llzX>N&8lH}eS3reRR{ZINCyL4N|j2t<$`3CuXS-e(eB@_Xf8s#URZTe?U84)3Tg3!7j6XsYd'
'H@a3US*!Qi3wl$68pQ{1>}G4wqcA+}S~Jei3wpe-v2t(UyB2XXHLH-D!L56+M{;w--S1sJaSO!+C54J*0>$%kVM31@DEi11?KBAk-EjZu8bVR-'
'dv}1Gbckqi(_cvcJ`7S#XNq**P8%F|Yfz1<yPdsB#j$UX;KHy`yl8$k>YW-@%p-}+22>JZ4gI@aFQ4BW@a89<8&}5(KnR3Qs&Oc18uObfN{F^E'
'X8}299!hCRw^!%hX9YY`Y|RN#78#~^or})E<yl^Jd)eAjV)n4Mhli7e>*&n2aMth0`A(m&TYcKU)6VpqFHVpqmE_B(=Yu-1LFhd3!r41=RPT1W'
'1YB0A0?*MV*ob-6?((`i((cue24j&ZfyXH<>2+IH5~0_Cem~O`y7HXd37wf_>}4R*?_*oq>i3H&wk8vM-Th4T@=UY=`N!s7b<HONWzsE@!_X2+'
'jh;9B+z{*nS11Tc@p^ClKI|$n!ET1Ry+Pq3%R%kYQgu-(r0S3GR|bKIxEicx-7{>rYSEf_`3-g2+f7G0JHtO49IOO9p<Rm-gDy7*J*Obb#El4N'
'UYi!Ypk*q6xmm{MMJBvY1bfiWFeUHHUa^MJpam2v4_lj~Tp*S+WSz4<821jh(A32eGl$NzrNZMGyj?w=3**OK{|>DOo&RbbFU@n$+qUP@Mo)p0'
'oFrAWAm!JbbxzN{lSPxrCTOTs*n*U|#U_3Xb_k{c-iD~}1U-S)LKbO_s4N|kq}dF{pqD5oV5DDm5S`y+tg|0(&OuVybN(PB@J<>AQ#oEGw1=_x'
'Q&3~4O3|k>)WQ~l139#(E5z8?I614-2ILBTJ>_H-pV9^>vv}QY8Od9|YuR#3uFATT(BX3u^E}Lkzi6}BO;Zq64kXBVEh~-~-Fe@9uIclT?Adq3'
'`btw(poisXv3Gpv><XY`wv=@huzY^^Kcv1fQZ1exEIrbzBq_%;RwvD#O!hZ$C6U8)Lpiw3;LK@}dpJ|h?R)20&XsR(4=~IrV_501F+02|A4U%='
'hI^%Nvr6a2lp3AEnav%fXy(_RxemPd*+Oc!0eM8^te;ZyP@LTaZh5D)BdGX4_fP%dJDgPbh`!rv+i&~`o6Q&U*6rpLyxFqf(97(rkdd>}vy=1O'
'?#eMkWr}AHA--nb-glI?<MGjd0Z>Z=1QY-O00;m80AgL+@0&^M3jhG6B>(^v0001CX<uV+ZE$aMWnXY_Y-wY8E^v9JT3c`2xD|dk;QzqtWoJ-%'
'u)w}Kod9W?pm|6)o1|!gBv3dKHKSM~YDvlV)bfAtcX*Mg%Xrh&yRk>);kohLHBl737h0CLf^oJNl~BB7A}gAr+Ot}gMS5bnl1J7Y1gqN}7TAk7'
'?4>-`QVXWU@2$|y=JMk5;v*nRnc{p_D3LZ&F@2D2nXyVX?7%^yk*uh4p_sT8>cnIXa!nyCoKbBlEG1cq%Zrk$Jv|F?l}mMGHFoM|!`R0|q1jPp'
'1m3i&(uOm~N6fTrRVskL$V8QjnDHvRyx8&d=8miEDg_o$+~KgSiLwSm;EFSS<YgH%abH8kqVclJ-8YB=8oXgCuh<T}wOV9scVczPs|tFGLB7g_'
'G9P)H71EGpzrBE7qA0q&FmfeH-hyM1B&>k;R0AMr#wb9O0^O<-{YR<Y+2&N!g}CJx)rsKEPJPOF)!<g#-e0*EZ)GMR%j-gvSr5pIB(=Ae==30t'
'yvO#&-R66%L^^SKk(OL*=4>N-1&ffX;N?4`mQ_z>eSLYs@B=Yl<vCPU;F_K8yq@TSpk<-xp6{f($xC@>OomRH3&sCXZV!~os!=?}eUj;LR*5@r'
'e9&|)0SHrgWsE2%DS^OZ(}K7x{=?~tHO$Rl8-uOsLN{Qks0>Tr<PN${MNv2Ax|Hw?bBn9wU<b>2QHi6da5J*4vQ00rmw>p%c7jV1*GxBxZCT`e'
'DY78WqY{#^(nWLX;nGs@%0Z=VlUUvL+rlf}fmr)ajFt63wGxo~RydejS)@WjIX!^PbAP_A>(WTW)hW3xbjwS<0C$vRmdRf6`cR|;gchZ}WY01`'
'Ld?X14^U*)sYli7B3)u7zh7EyPNhiT?s-v)v6hmlG8Eig+$ANVN!f#1nLM^yi1d&c1y#*8+eujh=rx2EBM6vf2LEUY2$uwypGW&z;FdG!0k+G{'
'_U7mQ_CcBB5Ozyg*e5)GAtGd$t4!p)Et@3g6cJBbf>=BDUF;M8!#Vm7TP$hBeY0|5?|SCLG5J~M8On;(V2>$seEG6wSI<dX&UL|jrBg+20QIqG'
'^9}yB!v3mg#BsA1%_<^nQ4FJBuNmSoqaE@iL&b|DC?2ysdhl6crwWljvWJ1fW3++q3$+SG@hXPV=+z(b-L$aZPjrtlT~ur@<cpK#7QUZBj8FEF'
'ehjNKo`e0Iot1TiouXc?2WScuhGK94Aqns>h&2Nl^Y{4-2-2x$0%p;Ev46k^1v?|b^+3>R)mSV}$AjS+jF9z++xi}ABPAFzjsfsbX!!jM&4Og-'
'kT^a|QLXGUd%k7A#KA&rNB~J}C|+3sLPIdMK@x`3r!1U2z<B+{p4|~aLU~K<BZrHxn~+Ga|Ka4H;0e5KWHi5t9>`HcA)-NPH2^~WDGHY=$;44Y'
'e=7RCTiK;f%e+A)B065z-JRGW7l<weOE+EGDlJu)aedfH6f0ed1lu76;evVDwZ;1LE?UPS9fZlOO&d?cJ$AeH)M><(u2#qdT`oA)9i0uAU0dzs'
'eTRrERJVg?W^=rmOk0;DJZ@xZibGwn)p3;`kP12o)D_h&v{LL%OzV98)iuRYl*pfR8cpkBxhEj+P#<=Fxr2&p^GkOK{;I-nt$}*q0(D1q8e;Ao'
't;sgM(q*;p%t}|DpbOpYpbID#tf{6$NkGLVCq^^+y`W*4rCbjhwvU~t3r)8JOvtm^3Vo2+*^_b1@tD$v)bBC%3!O11I*%ZU(AXgy@*ANRG{(yt'
'Dk2VoXC94PW3=W^l3b)Im-Vo4l$9(cSPPLl$0diZki}z$%9!de@@!+(*P|e*#6}Cc@s)$K9!LS0@AVejZ`Q$yz+(veC`bgw(Zu)r`sgXgpVeDG'
'5&&iu+5Z&<hGQF}b;M;;ia^K&O%P)79EIJEo<b1%;R1q`3e%v{T1Mic0@YN<B~ha6gc}NxgbF5WcnI8JYK4JiKQ6Htl(*A0fA`8Z8OijHR-@}&'
'umo3M;vBzb<6?nz>>p?(VFC2VK|WDi%;1epooRoDp69(4iB4w1hH4;3o8nt%I2Za#*Gl-_hce$@=)MJd<(tk@>^V+3iTf7y4tGHHyltKpEYnqo'
'8ePzv$|9+eOpCNd;~cZEUvtr*q(TcP8of2e!`H8-c=`Gjb@v7hwAlWH9Pkq@D}nZ0WnfFz`kAW=(INGHd}=yX!-GQfrOo5K4>Ib72JK%Q`{YU+'
'iW68L)vWQTb{yB~cF&gM@uUJLJoAEaJ@3Q_@O(jGgO@wj$AIC?ED(*u&ieZ!&6Bmh*U;G}f&k``X)|<+#>J;WD%>90!$jPZal&^sdMl*61aLYs'
'74S&^Ou$~|y(g`nBeKNWBIjWPyU6<(AZTtE_=Js7C$L(X7(kB?k84}H9=NtRsAaM8D1n+G70c30$SyNHmuS;B+s3C%%AjU8z}!OwMlwDXgiUWI'
'8`OuWl2?(Ls@bqGAhLmh-ErLytjZIrYMOdvo5<Jo<w*0x@uj@Bz-4%5QD@<qpoMoDn)Ke|(jYMSUev!y(*c=Fa<fQstw3!0MR7KzbvNq6#hcQ`'
'h&^hhyt2bsNH*fv(T>>Fq~s@*Ozbac>jMC~{H1Y;G26){p8;G@%Q?=zLqRVHkkTmnYV=ezHOmTZF5S=BJm$d$##VlY%|DQMe6MG9GPS8E6$86K'
'pp9nr2I=R?n&Z*Av!S*aV<y9^Mc!sGEIJ4!4L?IZFpStbEglM`I4r$9JJZS7MDufcXZc-@_~D>!Ip~ie26lf`q}BK)qq?nNtwr@j$`}p)=x5ms'
'r>ve4b0OUN;>E_WjMqlvp@rx7L7esAivf{9mqOtPg@s#oCIdHc@x!KL=3|Aa^%7Z?r=b!zB#6d-4?Sx`#m=f{WxgAc+P)Z$HD~k;nX}QRr^!CN'
'{<ldsnY^oDr5WN4Rkjn7;|Ls>H#OkPPH1;nR2Dl$O-^{#W;1D3Ng-KkO6#J!1z&HHm%qM!_v?pO$=iQ^c=IxO_u}K<?ZD2@>zrPP*gJJbM|Z-b'
'=YBMU9-BhAGu+Y`e#xXJ7$2st<zn6SIMxLd?|5=z6cWGx*wC$A2M<rt@naeT5B;K*G$l%6+qfxwIQqICJtjH2Sv@=8XIR%mr`kKq1a#C6j~<i3'
'siueQKnURKaB2=f>w-%N_-JazFb&1?cMhR(pRH}?1>C>TpIp%xEOV!<C%G!-*|dzIq~0iDDj#lNZq}+eLPl0`Md)uupWV-9W1B#2hK;H6{k=sm'
'H}23EOHEhM*VM{3I(Rp5#S#1^^`1mgD)WNale;^Jig9WavmyiQ^lDL|TGNbPIVy3Wm?*1|?{NyS{Py`J2$)Z^{L)}Sa=o>0=%74T1(Wg!`K{5U'
'IM!$g3hiDfYGRf`9;eJ@-dpy(&4}dDOKTX+)F*CQVF)~gCiD0#i2fs}e)%y7k!~7t?hWfGHup5PBO4uHX49-~|0#ePTC?r&G1w1Voo)CK!1UGl'
'w*M4jShB|L;%abeQvYlx&!GgN)oe#k@g%%&&F_%6{C<mn@tLT+9X$n5;akS-kP5=(5a%c|Y$t<_8Qd`F*akC?8JHV>Y|R_!)p#*xzl3w*AeC=}'
'&+&pAZTnBdNZ1eMtZ^jvdlz<ua^I}{+vzh_I~l}AQA86vo>E2Mrw+|D&)s)H(z4CWh|zrk<BE#4(tq0{m@PSP3EUzNzHrb7Y33PpezJX7c;5p@'
'jY2$w(>$CT4&w;TN(OYj`%IBp=kai%69IB?9uLs3Oz{j+ws)f*>5@B5&-c#cPfeVPBr@(Z$P{|h6rFXc9v3|Ugj}lUD!-={a-v}^bg9S9=PJM%'
'*{^R@vbsWaM#$u@yS1}VkZ71IEg7Yvov1`!*sp5r1nbjLh|+wMhu-eMTm*ba-XJfbdbB!XMJD)$y&9-#GyAkJjj1BGiEM;%$ca+*Un^oqej`kB'
'KC-r|M2b2YcQF?5&n&{TeVcA_h;zkjB!osBAEVK`)WJ@@$F6&y{4Y>T0|XQR000O8003fLr)uo`7773Wp(X$T6#xJLVQF7uZ*6dIb7fy~bzy92'
'ba^gtdCgkgkK4Ere-E(#15Ym}U2WHUL5kKF1#NS=B0yWDY2J*5z?CSQh%EU?YIj}N|9fZnCF;Ywn>*)FKddcsIGp)$W;iq2D2je$iisVsxl}wS'
'Kex2v>Ok@>%Rfs}HiD=vBZAch6HLr&)+#|O!V0d)UNjp)cZ(>Bu3nU)*^w+OTh$7dWrXkcjZlQvb)%@_O)alpT)nWPa@#6ib&u^W@XDE#md~E5'
'ZN+aLlMnD{XVhWO>y4ZEOMRG;x3sG0ZN-qkS$!yIjoMhMzf#Hm(G;wjk)JrL3QtIRmN!5nS;mE<eg~?3sAxT=k$}pM`n})3HRB)I&n=S*xn8}<'
'D=H=Vmx1xUlr4+>_Q~q%1%Z!bcTB+aiby3$N}_uzYS1b&(>Yaa(+D$dhhTZ|xC5l<CLCdRykK?CGQmo@A{CtG8YXXmyH+_aSV`MTWhKoO@J^8;'
'2}S-0EZF@=hPq9NtX0oIZV5-}1()09_3tU43%&Ty*H(k(^NQRyP36`$0%R?#hN_`F>GfhcBO8`3@w;O9oS0=C!E?HVfIg{|Mob&<iWEH8NX*by'
'&aXr=yHYSX?;RR^k-;ol?iT_IQ-s%Y&vKQaqhxz9r`R5pdtP}%o4I|p8LdW!W#_;1bh4@LK)8w?m;mE!Ygoi<#C8cZ37Nm?ou)lP@Y1&k3Q>H='
'Dqb^Kb4LXSSCQnFfvfA0<}>;i^)7F``x!^FzOl$xd@teIYr2q4E5J1Mgk@9`2n&YPir4X9(ET1Zk9$t!d?t;U5RHgt-c8cTDTISyv}sf~S-2;R'
'TaQVjPY}&a@B+zzyjB#B3Zx-sg^_6yCDt0)10ExwZG#<xyjYq!jE)>odYI`|@SCu@-7x`@#hOnNDpwMEEA{xoGRvf*5J~2vT@vyJxA{Y`{F3;%'
'?KgBEU{gl_?jRsXt=Wkb4U_t?jI7|ocp>c3<C|8xJAmY^kPR5itRITfbs5>CkZc29r83Jnx^WO&a5ONM7StoJ)83HY{wv{uj!^N!8O=RN*xWyf'
'%$7>H-iTf7?8hwb4Cssz^~?aI_kMvkGNPXc-c+<g-=`Bcaw$CA7TJ!=9aZ@@^dJmd#8GtSq3N`3l)Q!vA#f*e97*CL3A&DsU)pKBZg*UQ&?d6U'
'zVGO<5HU26{@?{o#0WAn5EF7I?BnlRm!pvfu2ZHmUW`n{9p{vdcoiSW@9eL8UZ^dcGTe#FYvK@F9DpD;Kevo;whE{4-gKnD1Ok_WJOYAIzG;;N'
'2hU1SggplX_Ml*2N$f&xB6TBnAjW@KK{TJCY1pU=sV3Z4y|CKZN=|Vo(=@|Cs|Ob*V!bAFa1)K68AjWAGvQDGD-vGDfQ)n=yHHpUp*4pdh;KEa'
')!apzinBg`&XAnweN#dNKeK~I7SS_myAe55u=K<Wj06B!2;ZDI^!ik90yHOqyEu4me~+IQU<y62{IMtVYVoo>$pM-n&}YmLpx$q>oR~I5J3X{P'
'J3O{RH}RX;p%c!7X$KxP!oOLC^OpTm8?+go(^Wwf03-z8$H`YxjP3U7klEoU)xpRNNgsqb4y7Z}@&%{wZ(y*GT5??lz^%Qb53#OOYMbAIdt4{}'
'?(WVqTt?xXamD0pLcU2Scl*+7eJX$l>^w}}m?<5m?)cJN@;XIEppNIoKao&Xbedy_&dFrcKX?VnaZ1%wXFC1hZwbyhUMR;X{@a-a9o_)J#8>ye'
'Cl25-AN<s*|Aj6%VUGKz;`w2*VJePfyWdv~Q*kN|nYrU-l#mqad8|RXV3D@jsX`N}mX|P1ogZ|d*o~p$)_DWiq%<pfg_}`>IAVgTK!J+|tJQ<t'
'whH}Pv#}ZGJy@{QrL(m2_?RMqwgqo2EKu!8ChjaQ7}vU`nl8&*b`g&?yE;@1ZzEjR)`Dq4YH*)3$T&fzOTRS-4T|a&zBc@sX%M~_`_PQtaRbJ?'
'SgW&el50qets8&ES@cn(brN)aT*$=c4y2|KE#r{;ci?K@zrr9IpV`J?G`W#pzkHSW#*J!4-EU-dN78&wdy1K6iJR5ZrUIslSpS}pm&t5+psD!^'
'#!Z_kJ@%tT6m%cVe$gtpG-d<pu<GfNpDBz+40_q+G=qM&JY?XKg6XwB!5J<a-X)`zVT2wo_KtIAZRc2{r7qsVnsERh65GBS#Fhebz*_2HydrBm'
'zJP$bZ&jv?tr;1NngZ)4;CTvuPqYftwXtIBol|y;p=V&~-*}I5TTQeQO-Ig}G<L{Wx-mW*WGoH`+i`tncM-0VTekAO^z|B9-^>R6)0<&0vaPd%'
'neH?;e8Jy8iIzE$#=TDRU-HkU#<+lAUN%%LD;dRi2jiygaLtn)PT#$R@{(8BlTlkZP}{l3>q1AvdV^iinBFxU>QhScLte4uiyNLw9D>ll1Fb2{'
'YCvN=Cf`ZS*<O)%`U|gh2*EJ63eD-U0EQn3Vn4wx#|AcTq8CQlsjMYkKR;h#`L^bHQ1ojxvXTOf;iUtI6NB){di83#yh%a_vFiGKDm8W(YhcV3'
'>-{&uoNU>({L60HYkPBC&&H;$1OK1CPcG2<8Y%31Uvdlh_foQt&ZxW`NAGN=wB&H{;xdM>&A{3x^4|W6UOo~aq@z)!i-V~+g|G$GCQ40`o2jn?'
'l#iy<B}9joCnOY7f@3H6w_2HRlhZ>uXheq5ux^+_0hUc_d+MR8o09=nRJz>m@(G*GTQfK6b2JZz5wxFwfip3DjT$r`OeoCGzQsD8##t;{A1aGc'
'vR@7j0xS{^h<=4-SmyXD1(FLDr1ZT_&*k*)Jrn3?Qfr|DAGwE&W2=+Z8pF63T@ly>WUCGYB!!Zw81&j9JPs7|#f9er0FIkgzNL`4+vC}pkY3(P'
'vCi{&G!xT|f~+ZIgDJ8JrChPy@I8RZG+?6NQGX-@JM|y?Hir=Tsth(CVK{>fP9KDx>>c%xF$h-TJJVTLxq*x$jti7tj`~CRF*|`wyjJmGH|^Gz'
'|I%D2$(RGaSuspr`<*8?(xeylgEBg5;Z7MCcXyl<ky2RG$8cw4JrTS1he7#fHIODQqAsUF?cZf)x@I;1aIw%%6-Q;c+p|qC02<mt7kuuYF&4mN'
'kbtbG86oQK0Sse)hkqSJ(VW9y%oubwgaaiG*KJkl>F8#d&!__8WEx2F;r)kqV<On$2_kI!BCD6nVQ!?&e3VV~JD3)fg-OhGn%iemFa?<rfaWNT'
'Pz&=IN3AO7f5dm92!%u&f#b_rl&t61%jI%)bD1vF_-@wwXJ*^L){MNrQK>(y^k2f*uPO8Y@2dQu3^hD`3EF3?iA_egnn?17ls21NO=Wl^TA$6B'
'0xX`+bjBS6Fps>y1-`b2dRtG7=m#+uR`#!pix$X14g5b)O9KQH000080001DT^r#l8D$Ot0DLz902lxO0AXogV{dJ6Z*yf|a%E;|Ze?v{Zgehi'
'd9@qcj@vf&JwX0}t9^*n)u>Lh!M1K4AWf!dfC(B*y9+E_L!c#ktd1<XqBIu_=HK@mUL=yDY)_iiLq;a4!*jop34-9qx~p=Qlf0pu_N-wy1-li0'
'cu`lRZn7=oZ9`l9wW%BO_5;c4-M;1p4kgc@J%9GwmJxPW@V2O~$d2U&{W_z2@kLX2E#stNJ6cqvtJ=B)fO$fG+ZLSc>bxr%fwux6HZ9AaKdY<q'
'o~-Xl3qUJ&3;UhP{TR8eo9j(k-x9!-HKb@c*%S?LNzUN+uBZ@&3^oIT8o(?F0z_}q)H{-<o38B|mZoYiLaVB71zj9h=}~@PWkucp{)5+5|D&w0'
'u7Jk=cg_1(zU|tg9Da7|eN$(QbG84Tw%f8;>&-vGPc_rt?~!k6;%#*wlXtW%>AGY+_OwV96H*TR+fltdeEy8U^Y+6#Iep3g>_A?z8s?m}_Rx=%'
'vtQZOj#YZ(-5||RC9TxxCF6B@!}1Gy&l(tu#+a}Qn86%O6dvbIQ#T6KMV-;|ebKP2?Z?t?U)D5FK|S`J#$*FzuW5Fj?gu=8i))q&rBWkOys4pt'
'{e(a{rARgR<6JZvUmPh)pdv3?vEg{}zAlUGUVos$;|Fz9&{EBhzx%VJWzqH@78TzE?ZZxSKO|{EF{5gMPA{~MC0DGy5MyCrfNt0Zc#hHwBK++6'
'vz%>6nk(+Yx@uEgfu84KBjr9HDMv)k-jH=&mvhZGFpNr!C?Vogb3}nvH(~JhL;CKQkDq?|^*sIgr}X15@6RuSm;^zTfM+Q7VPt&O5Vdbomb<iT'
'N^8y!^y1=U`tQp_FpcjnF5Z58oBn!!`JeO4sjp1dC}E^vHrchJ4~=myY*48VyF+oXJk2}l>gEI-4Q+ePnqVa=8FDi(vUUk5#Av@O#aD1YmtEyJ'
'C-@k>EQ*~7lsH*4@T*L3g1&$f?SK>E!w_1*IZV$<32<<$73j)xg=FGv+uY9wjtO&tdBb=@q&d>mW^CV*bMXhgH03yE08lFNFpLO->g!-mexfC3'
'u`?sFgW2UKkh^XuSXVwL4}fb3(4*!2)oiwU3|7Xsj~ai?cR*S&J`_l!(8EB_1re&kESV`Xip}uZ4`fI0!ddb%9?Oz^1s{F)O=Qp~(M{-n50VHs'
'{T3<Tg2m1sdNoL1Z63dV7^?@4OAAz(gbw#5>bFJSZUOJVEqb$%9xoL$mW~^rZCP=(ZN1;gDW&F<6xq@N`U-Fvs84+UFzoX}An_u4yfX*o4lw$C'
';DXV7m_PWa0We-<9}PGr%OLz(L{py9E-&g}nyz36zrjNELJe)R+3ndCD31)o$?dktwqYQLgF#>9fB`;dS&iAx!id;O(^U=!m-0h!HD@>Ia2SR+'
'd^s!NU(|46UO2G9;i@Z3F>Pr(KD^C=ISXUAPai&=j~BrXM;5gpQv?N8<;)1bvT*Ix@Th1xlJG=YI=;_sAmFyZf`Ku5r9>)`Kf|DFH)r1hje-??'
'5i|@!TgHNDdG>NPo6T1?xRJ#PFsb|+3{LE3nAF$<N$-P7BVie&4en<=)q9UQCZn*o9*D`S$XqCGb?z}=bCoj9xzAj!b-gyvWn}ehy7dimB(p)^'
'6gjI?Iic4+8quw)n;pbAOjaPV2#<#=oX4f}iM&tjJ$C|{nuGj1*Ad8Gu+o9h0Xjs&cO~8JbzlR7gVdju0u$2KBH5=|3!$3ux@$7_we%naLOv_&'
'JsaX6Zz<$d3jz^Ur#OiiOF5`imGfAIf>|tySq>_&B4-#sUPn3*R=I*SS{e6_7Zz#Iy_+^y_A1L|TCF)`WZF7b<0`D7ik9sl2NdKdYl>OD{?;;5'
'T$f0Oq=@Se3q`(yq37h_n*|c&m@(iDDMiOLk_Mt_&r0Bzc#j1ya3{&9c)Uv#jo}Pzgg424dy9W~puy58GO+@BO@MD~G##=uzx)9YAA|j#SpmZ!'
'Yhi5`AN&CJTNs#5EN&Qu(s3b3mhV$gGsynrI`QgR7TH5?5cMdJFsy}LQE78gb`eW^9?O$s6#^O@j@5GrxJ8S3z(GxyML*`VQaQ}@QkoX<8oXR@'
'xCSQOE_9Xu%%k$AG4(jcMiC%z^(?aOY`_#O3{}<9wlxUw+Gjz7<~!JWEC;Nh-zS<>dmoi=d^kx7OzXb@BfKnxkNthv?Q`0)#X~S438BW=`f;pf'
'=FfpgXNzMPYgUaF2z62*jxJ0O%Ec;YG%5kq0)tV<^<h|i*vmQ!asv@GS8W5O(q`x6p!XJErAncF?pyio+2FIavvvqR!EwvYOt)=+gF(+85wT5&'
'LzlxU7{&%(kw0OS8xPv<K#F@(U5`F|cxGYmRcTWWpQG3a8Fbp%2UB&S=RF=CEjM)4mL?EuqGT$|RYUhsWAH<&q1XIhltPcFO)RC5xzAH2AB};Y'
'CUAo|^TRU-w*w;AoYF!+Dq;(AFJxo)M@^0;iC&<3K&_E|zQtsV?_lHvvgInkSo9Ev-4;-^Y&L-tmlKcj3_y6d-<P7pN1OW;l4VGWr$LPrb`0>Q'
'E;3f%U~_Q-NWWqNqs`sqr!beY8|B%Dz%Oby*;MUI72VR(*nWXxMfLV3$`w)WrB%0Evj!@|s_)jhJ1L^jxY!*Uky&*PXd^gXMJljX_0}~F#zaII'
'mI?Mq9nQPmKE&5WfX4}Tp(xH>be9soQ*UOihcD}<046TXKGP=X`z>Pj@%QTY%ILKO|E)VhMQ0I7z`kNNoYRQPlAxxm;}j~7fb{VRF(G{d%V5n7'
'f-2b!ty;+}U@2L5_@lx8GqxlEbWFB&eI4uY4u4CS6kfxGDO8t;rhzz(AZ<e{DPpMM)BzH_QSc?ATE<R<5Ctc<TLyG>oNc<&#`IF^9u@ullC_9('
'zXy@scg&e7*@No<6USz7wzYx0uVtU1x)Jz`3`6u=z~Ail9Nj4g_KC7p7VFmMKO=Gl7t=~BWI~Rl>{zy?RguBYn%{WjHn&6vqD4(fP;+>w6f=hM'
'j^hy)<yldv{tj>#2%#<ZV8R#ii<rEazlb8;bL+#~-wB@5VHTt<j%+ChHoTtAW@ms>1NAp(f$)zqNR%MWHX(x_CV9=c28Rp+)a7!-1FHC@t1|e3'
'Em-gzMT;kSy1Cs64Mdjc!-~Z67Fv~kZeOT3Hd65Ka*imk9wXPce6b$LVYl-69TFkghVIzmC6j)u?LecKF3!V=z^Xscu{|xgjj{LbLz=V+?RKo)'
'BTGWoIbP@jj1s|pGeE|QS|urQv`Un`#2MZVg&+&D<3ySLQLiWXE=|-r(m5oiL&7idz*7szIt4vl$aqi<@uF@mjBaUz^^<vqqb2KuH8;zmiT#YA'
'`}s-CkiIH;U<M1m_=<%uWB2?a1NFdC_NCdR?@u;i`_Z|olfpNaNcCf!`07Ur!Woo$<$Np1a|<8i#8^LyyoLJfFLe-;RQhC~W^{HN8mG8DLlw*O'
'RNyRmv?y5>N*W?NNk-3U?2^>iPEy=ZaAwv?x~B}<K&X?gR1w1k_vM+J>BCK_KB$++ISG959!#W$%q7K06h@EEHwH}#uB2XckHm{bv_|3$c{wTp'
'oUfN-<iwmA8vLy^9WeZ}_yiDocltbF0!qJjLv!Zgl=%Q<61N_%tfu&F(Qd<lgCJNC9frXmj=NB%znB2_c(iMX52}OM2~FIU5<V-lewSWaoQ~03'
'TJ)D=lD1CUZNsQro=;xQ(2w-fCz$HVUc%O%6TC(5oBe>?)yii(!-Es>`Vhs3-iw^ZJ^NCKvm?flPzz)klo*RsJai0J(G+q?U#_3T+|f;&5x|9y'
'ZiC<rqj<-PFEpa^5X+ejCYnqGb^6YRWMHroQO#iKR9!WOl}MN1aN53o<Z%N};5I&Z01U~=GEkgNi`>7^uVj&z9V+nao`}ZAJt0-g+!RY*6f#Xw'
'k@?jLcUo|MI@yveo%grYmPwNQm2B*bB+dSm>#rqxJ8;Jv!J`8kAI#5=XMQrLm&+#boW$?Ij>o1H3Vp?ZNk13)PG1=$!(FqqfYMN2l{C+u)K}QZ'
'9N!(1DUx!_Ju7u95{8D$ODZQibc-$eoDFO%3qD99L>EI-M)-9B_LW1On$Zs8a8(mw?R}L>@caz>nNZLVcqK+&uo^6~XP8IgMy_tQitw72Xq1@~'
'-b_kQYEnhFSH9W{6bHz{bPmFf@Oqw6-`R=2l%|u~%h2WZqLAoZf$Mdg^*E|rfzRg)>>|&S@BS6zWvrPz68?sKW!Iadu+lSBEL%+U-kh~<-O^I7'
'<#wnU<>3y?jwj;5J8^GVZKwK7%4i8DeE>;Zb5N<1=<x*B*5eCm?opRO@F~Y#TDbcB!BjBc(U)R}e$^LN7cM`Rvy}|tj*!O%jqaJ)I1PYLOM&pQ'
'Z)|#ks0hwuM30B*5#v2;{JD`OKl`qw{76({sw~8RE67!}4|p2&Ml0)N^zbo7?wPjR%I4$C`0(*1&-ln-?89x^I3rX0k6k{+M;ywJ!Vq5a&D^UX'
'V(U{XW`VpB1$qy!%I2byB#;M_5`VQUVHtN@UyHk~o+6Bh+u^XtR3U-$I9f~}dVU7p-}|flYsU7v6T8MJq8sJPS6Mf9zxn5wDyOU*-Hq_k#qL_9'
'%e4#xloJyD7u$v{d{$st;qemF)o;tjkB2>d@&+o2UGQUZ?^_Y~Udg!k@(ad2i3jil@Zh59WI2EPfH0b(*G@d*QLYiIeV**oM*k_}xEv37GGFl?'
'ouqm;`cIU#(d)*reoWTi30c4X^0Iznjx){&C8-nb_rp?wdhm0bIR^yl=-pupRb$@vji*uLR?6X+GY&?R*o@3!lF9KwZ8Wc@{SQz}0|XQR000O8'
'003fLbyZ0jSq%UHWibE%6aWAKVQF7uZ*6dIb7fz1V`yb<VJ>iaty*nw+c*;bUZDSh=l#+EZq`YYBHLaL+bh~dfxX>gTPzM(6bda-HZPK>l9Ur4'
'?#JKEkd$b>*iPC7y0I;CG#t)6^URR5+3fSS78y%bWi?N&VY$-Gt_3Sp$_w`91KY?<sOPCFYh|RBs$zFa-{yt7JAL}(>66bewp5uYm^6$R@H2?B'
'iq%@d1c9>P|7!Uk0n+De*;o><MU@FHG*eZvWjswqZOzjs8(zqaTM!{DV|kSdbIMrvj!JA`jwmH)$}9H%eOZeYXkV7R%2**Q0ZPqgv!_pTtxA?8'
'd1D(bl7vanUt5?|Da&yg2DPVAc3aD86-K<NwsZEYv_kWu9a3*IURjxjLI31N{0lXjv-eUI+1%-uc>R^enNOdj1viGhF-F)=s?n)<tEybCp0(3Y'
'UOj!n;6Ji<Qply|di#nkl`7aZ`>Y$`p4Z}D6lUy!yyr_kHl?Xd4X<CfkKXgb_-DBm!l<Uo#vb0uTUm-s^4*7w{lMyGS%48rVL7alvo{|<yk-?`'
'HoA#sw_@vrl>9zPVNYl`O;QV;d~F1-O;#2~qSc;dBIiwEudn88C9cortPt1di<8)Re=3;@1-t%xO}jR<9a{cX*s;H6D$i$gHd`xoo1k;GUqG8g'
'HC8M9C2xvi_5~E7@lKEx_x2StR<r-H-&7?AYT}P`T3R8hm0e$-Uo22toV-CjA2$0%G`VO>m<>_Iht@eeU66ei_!lj7x@)3O`TYZp1G@(kHUd9G'
'+*dBJO=~Z{T4`YnjvyPg<Bw5yae8?<qVI^C?-&dJ#H)IjNr)rKi;oI?ZaZ8Wg!wO`!gb020$_+R-hr7a40xt4{Zoo!&deI4w=v+VOD#oC5Qf4&'
'6HK);xfTpwC6^%H3Ph+3#DfI@jo#f$V-YZ{s-G7CNCmsOsli^t-rUTcxSN|z@Y|akhS+p-WAc0WoTIG_;9tpQ#Y|Hpo`~!<p<+!-g@;BJFxANd'
'bXx+j8Hh%^Mtm;eqkzAlqSOGZh{*t!+M3hWGGSe~B2n1{FfQ@~_l$<}#!^67UfjZ>lC|I&ey>%Qa%1r$M6A#oDeh=o<E%WJM?y>rfKnxyC`lH;'
'LDr-+H&ZFNN(I`HV{Hnx!oMY`#aCiSNZA;flJ`}zB!~`=oe!d-?mKQZ>WYe??`|Ux59=u<N-6MTS_oc|GTVxm?j0Mc{X;M<zAc^g-;y>qaT_us'
'p7vXBL?XO5Bt}A5ry_xA21m61I2ushYYZiZgg2Q~dtQW}wE#p<>0{a0p{Um9M&vvc7Gn*;Nekh3G1*AdaLh$huK<m6p`E^88}QJ9z*6DvCM(VB'
'wM_SjGNM5LcB@|t&~dOyyo8yFlQ<&CSlU}utEtj8xQvNCVW1iM;Iuv-x$xtPpa&v%Xs-NzTEZyWMrgdxQS7!53B*<op60-_553`S;QS%Uob9BX'
'v-1GOT3|#cgkO=mrAXHhf2ng@x7W^&T0%sVPX5;+SWjjl0Himd8O0g%=zypd@b(lywT4KOmVup^xsx^pKn^Ggs{-7XxRrpw5xR+rFAI?kiHfQ%'
'5RKO?L?=rN3I2-`ElLEim5>6&ns1~EILk@>gzOxdq+V-S4|$+mtqb@^$ac#$SGOYUEm@`;;2FhQl(DECOjxQSi;;>XbR#^!(~2A-S#mu_|FlX)'
')`0MY+-@-7c!%yp4Wk0w$!&U4@U75hoW*&TLOK8J{G^-P1^U>uvuN7YN$_=ax5NVoLeD7qEr8A=_TI@VQ+J6G;85ArYtEg1FWj46w&V-LqbF%X'
'#Q8<UQ(v~^%~{3=lvjC_SrdIWYlPdniB=R1skqj>hh>4Z`K^<c2W$G!VYmAVv?f~#|7(xnw!)KT1A}a5unFRM?wRNLX)sz#<vJi6@h-@LgtF2Q'
'ks$!NleNoS9AbOmV!>=lE;v?fpc9zN{?&?Za~lcE?_Cn~jA17W^kA02+B+c4-N|ND=r|D$yk=j%^zD%3P#G(|y+)08;Ph&Thrjvoma1g&2P{Sy'
'JBouJT?wV#5MTKML0xXy$KOAFW<QcfKXU0y?j5{qszO5T3uZ!!h`eFci9DsKxm}Nv>}X7*i^(BCd<9spOfcajGMgOe7c(4kpa8lglyEYLj_+o*'
'jvKc5?X3p)5Lo)l3YjovggL`p4E_vuC4&DPAGyYbBmT&(L9(h=X!BALpq$3+Gs>7R;-4dB7mI~|auUE7Ie+B-L@E<fq2--G5CL3(Wa<$*>{B5o'
'^-_i8+R5<S!1WQadU5Gv_0q@c+0~+pRw@q;iCA=X=1It}S%1>ry<OR8GC=C80g``=PjuEl@9`Bc%t2<7s;090(ul+67k<_c?4Q`>fbhyR&+T)E'
'(e%d<|JcGFLA95|29lHHQiFrI=G19$?i@=5ETU+i2_+`-LzKxbP6pFmQX30czq&h_VZ@HEZgJ(I`0}M&;?>1s*ET?DfO$+z|LJ^kdR|8i98=sm'
'l<$}&UoOxiUo1R5()q-yz2&{&AeIW6WGOAO<n9G$=T~!f_7cQ{PWY~u_!9WsR;(U~lRmlY@_sHLg(Mu?YGbs39waS34%%;%_&yl9)3$J8_cbrN'
'v5=q+jzl;2GWU?Gt1(j5vc$GP?R85gm)IM(Bk$ENjXxm6k7@xtDn9!?yGcp?fdQ?M%646?$?4sYK;@7S?9FLz#GwY(;%cXXb><sxw7*3;@`6K&'
't|gl|om$0@Z55(`eH_!X(MP|-N%^_g&VGj9p}a((oOw+^5DWeK#8ZH0P_gGH?D;>~tj~sKue!7<p~h><XQ=%%5YdPeI(MLt5Cx3HoJKTe&f;Id'
'{{DwPML9WPe*-j$UPTzpC6tAKqV#uK!)j(9UBepcLvW>H%g{h7XR-Xg=s5!oBb9;~!g^4hC|}}nNTsZ^JvIOb1O9&P7qyd{4{07t0B5#P#IhB!'
'BFtdC+D-LiN;=xYuCIp-rBf6_@D%r)hDA+@gZlME0-WQC3$|QYCx(4`|L<5esLC5%(VJsJjr{qOSYh%Ua%sO6wA_9q?~oWjP~d4tZ&*xMz=j5k'
'nbE$AYeWI>UTzcDbXCabMnl}BN{azq>LBP=q9!eAbm}1+Y|Asp1pR|Jo36m74?QJ#TsQ-F6HzcnJJFBhC@#Uooma=(bI56;`Qttv2|L1XD34DT'
'Vg4K{-segF+2L+)*QmIls=LHKwx{m+TSU0`y5%8%q*gB)JbEj&XR*9hwd){Rt4iwQh@46e9lBFBFikx8goXK!RQuM2^i8~x0q(u`4@`?VXS?St'
'=wf1yC%aSgt}a43ao(F4c<#~lNMAsEz7Scqg7MS<=b5t<1Og<2y8E1|yQ=#Z4DP<QW+$Co>w6<YC#{v$d;Q^h?7TJjrC-Bk;QTdD>EPoa7+m&N'
'e*ou7I_FOO!PFGSEurGh<pWPkyXh&i>kN-ej;KX@`iqU536vubT_Vup0bQG}HwjpiKz4$%l6dQ5Wql`*jbUpHWq089C|srTf=$Efk$4I```u<V'
'3HMO(ZmpDAyWWRPZ{0a!?O+ld)<cx>O}x+%?3#76FFr)}L+7K<pZEH{XU+mwIvZuigF6Wy3tkP4(%mB%$uDP97YIf(#&m1u+pY}!7O2K2cYZwD'
's{ByFPGN7dB_^*K9uUyc4ba;XtO<F;AFi8ig?D}Ew4K<>+~3w&fj=9O#eK|hl_+wZwyW(L*mOqWde{#(YS-GQ*aMhdxjc&X_5OSjD(%akzAxHz'
'mjA`IqQ^Z4&&k|PqUVlf;$13O_>aDU&C;g5Iz=sG>{*4YSxUzy?nNdZ^>#Kz#-Q%NG<GdD9YTF=6d+Q{-M17ka=9LUYF&GLEz(<DTR2puz?G%I'
'J6E)AJPSN1%>G_X=mA^brI<1L<Y^P6HduPd;Hkjt4V_8C@19(_6}d6rgFbjXjQIggiq3c=x!$|z_$N&rFwO3`#<Qw++rl=HurT8Yzzc;;l8Kc1'
'p<}4`U4!7eq{%3PzE746!S&5~@@;oXay-5bgi*;8Kt;3bKW4o+i_Uz0KJ(0R@UxEyX(z7$y!qe}0T3b_MU3t$-Trvgowu&<tkpM3Sb~89JO{^$'
'O3g3FV10mk>ZJNMRFD-js>cvSY~vCi2m%8WdVsL=i#B6>))0j4(1-^}45u62ajtP$JP7V*_eaR?IN9Jn#4(4vBW;iF5&-EKMe$>ddBiC}3va;i'
'#p7VBcNg^!cP}sd75nwW=yopX4?*M8fc79ZZb-$Wzz=|HO*XB$wOTG4mvbh@bX|?3`W3_6+|R|k-GJRXe^|lqEzP5qbh*=km&=?73u7#WCkkJn'
'x+8rzoU$0pfx{Aa^a%gk@cX$I1{`Yw#=EC2RRLMO<Pmgt9@icbkrVIu;eK}x%@#))j*fyIPz8I*91zq`&SXjw=6F^9z-@`)c~!W+wku^)2o8&c'
'B+gb3pZpa{J8&C+OXo2XAW|g$Y#uuK|4>T<1QY-O00;m80AgJt;+d5m2><}v8UO$m0001CX<uV+ZE$aMWnXh;V{~b6ZgVbhd6ijNZ`{Zceixws'
'(9u4S6I1k9z$U@i46<>IV1cY{#KPu584x{Wai&)!o9QNHO%(t4RQ17YsKXQ>k$u#CeBHXZxcIr&pjuV|*PVi#)m$mw3RyE<%2u<lqCLnC7sxwN'
'@0nCg-huk6L`$Dqb}9pw3tq~72l4sY`Pub>1{7qiRDou#s6e;8YBW<&fMTs=EiTbe;u7Szlld`b*9V~)&w11061Z=8ooiZGouA1%gNTW$0_4*|'
'd1N~%qynT@=b%uInEewn5KT)wS}tmoQ|v{}OD|z2rOI)j1&u;}l&DV*<>ek~Ky^`yhPMaOE|XP-yctRZ_mFishZ`;~F3!&iB`cPuMc1OG(iA0X'
'B$|%bwQP;aG^uY^<-Fxt$+ZUUm-R#EXKv)c^`R8I{<}gE+<uY<r~KZ}X2yMd<#*6C;vz+f8>u1H8Cr|`Eq~=2zQDeMx{cW16>knA`&{xmV!uj}'
'0XdOAiO~7kUq-^2KkT!DhGRT@R&uX+wK+dy_%yg<!{qKa^qd_j`4N=fuu^C@!n<5ZJHY<OKFS*Gc*UCrUFDb3PYwlZETdE-Nit!m;6EJ(Ok|iv'
'jzPqfl<meCB0>wO8cFGu6AQib<yd2|Xz^cQg;2EJMKsGh8_!4sSJW6FD2WL=mLVg?mF&phCJ%IMJWWJJ!JH^D#vR6TJfudFus{|?Kmmi!H=+*H'
'y)I5XdN3?IRnN^DPIV+>MeJs~vis%^h8jZ}yn|EStDN3>mm-n(cJ&9Pq#`2|C7a3#d-&!kquio}3blTL>^m|LS4SieuE{R6HAW_)H-@NNB?AVf'
'23A6C^on<!qSD5oUJdld&muLfH$An`O!ETEf~Un^u{Tdsn=Z#xCMS_~EzOGGhi{_C1bzMbYPA}PyC~vLiF^+Mqf$Y{*X-S3r+1H$-b}?-T%a^#'
'hlRmU%#vm`09HUgP~AcM6)=F?uhLgFPdJ#0+iwxBiobyGHBuebeisESGu4gBEbE%1t=qhUp2L{2;dJ9?wiqpBHXnj6&+}0f?mE=7#esFa9^%py'
'i24ZC819gVhJH+gI$VuS=@LTf(NK<N5$(|l5w~zpVH5m&9pE2dW8yw<?3?LsI!2+OzC&01oWB3`%jZvDKBU*zU(#Pb{q4g)f`|pdI65+pWf$2{'
'ePXX!(AzM;FIVAXec)F={vq^I#ZYJ1eo%|}0Qb4rqb%X-W^=XP&IrfVEw<8-EQfARu2qKx0`9Sbrt;Q&&P5>uTbWRpUcYp`j<Ih9lQ1F2+C8z*'
'X(39KXcj5D0mu$W(gd^VnOKgRW7sv=G9Y;vSm<EGZYI4YB&#RA{UdfK%dUcC$@(<0+J2i=38?w5G%EQ$BN1|dW^RJfrV)ACgfxPpE6ZTIcmM#&'
'BNa7t^de`B$l7ewg{BmpVSB=x(^9pRe;<e3b5dwlaP&$3)+jG1cjjri!*skAO#}Hj5ec#xln~1X>yU0$Xx+TR;8nX|#ki*VxvwFU1iBKMqa>&c'
'Y*B#8hN0w!%m}F!PAiVz`;NtLUQle7l(iiam4vFn#5FKttkn<fUk6!YdpAO1Kz39e^?_1<rvQP00i+Gj;4ON9Asj{cmHQC(Fd!Z%`XSVqpjV6t'
'vJ6A7$}1LM{kUSk??bO>G+AQwIckx%l)XxNqQji8e3Q~>+L{X`3@yKF)ngAw@z0Nfkkqb7NAj-MW8*Ivz8G1)<+o_Y2TNrWUlrf<TWe12F*;KQ'
'O(Rw>L~H7Pu-tOpQv?q$6x6aB^%^@(uC^R3_%|6juR8OT_e-a3HX6o<G)8JSTLM=TG#c4BBS)xwSTn54I)&{ACk$YmwJL!;?x{m5Uhg4%r0Ek}'
'nY627*yv+pmNPau8UuDmQQAk6*}t8Yy+vEZs3)G1!qIUcwBRAdLz!xmb>zSV!pl7I%%W3g-y+jw)ZU!w(mfuxhzh2lmUV)GFhNV-(qYSLUa3#r'
'<mG+3kyz2KZ=P=}Gp<H#&nL7-_M{%85siB3i5fq!_jqh|by+D3lL)kz2qau8(E%4slo1^Dbt}ipx33XIX80LTsUlD)P1j;U{F@YY*y|tbrW6J+'
'{Ar87{&+Qu1QW|%!O_t#v@SeIc^f<r;~fe=4TNVQdyw(6#DQS$D2g!q{_2T9GtM^Jn=@br6tqc@;cqlPhFvWxvo{HJiI6Jo8pG*WLK5(d)&&b>'
'Ul5-t;T>T#Oa=vowh=cOn!DbW5pV2=fJ_{0#%U6GPQhlv7$@u@&?aKAaaMY=%(8wH5TCg*1G1gtV=^VMIZ8nuYzPR@+iDK8T#;MsUVXKC>7mA-'
'x?J_1tX(WzqXOsF#Y076m-9hmt?wy)e*a=0d7(coZ*=BZ2(;*eX#E=Fp;L_(92`({@igf@vLl89y^TD)95hdc#rL?I9@FY3vv%HG*9gcE5?W0u'
'p~vp0dl)KBn4%aon5}obWW05ED<k|$J;&T5mi&lN0Pqc?i{H{N$$tGb+``a9t{`lejBb<Yp%$E#qM=6WZ<kK6I4!ubZrrlpVWo1eT!7^WPg+rT'
'Fl!eP+Zlv-@3pohM?Tvku&%H|lXu6n<c#g<@nCn!5!c1*t0fsarEK<umeADS+gJSZl)Yy<U%OjayKoPKB6$;!uS)1@gYR_cM7mjTFWs`6H`^6K'
'{VCoYKZDCczh}LTJf$~FQc&x7gxa!zk1VF=D4@*!p64WVQcX(U^Xz6UMj3*=x(#_dPK@u5)MSWM<A_s7vUdsP*1MB{TRL#s+sDNyL3oN}E*CgF'
'bCmdrS_|EwvIpKGqDNa{)7RK74FED#gf9O}QNt;^kM|as7|@MG7B3fEk>gQdOX0VbH6`-&H<zBw6K#{W@MveY*z!gEY%<duRWUMy!A&zM2`r`N'
'_9z(!4cPIcLrv=Eb#b3!O{?1{mE)+H4HjB~0kfdph!RDJS!Zq<^W%2Ll>XV!rVhZ`qYOb)biEQ_3UP@L2z_!e%?+zD9RlZ$8cs)$r}kc&NSU|q'
'ago_TYB(CFKCmRM7wo(7gB9SB>vFGbG?lPF#kt(VVEaE%O9KQH000080001DT@bsJ!yW?w0B{BX022TJ0AXogV{dJ6Z*yf|b8mHSWG--dg;h;&'
'<2Dez2grXIs3<_HvMei3Hc0EkV$nl^-RuYURul#;jbg$QDUej0wCI2D3?+Nhjf%>KWRk=8=DnfDNs_#`m1@C1o3KZ(|HK-CQ^xC{t$}Jl56aYF'
'dmdAu>UJ7U0AE>$Kf}%rrlvh#-hRlFB)Pn3ob6z{ZH6#7+-^a4y>)>wj17u0{N=?o*Qu~S4~Vn2-FfJP8>(;~9@QgG*gfrP_3i}xDT45)#^!|#'
'!EtwSIfH{X^&MkoP!BGqaR*InRgia=7eE)~{?4Ha2-fKb%}PJL`v7XFwFPa!8eDj+ELl<Q2XpW_+*_#a(GUt7;8oYR=tmP6k@@ppIWjBn4z+e-'
'9Qe&wH(&<%q<z@hLmbL+52Z#c&OE>d%5_l`+2l+9B|gOJ%@ldmVIN(|ZI4Efv?y3+kQ{f31m(dFp&!DekFRke_ZTP>)p;RS8I;+?N?ghP$NOIs'
'dQz>RvR&`cd#n>FzkviQV@v626U50&NF&CJsKz?5O+!>{9%*<$K?HI0NGM+<##eNeaCgrEm2>J@$bSbgr1^$6qG!-)BPQ61@TU>~dxDre5g1s;'
'fY+;N9<e1mDTq<YoUqJ`Qj)xBd{(>&l9q1eYY^SU(*~OjB;DY3mCWS9*AMV{(5%3>_Sim?0i;V)>3db@@O7^}&{sk4Y=z9*!Vc>UxHb=HkP~@C'
'h#c_tw|t((SsOeF6055sza<H)Rgu&E%{sr~(fTI8y_%76+Dls_#P3}1uv|;{5n4KMaZ>&<zwvBjx`F(P?c;aBlZWv~uW3pILRw2gOw)26=25i+'
'7sC&oI3twlgpqOzsY`>5N5k>a;2N#+@|y40QZBD%irk(mMDc$mRjaE5ANk(){GhI)s*wEsA4NN*G?R|FDT*{KuK7+$$#DUnda(#;S<HBht;hf2'
'p;nL&Mw7r!ELfbJaWf6e)460bDI#CLhK(;6<rC1<Vr;L}o~naLe%X_9u5YKFm}NZdF@^M0O3B1!Ni9jr@1#s^<>++4hWZM65xGu14e6i2#CCXj'
'@efc-0|XQR000O8003fLJreJsJ_G;&<p=-(6aWAKVQF7uZ*6dIb7fz4X>)0BZZ2?njaA=I<2Vq0N8*35`r;IY9H3=y)l(jTg}q8FaJ#@uRatH_'
'X)O}lXFCNJ@xR~LPTIiz*ozQNCgYiJzWHV{8jVg>D@!Peko(%G0<0x}Jc2yRj`!~`(OQ84SwKo^#l4>e%?)=@Ka<L+hPV}wQWz+lGTCS}+T96e'
'#;Vq}1{fov(aJc&rBsePp`_j2+1=@^%VKk@f_-<Vgo>143nLc5D#IJFiICQrw(zG@5^dwj?he5Vb55nz2!OPjI7L7#SB@Cqb?5mTKGeW1kS{lP'
'<T1RrWC4{jKpZ?dqH9Da<TCU5eb74#mfRXHEjl$|EG4>K)S?)W^UF!=)J^0}8FF$nwHdimL~~IRkfl~ax-`Z!&?pC_P$dv}f})L;tVIb5i}iLx'
'R#0m&DZyY>Ti5#rtg3@2qLe;|b-Z}UHg(6Fjy~kSSVc?dz|Hise>H9R5(cl_ps?QOO))7s70#Z>^7Pq(JyWt0%QTkGj@kX?g)(R?g~!J0FCf8$'
'&i3>*SWnH6h8(+^fm{ot<cyw7+1d5io9plA?0))vdVR%yzhGalKc8RHd4>%ZIvIC0RRq-%Ys!kY<e9LHueqrCqK2gJeV~>L?CiHzI?=$nG0G(L'
'QWe<!wXmMM0ES3_(h^S2?mnM*9niO#qEdbm_Nwny+4++VB!YDMge~SZuom#xC(s*(<ZnsNd@q<bfb9?x%A{v4C5@0wizldUet7h0FZ`jEKlRq5'
'<B!qWSK&KsQxup)sW?nEo2O*<Lfx|lI-y?WG>t!#!7#nfJ6U+m`I2zryV_=xPxApEkHW2@kTRt~Cc$Y&HFqmI9}{jPhQWy~u8eCuQYPUkLyT)M'
'E}`GPoYL`ld!PfKAxQ82FX*e$Ep~*wbO3yN=zNkLRj+pRnv%_-kc__0#{)?sFGA`Tryqscy6)~u$!<EFc^(wdANjfN|D%;9ilG43aGg_*?WgHm'
'hJwUd=|SgF?iFrYcla?S+&Rf`JZor>mXIi?w$%K6j6YkBGW`)eSP`Z*|N8(ZeuFbq)`x{~`K@VjsW0RrJhBji1r$_PAKqv9U)LsBp&FzX{;8^A'
'zfok?%fqYng&I}+wzU=v_PWyoopd+7Py=OtW#`+$^75*mS+U|+0U%%ac}n`=DKh*3Qv87Yg4Y&n=4Hv2o7Q315N+gRfNdj_O_s!`&1e2Uj#d<Q'
'-p`%Jp!oghSVbMQMzXlHZBDhL<IJh3iYzK({BE=&OPlX4^S*EPoWHi3jiI_SAlciCfa8*o@W?{dHvjeD05zzX<z7dTf<fwtbU@{TJ1@tu<&rMl'
'Y?8+4Ga7zT*VZrW71uD^Kb%jH5BnfA^Iu`X>ppj&bav3M7=a^G59iBISC=9s8~nbobc?8`dV6eH;0md0{NN{pN(P?_lz>Ay@L~nt<1bK40|XQR'
'000O8003fL+k%OQsRIB20|)>B6951JZDnL>VPAA|VQzC{a%o~^E^v8;RcmkJI28Sk#DBQ!FO#f7W-7JP2q`O}lAR*818QMLD^yvA81QalN45iH'
'XZzpx+6hfyA4Le_>zm_q?#qObspJE*Qx-PNSgr?$oiHWWa8UXWu49;31Cm*>=TPuM=_6=vD=iIJ1$M__`UFY>e-Sn&g!FqGtqP!YQ=u296hu)f'
'ZGlOtEaE6>`n`UyvHot9d|N2<_Gpf#K`NQu<zjV$nc>pC-|uC716DI>)>^E1z^Y8Bq2G@nXL4JyEgyoh8vcT>O7aMPj)3L4I#6De_Be!<QaL2>'
'E6WWJVekv&!q{ad*7kYW?*V>1gE}zu6qts{_j>uDGM+m+&b@F8MC|Sb^_Kf66K}cYavzX~bUYc++xy9ke!rVM&C*BuFde7&Bm!cEJd!X*=AsNj'
'P`WWiEmFq!V$GBP2OOv0Cby|WT$j}aptE%F)~YBg%c(t<@1fpKA7+brO6R}LJhqVb^98s!b=j(e%oWS1R?5ByKf1k3>3H(!us_<O{CHlM7f1b@'
'M^{%_QM=QeWhN=uOK=qdk_+CUFk8_1-D5f$KRFUtBn-n2rmPBDH-|~C8hcp(8NMyJv?c<IRp6^3?v!YYffDM14cl5xP7h94U9c<83Tnik91n60'
'nNceWZY(QGlgxF+JGAx~-;KyI5bs3z+i2{o3}44!Vmok+!4wjH)HW{}e~#R^KPAcQS*6HKxGtvnEj8AmstmRK#U{RwBCyz9Tr=1G$*1_!d4!w@'
'gRxn*h(Cxk65p!9A$AL&+BVb30yS%kJ{znfi(Y>1Hn_|>W?Xes1*Ft!#a1F0_SkXjSW)=~&MY8s;Nd)UDFBX={L^cTx<PhLrd`-hN%!_c#}e0K'
'(YkzY-^SzqQz#C}3>tA~G-^Al>so8f1{z_Vy*|>pyH$7EwG1>9DC)72u9#A-l@7?o@UKTacPJhb+PWi6CTo}az;+-POvu2`Qpa%k#x3VEz?mL1'
'z1@eNn=82M=;qahAUNMn61ediPNwDcGyIgmhrz%r8hAw?!hdLbI--gq#j7YFmGSexJlY*zPlEz)uS_u7wj+!d<H;1Ji#g6SxF3D}vKW0yVR?4Q'
'^GUAn)(BK(aZt8Ri>3TV@@jb9B*R@oLG2pG2QA!W#SRJFZ5L-1MltA&1@C=14cZ~lNn&yqrzE+-!312cRVL&%A(h<>K0Ctxg5BUHELq{MU<@ds'
')a8a!(nPJ!RKNE(P)h>@6aWAK2mk;8VqJYSYHi~R007}1000sI003=eWNBeva%FRGb#h~6b1raswOMUX<2n}pj>LbkvR~4<ZD|=UGYiv^n3*yl'
'!O$%*b0vf#H*s4JNt}%xUex8k?>V;f+7wu=ZY9*lv5(K|<L4O1alVKw4#@Fwl&}SfSR~RXB4Q#V8ir&kq>_guVQIt_VUJ13efY8Di)AlmQe>&m'
'$Zs)|xI`q11w9^{Rei^CwzuY~h=}LSGnJ*x^9YX;kt#ytSSYHvh~@Uy_EtVBpJJbj;@KCmVvp*crm`3l<<H+z#KnVLW{QX9V>U}t;WJrIS1d~A'
'JY?knkC@dkp=ufOS+3p$9#Jp-Zce453D4i1TwP7Zk3LIKdz|E*oPIwcqpG(3ujcnpnz~NJ0!}@2nU}jvI$aX5In6>fI;Npy9Tc>^1!JT>(PJ^4'
'g)DG8r1zQ3ga{9|w+MVv1|E%x-n_qH%B_gFry&ohVvS8*E9*eZ(E@*nm|1~*PV9S#P5^#JsVu&Co%8Xx(<ATb?Cj!~cm3n+^!)T{{LPz=FE6gA'
'U&ohVi!(<foK8O!4=e>;A-V(-!F7iCR;3xk?<X+4W4F^bP%c^Bq3+(`J<L3*C0HZ^Mz|yamqtNIHnoHS$<lk4Iu+ApLLXw!xLi?r=lUWQ-G)=G'
'1FR-CZjA`~B6;%c0Q$Bvv!7<MYaB0`Y6|bv;$u^~@23m{Qc=kY8`<xQ4PGjxH+xbLv4lgkc&v1=;M=OK(x-|WAGlgV@v+#QJ3E;5J1bcCyaQ>g'
'GE6lV`y$}+V&qWibMACNEg}`D6jj9FXD(6_@|a=pS|`{{!8v`9pDKyFyUx#OdU<(Ji*EH}y#W@uW+?y(xb<du+ad4BaIm{))-jz2#J#v2r>RK0'
'<R=X?rhivOClu?tB6v{op1(*%mUsz7L;~oT%28PovK!dHi%Q&*|5Q^SkZNk=U@#~sn37D=1=E!fa>PDJxicPTkHm;4_qzmqbw4yPw;q&K2uvwN'
'64%+0PjY82cg$L+Q%;NMBagBOX1aRC4si?y_u&l`6oeq~Ob&v-Q=Yh$#Sn;M{RkGqiiZEKd(hVMMJAzyMmiYHP5mrQS*+?CS9ZjdlCU=5em#=4'
'%xOx+RZUxP8P1nA`Bq=B#2s{vqCKss-dN+D!(-|ze$PAvbA*+F)9G|{TwUvq&i|B6^}xLnyd7gxHHP&#wz*ZOkjG<F)f>Iri0S7M8In(*$o`<E'
's#+uaN_2HjzfFz5JJ4?pwgQ?EdYeLSJynW*W5m}Sn1e^k{+)3kD#jjf35I})5o)wx2MCE>@(%XpG4AU{GmXyRr{RbF=79>pa1=ybN;uYtzhwR$'
'nKAGlpb80Tis%xC!q+V5ki-m5GJ=o`Jc#IJ@xo#cpmi4OpDZZxhHvHy2hEBy&p9N5zUwGi2rXv80b4Q0zFp%5)A7HqPp4z=>-Xu!^`wmqK%%VH'
'iNJF+9-0hHvlwZaju;$gEA9B_gB*1&KQ(U2VtP+`h&CUBo%2k)EO$rGI*dGs^v+qs81sxt*R|qCh2{CtqUi;UV}`!pH3+k2zp)$X(0KZD?45r>'
'1BbgGF+=s)w)gSR5Bo->%^NFne{X}x^V1*R<<-&j$~ze!eVblfoEx!_er}n1ot6#s(Tv9Q@_Ndx_>^y6aeFkw%&o595QqjzD8s^Ina$_W6TqR7'
'2gfG4W(OD$r@7iS#XMSR6?b+(k7shuzt}*GR>p+ti!~Y6tYEat4XjSLWgK|(l)*l^+si4PioiDFe9r?WJd;Q+HU|g#Fwn(+$Kyal0XWam6P=dQ'
'pR(V~#~jUM26g^qOFm#g#lUJFCz&GiltxVE9y2}I&M#oL&~Rb-g*1x+IsVV07c)#^(;)R9mfVL!Z9doIPFeOy+4a~OL9eAkz7o~T-K@_9&AAE9'
'5zi@ju!JrL_b)m!&7aFnZnQjzaqk{Whv7E(qbFGiF7yl!MIP(i9}JvsB^6-KQapeQfEnqg$}y8i4?KWqEg85<X<1^9U)V9)I>MlKZY#p13Ph$x'
'6*VWPlQB*XK&9&VxAD&(*Jo#4-9X}+_i%h^fH%#Y6<WI1_@Hm1g7JUF#a~4h24z5OvglNU136+bI#}i&w*CPZqj`Y3G?1>{(iqh-XKCrko7;Na'
'?VDHBx-{LG<;W6NTAfr%{Q_c*>xuEbB_dY-&EWRN(eIsGOW3M!Y`AjWptsWa0aa#U@#rt_^!(`i*qfaGH9os^P<^aM&3<_VkmL{s0jMqbF}c3V'
'7g!W7xEiAHunEF;Ur3kT!H4(z1I;xGcWL#>rx6(-|F>4=kLKmOeE4&|)ys};V%w<>_X;h0CMzxT`Eg}rK5L4@0%<O2dU(~6gGw*qk1Slr%AzLi'
'*|4%&Js)CG;)sG6E?MJR2SvBTq^GhNz||K~1PzjFzYjpXR5uXOThkL)?iq?VMbE95KnU<nE}`@?pbTQ6lV2ogCT?dzu5|h_0OuOu<zmhFKgiV*'
'&IU3M=|YkylUPp7XMrs!iDqLmry{g2Om)KtBvZl9MM{&UW)Cct{TlpMc1GyK9n5t{Pc4!BYAEU%IbO>U=n3uUWAUoVO;IREi7Aa2%nhNn&s_vZ'
'`mn9b2F-H+@LMbhtrVJ543j^|P`?5)wxLgN&PAqednu@^%A$PY@IX(yae9zco+h*wVE6Y*a?<IxZ>K+PjvBt?t>S%c;I?)lP-H^E?6dq@=%uu^'
'h^|>Z&=ERgp+4$$5?79ZuUaKwmlHEkp8J;?(1@K#Lb1?!7I1l|qal?@fZ+cYv;5fSkcqmCm-q|FUcp2As65JrL@p#wFE3444I|d)jwBQr7e%Fj'
'&tlO~vdAc;j{>kDOPU6jo*97}2o7Nq<aP|0fxG$??|gU|65;z<3hJo`#$qd0BEA9Y1K^X%lUj;6Co28IcJl)Z1UroC&1Aji+wsGIM*U!F6x*iu'
'Ob03_9C&$_1+Z~>a{M|uy_}4v`al5xA)D4HWA6#UP0~*}&>^h3T+ixBa(sL~8Go;?FE`7(2Ruyvx!*iZN_Fr+`E$RmE%+Nc4Sgq=-+1@07~Wme'
'^ywI$Tz&lqS>And0+62SjhoHB8jqXkOUEL7mO*LBmNmg&{q`tT`gQ{AtRD#ov52|v*19taEC#03Z#LHNXn@FuMOE|@5r)`muC=Tcd~9wCP?~G>'
'xv2b2_m@^H=zVmu<>xT0UD>72G8f{s3z>nGyH?FwQoXjjc6{|(IJcSyo%SWjI$_o4GDA1We$Bcv+!}doWw|Sq4*ir(y0xb2jke?G4Yv|4))lL;'
'RPzU*24HgW1V>ZR0KZ^<dmFsbN$v}086g!PBGf0~LW}eDD-8Cyq(DiKmA&9LBd}<gt9G{Z%~iu+=$_X%c)6Q4Tny81l!yK8qG`g-%1ZhgWP{{&'
'0l3<m^5$nlBH(93YUyg1A!CU<=nvO#Sxr1!(jqS|h5z)7scOx_bv$Y=f1q5sMTgiT!roO@@bt>e9zQ17Rlg$tjw9l|L5Iei29bMh$sAk|hD~F2'
'0lpD=UsHI~&ftl@WbDozR0iQdv4dT*+cs`*lkr(Gtr-yJ{r4aD>i=d~zm#ZFy7F8rt}TlGy{nZ&^67TWUgmfGe>1Ha*3$o3Wch<Qhj?qJTQQ3g'
'bH0Fj)(nyczp;hEm)MhFo3A@i?9$f%gJLY)-uf?4O9KQH000080001DU29Q+E~*9q03RCw01^NI0BvPtX<=V#Z(?6_bZ>HHE^v9(SzT}2HWYm~'
';Qzq#C01uUZjk|lHw{Z$cT3mQf#U!hf<TaMDp4bg8b!4+JpcDyen_Gu%5~bH!1NH=<mDxK&f&e6PQd6Id0vq5j1kX6;X0v=BR`HQ_jyR;wA<-+'
'bSq7Zk33wHZYL0<`6dbD+kE6{yfH>F@=r!mE}e8c=hHV6<bs>wE#V~o)WheKH)p5byXh<Maz33+JYj+R2s5&#oM2~|auy~%XBg4Xg!P<m=k@f%'
'd^UaWUA#a2F!5fUUQWEL+54(jyN&Q`cx37WKE~j+zK+w2Gw;vI-}@J<jjDdTo%z}MWO_CCE+?>#H<#dJ?LYKh4o7_yMAYZKJuYiwh3lWN<AB%M'
'WIj8a<lbKN(K?KKCw&ygdmZXw!qH3N$WcrzBR@^iIa!7NJG!{!l#y|_gWy?_0C^Ez`jMB4=3Yvoz|o!Jk(`Hd!2DrCSQ@4<@si~9%Q47Lk>R2o'
'%LvFUuJ#$zD%QOvjf3!3O^gP=BAz7?xdwdu;GW$WYY77cIz@KA4LdovAEpG&vRFuW!Wd<3LP23F-KurKxmXE_)aGmv!vT}FX>ISU8aexd`)#YK'
'p)1c~uXHL9?w4QJgx}HCSb%J_CO*Hbw>>vLe@toYE$C`9Myqhi6;BWaZqUEz1C0qNR=kW6egvo4I~kpf(Sp(l4Bq-tnp;T9^SCjbku-rIM6@#l'
'-h~fC&^_6}c(IXTO+E&A855HPcq_*2wqi#+Tz20Pe+5bglWmnf{MSGNI{;t@;-rtoeEpiy>2gUD4t>~9k|<otuz4(C!G`IsqM!n_ob}q~jAdQp'
'*EEJR#s~9FLhubNvpG>tw0PiXbB_qEjnU*mtPRYnWEiKJGURhk)(MA04SJ1lNlyn2r?w!@HnkS<u$&>cWUVPR^HDnM+dF})mL83VtKThF0uHx;'
'x5Y@DMw)TA7H%!pPO}N<D-;Zga#bJzDZuAh3IbdK8ZHtFZ<*hUWd5uO_rplysU3^`K+1w3-@~dGTNWlBA_*^fKV+J!B5GUJR-lb7zKZW-`Z)#y'
'A#b3MbRp78-Uy6mL^kgBOseBp`;?Z5+3pRW=PS{Phs3LdLQ2&0kQESwvY0$1WXa*13_7^oA;gFRbWR@l_4vhTG`iWf46&=js)t|DkE2lwFvS45'
'c@NH{r-t@gSk@@!<C;cg3;=!)05fPHSpfLQe13tpwmt1yp?n04wH<2x8u4F#DB~EE7GPrmva)3`F+-ww#Ie<Z?g#V&_|Q?#;!2=3Od~>)-e`D&'
'o}->IQk75z6HU3=u#}rX0h}ZP33gkqb%z}@NhcZiU|f5ShE6YR<mme_ZnP<)n%R5<H7FQEhHhqNAT5h{OQ%jL;Ln=bW!|o(F~D5a{E)L`(T8hj'
'tHf=Vo6fL_L3Q(%C@uDHX;u(eIy4Hdep@UQ60Uk#`Xv~y{9Eg-30j?_CM3_?;+FH}nz1>50C6|X5>a!*8MR%b+^zWgjO~6#Mff|CNFO0MBS&iT'
'BivYhReM8A-6$S>aW%)atq1rRVlQE_A05jaF#yKW<E`RtS2r)Rw7YF?5DQL4!DXiwD{k?)on)-p1)FxBv0@l(TB*j8Ww5ZZy)(X7PBP?bsbYD4'
'&0{OiiM3LG%)!waz;=G}nbn-m|5JKKRk>G%zfZnZg+(pr&RIsPbQ;JN3xa~(Qmk9+8d=~s%D74}hi+k|N;^?JHSc`F3KkHiVGLS_yS8AqFD2w4'
'0v9JeYpFe1ig_fK@1_F#M!qN$LRFF$m5yzJA!KmjifFb9A|F1g8Jz=o#23@cxu_qlWJy2AdIxpJtd9{)u-Yq}Se-Dc!Nd|6c1_f4q+LsLZki|y'
';=b;TKLv<V{%wj^AeOqXN&V10SJb~pZ_CIJ*BT4B5D=ynb{F2>c}qX>7ZIu6_`e%W2WU;M{?SU4`ve`yH%wse6`IYjU)ES92-MMqTQB1a;#<^K'
'g`3lR5|=6zcz+d|I(XSmQ+?NRO873{yC))~*f7|Ykc&S}AjKwoRy6%6{6w&00W=8XFugk*%{FP6b#_reTY~9#C*qK*sZswt6?+QZZ=z1Q5I=)j'
'*z=bPIJjKO6?`Gfm6S>@$~(+cGg3`76@H!yTD|o_zV$~2EjtZX29%r%^nVkg376{UyONMSknOn=KB@@SH){10lb$ZMm!8U}{9TUU5%juPllWns'
'Ga3z&$cMXY!sx>W>wkx+Yr0$ghlpVOKTt~n0u%rg000080001DU9eRo+Pw|{0CX|{01W^D00000000000HlEc0001CX<uV+ZE$aMWiD`eP)h*<'
'6aW+e000O8003fLvd@AMZw3GW{tf^D6#xJL0000000000q=D!T003cWUt@1=aBp*EUt@1=ZEtRKE^v8JO928D02BZK00;m80AgM1tGt8yBme+_'
'fB*m#00000000000001_fr=FX0AXogV{dJ6Z*yf|Wn^h|Z*nehc~DCM0u%rg000080001DT|Ac3l{IVt0MDKQ02BZK00000000000HlGoIsgD+'
'X<uV+ZE$aMWnX1(XK8L_E^v8JO928D02BZK00;m80AgKY6}YYK4gdfSH~;_;00000000000001_fgPs+0AXogV{dJ6Z*yf|X>u-bc~DCM0u%rg'
'000080001DU2nn&$3YAL03RU$02BZK00000000000HlF9wg3QOX<uV+ZE$aMWnXJyXJ>3>E^v8JO928D02BZK00;m80AgLfdEwr(0ssI%1pojS'
'00000000000001_fvds*0AXogV{dJ6Z*yf|Y;R*>Y+qq;a4v9pP)h*<6aW+e000O8003fL?4EHRE(8Do;Ryf$82|tP0000000000q=A#g003cW'
'Ut@1=aBp*EUu<t<VQgP>bZ>HHE^v8JO928D02BZK00;m80AgJ+iGDG|3;+NXA^-ps00000000000001_f%wS)0AXogV{dJ6Z*yf|ZEtjGZ*DGd'
'c~DCM0u%rg000080001DUEA-QN$U#$0Hq}W02BZK00000000000HlHK)&KxuX<uV+ZE$aMWnXY_Y-wY8E^v8JO928D02BZK00;m80AgLIYV7+K'
'3IG71CIA2x00000000000001_fePaQ0AXogV{dJ6Z*yf|adlyAX>@rmaCuNm0Rj{N6aWAK2mk;8VqF{IDj8)C004Y9000;O0000000000005+c'
'Q|$l%VQF7uZ*6dIb7fz0WoBt^Wo>0{bS`jtP)h*<6aW+e000O8003fLbyZ0jSq%UHWibE%6aWAK0000000000q=D}H003cWUt@1=aBp*EUvpz<'
'Wo=<DaCuNm0Rj{N6aWAK2mk;8VqGHQnUx+1007z=000&M0000000000005+cc?kgkVQF7uZ*6dIb7fz1Wn*+{Z*FrgaCuNm0Rj{N6aWAK2mk;8'
'VqFlsl*1ka003|X000vJ0000000000005+c%Mt+qVQF7uZ*6dIb7fz1Z*^{DE^v8JO928D02BZK00;m80AgJ|67Qit1ONc#2mk;S0000000000'
'0001_ff^P80AXogV{dJ6Z*yf|c4>2IZ*DGdc~DCM0u%rg000080001DUE6|*hp7Vq00RgB022TJ00000000000HlG68UX-pWn^h#UvzR|ZgXRD'
'X<}tAaCuNm0Rj{N6aWAK2mk;8VqJYSYHi~R007}1000sI0000000000005+cWF7$kZDnL>VPA4(b8mHWV`XzLaCuNm0Rj{N6aWAK2mk;8VqI%d'
'fi9{B0018w000sI0000000000005+ccPRk?ZDnL>VP9%*VqbG~Z*pZWaCuNm1qJ{B000vJ69DA|002=i0RR91'
)
BUNDLE_SHA256 = '5eaf9e1973fa10c9edb77c62a73fee7d3b150897c6fc993058d24ef17e424202'
raw = base64.b85decode(BUNDLE)
assert hashlib.sha256(raw).hexdigest() == BUNDLE_SHA256, 'Damaged code bundle'
RUNTIME = Path('/kaggle/working') / ('sh01-onecell-' + BUNDLE_SHA256[:12])
CODE = RUNTIME / 'code'
CODE.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    for name in archive.namelist():
        target = CODE / name
        if not target.resolve().is_relative_to(CODE.resolve()) or not name.endswith('.py'):
            raise RuntimeError('Unexpected file in source bundle')
        data = archive.read(name)
        if target.exists() and target.read_bytes() != data:
            raise RuntimeError(f'Bundled code was locally edited: {target}; preserve it before rerunning')
        target.write_bytes(data)
source = Path(SOURCE_PATH).expanduser()
if not source.is_file():
    raise FileNotFoundError(f'Mount the video and set SOURCE_PATH to the exact filename: {source}')
if shutil.which('nvidia-smi') is None:
    raise RuntimeError('Enable a Kaggle GPU accelerator before running this cell')
subprocess.run(['nvidia-smi'], check=True)

"""Repair a private editor venv while retaining Kaggle's CUDA package paths.

Uses only the standard library until pip is working. This source is inlined by
build_one_cell.py so the notebook does not need a checkout or a second download.
"""
import importlib.util
import json
import os
from pathlib import Path
import site
import subprocess
import sys
import venv


def prepare_editor_environment(runtime, *, host_paths=None):
    """Create or repair the owned venv; never install into the host interpreter.

    Debian/Kaggle can expose dist-packages in the kernel but omit those paths
    from a venv. A local .pth file adds the actual host paths after local packages.
    The host's pip can install into a pip-less venv using its --python option.
    """
    runtime = Path(runtime).expanduser().resolve()
    runtime.mkdir(parents=True, exist_ok=True)
    root = runtime / 'venv'
    python = root / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')
    log_path = runtime / 'setup.log'
    child_env = os.environ.copy()
    for key in ('PYTHONHOME', 'PYTHONPATH', 'PIP_TARGET', 'PIP_PREFIX', 'PIP_USER'):
        child_env.pop(key, None)

    def execute(args, *, check=True):
        """Keep subprocess startup diagnostics, including sitecustomize errors."""
        result = subprocess.run([str(x) for x in args], env=child_env,
                                capture_output=True, text=True)
        with log_path.open('a', encoding='utf-8') as stream:
            stream.write(result.stdout + result.stderr)
        if check and result.returncode:
            raise RuntimeError(
                f'Editor environment setup failed. Log: {log_path}\n'
                + (result.stdout + result.stderr)[-3000:])
        return result

    if not python.is_file() or not (root / 'pyvenv.cfg').is_file():
        # Do not rely on distro ensurepip, and do not clear an existing runtime.
        venv.EnvBuilder(with_pip=False, system_site_packages=True).create(root)

    probe = execute([python, '-I', '-c',
        'import sys,sysconfig,site,json; print(json.dumps({'
        '"prefix":sys.prefix,"version":list(sys.version_info[:2]),'
        '"sites":site.getsitepackages(),'
        '"purelib":sysconfig.get_path("purelib"),'
        '"platlib":sysconfig.get_path("platlib")}))'])
    info = json.loads(probe.stdout.strip().splitlines()[-1])
    if Path(info['prefix']).resolve() != root.resolve():
        raise RuntimeError(f'Refusing to install outside the editor venv: {info["prefix"]}')
    if info['version'] != list(sys.version_info[:2]):
        raise RuntimeError('The editor venv uses a different Python version. Use a new runtime directory.')

    local_sites = []
    for value in [info['purelib'], info['platlib'], *info['sites']]:
        path = Path(value).resolve()
        if path.is_relative_to(root) and path not in local_sites:
            path.mkdir(parents=True, exist_ok=True)
            local_sites.append(path)
    if not local_sites:
        raise RuntimeError('No package directory inside the editor venv was found.')

    if host_paths is None:
        host_paths = [*site.getsitepackages(), *sys.path]
    shared = []
    for value in host_paths:
        if not value:
            continue
        path = Path(value).expanduser().resolve()
        if (path.name not in ('site-packages', 'dist-packages') or not path.is_dir()
                or path.is_relative_to(root) or path in shared):
            continue
        if '\n' in str(path) or '\r' in str(path):
            raise RuntimeError('A package directory contains an invalid newline.')
        shared.append(path)
    # Also bridge distro purelib/site differences inside the venv itself.
    for local in local_sites:
        paths = [p for p in local_sites if p != local] + shared
        (local / 'sh01-kaggle-packages.pth').write_text(
            ''.join(str(p) + '\n' for p in paths), encoding='utf-8')

    health_code = (
        'import json,sys,pip,wrapt; '
        'print(json.dumps({"prefix":sys.prefix,"pip":pip.__file__,"wrapt":wrapt.__file__}))'
    )

    def healthy(result):
        if result.returncode or 'Error in sitecustomize' in result.stderr:
            return False
        try:
            data = json.loads(result.stdout.strip().splitlines()[-1])
            return (Path(data['prefix']).resolve() == root and
                    all(Path(data[key]).resolve().is_relative_to(root) for key in ('pip', 'wrapt')))
        except (ValueError, IndexError, KeyError):
            return False

    health = execute([python, '-I', '-c', health_code], check=False)
    if not healthy(health):
        (runtime / 'dependencies-ready').unlink(missing_ok=True)
        spec = importlib.util.find_spec('pip')
        if spec is None or not spec.origin:
            raise RuntimeError('The notebook Python has no pip. Install pip for that interpreter first.')
        # pip's directory __main__ prepends its entire site-packages directory,
        # allowing legacy dataclasses/typing backports to shadow the stdlib.
        # Its cross-interpreter runner redirects only the pip package instead.
        runner = Path(spec.origin).parent / '__pip-runner__.py'
        if not runner.is_file():
            raise RuntimeError('Host pip needs its cross-interpreter runner (pip >=22.3).')
        # --ignore-installed makes both bootstrap packages local without removing
        # any globally installed version. --no-deps prevents unrelated changes.
        print('Repairing editor pip and startup dependency…', flush=True)
        execute([sys.executable, '-I', runner, '--python', python,
                 'install', '--disable-pip-version-check', '--no-input',
                 '--ignore-installed', '--no-deps', 'pip>=23,<27', 'wrapt>=1.16,<3'])
        health = execute([python, '-I', '-c', health_code], check=False)
        if not healthy(health):
            raise RuntimeError(f'Editor startup check still fails; see {log_path}\n'
                               + (health.stdout + health.stderr)[-3000:])
    return python, child_env

PYTHON, setup_environment = prepare_editor_environment(RUNTIME)
# Use a private environment; preserve Kaggle's CUDA-enabled Torch installation.
import importlib.metadata
constraints = RUNTIME / 'constraints.txt'
constraints.write_text('torch==' + importlib.metadata.version('torch') + '\n')
setup_log = RUNTIME / 'setup.log'

def run_setup(command, *, env=None):
    """Record setup failures without putting secrets or huge pip logs in output."""
    with setup_log.open('a') as stream:
        result = subprocess.run([str(x) for x in command], stdout=stream, stderr=subprocess.STDOUT,
                                env=setup_environment if env is None else env)
    if result.returncode:
        print(setup_log.read_text(errors='replace')[-4500:])
        raise RuntimeError(f'Setup failed; full log: {setup_log}')

stamp = RUNTIME / 'dependencies-ready'
setup_version = 'bootstrap-v3-av18;torch=' + importlib.metadata.version('torch')
if not stamp.exists() or stamp.read_text() != setup_version:
    print('Installing isolated editor dependencies…', flush=True)
    run_setup([PYTHON,'-m','pip','install','--disable-pip-version-check','-q','-c',constraints,
        'fastapi>=0.115,<1','uvicorn>=0.30,<1','pydantic>=2.9,<3','requests>=2.32,<3',
        'httpx>=0.27,<1','python-dotenv>=1,<2','Pillow>=10','numpy>=1.26',
        'transformers>=4.51,<5','accelerate>=1,<2','bitsandbytes>=0.45,<1',
        'json-repair>=0.30,<1','faster-whisper>=1.1,<2','av==18.1.0','sentencepiece'])
check = subprocess.check_output([str(PYTHON),'-c',
    'import torch, bitsandbytes, transformers, json, pathlib, site; '
    'assert torch.cuda.is_available(), "CUDA unavailable in fresh subprocess"; '
    'import sys; roots=[pathlib.Path(p) for p in sys.path if p]; '
    'libs=[str(p) for root in roots for p in root.glob("nvidia/*/lib")]; '
    'print(json.dumps({"torch":torch.__version__,"gpu":torch.cuda.get_device_name(0),"libraries":libs}))'],
    text=True,env=setup_environment)
gpu = json.loads(check.strip().splitlines()[-1])
# Test the actual decoder in the isolated interpreter before loading any models.
run_setup([PYTHON, '-c', """
import io, wave
from faster_whisper.audio import decode_audio
buffer = io.BytesIO()
with wave.open(buffer, 'wb') as wav:
    wav.setnchannels(1)
    wav.setsampwidth(2)
    wav.setframerate(16000)
    wav.writeframes(b'\\x00\\x00' * 16000)
buffer.seek(0)
assert len(decode_audio(buffer, sampling_rate=16000)) == 16000
"""])
stamp.write_text(setup_version)
print('GPU ready:', gpu['gpu'], '| Torch', gpu['torch'], flush=True)
if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
    run_setup(['apt-get','update','-qq'])
    run_setup(['apt-get','install','-y','-qq','ffmpeg','fonts-dejavu-core'])

MODELS = Path('/kaggle/working/sh01-editor-models'); MODELS.mkdir(exist_ok=True)
def download(url, path):
    """Atomic public model/binary download; retry transient network errors."""
    if path.is_file() and path.stat().st_size:
        return
    partial=path.with_suffix(path.suffix+'.part')
    for attempt in range(3):
        try:
            request=urllib.request.Request(url,headers={'User-Agent':'SH01LocalEditor/2.0'})
            with urllib.request.urlopen(request,timeout=120) as response, partial.open('wb') as out:
                shutil.copyfileobj(response,out,1024*1024)
            partial.replace(path);return
        except Exception:
            partial.unlink(missing_ok=True)
            if attempt==2: raise
            time.sleep(2**attempt)

# The standalone Piper distribution avoids Python-version wheel mismatches.
PIPER = MODELS / 'piper' / 'piper'
if NARRATION != 'off' or 'voiceover' in REQUIRED_FEATURES:
    if not PIPER.is_file():
        print('Installing local narration engine…',flush=True)
        packed=MODELS/'piper_amd64.tar.gz'
        download('https://github.com/rhasspy/piper/releases/download/v1.2.0/piper_amd64.tar.gz',packed)
        with tarfile.open(packed) as archive:
            archive.extractall(MODELS,filter='data')
        PIPER.chmod(0o755)
    locale, speaker, quality = VOICE_NAME.split('-')
    language = locale.split('_')[0]
    voice_url=f'https://huggingface.co/rhasspy/piper-voices/resolve/main/{language}/{locale}/{speaker}/{quality}/'
    voice=MODELS/(VOICE_NAME+'.onnx')
    for name in (VOICE_NAME+'.onnx',VOICE_NAME+'.onnx.json'):
        download(voice_url+name,MODELS/name)
    download(voice_url+'MODEL_CARD',MODELS/(VOICE_NAME+'-MODEL_CARD.txt'))
    run_setup([PIPER,'--help'])
else:
    voice=MODELS/(VOICE_NAME+'.onnx')

WORK = Path('/kaggle/working/ai-compose-onecell-jobs') / time.strftime('%Y%m%d-%H%M%S')
WORK.mkdir(parents=True,exist_ok=True)
environment = {
    'PATH':str(PIPER.parent)+os.pathsep+os.environ.get('PATH',''),
    'HF_HOME':str(MODELS/'huggingface'),
    'AI_COMPOSE_HF_MODEL':TEXT_MODEL,'AI_COMPOSE_HF_LOAD_4BIT':'true',
    'AI_COMPOSE_HF_MAX_NEW_TOKENS':'3000','AI_COMPOSE_VISION_HF_MODEL':VISION_MODEL,
    'MEDIA_CLIP_WHISPER_MODEL':'small','MEDIA_CLIP_WHISPER_DEVICE':ASR_DEVICE,
    'MEDIA_CLIP_WHISPER_COMPUTE_TYPE':'float16' if ASR_DEVICE=='cuda' else 'int8',
    'MEDIA_CLIP_WHISPER_CPU_THREADS':'4','MEDIA_CLIP_WHISPER_CACHE_DIR':str(MODELS/'whisper'),
    'AI_COMPOSE_TTS_MODEL':str(voice),'AI_COMPOSE_MUSIC_PATH':MUSIC_PATH,
    'LD_LIBRARY_PATH':os.pathsep.join(gpu['libraries']+[os.environ.get('LD_LIBRARY_PATH','')]),
    'AI_COMPOSE_USE_NVENC':'false','AI_COMPOSE_VIDEO_CODEC':'libx264',
    'TOKENIZERS_PARALLELISM':'false','OMP_NUM_THREADS':'4',
}
payload = {
    'brief':BRIEF,'preset':'viral_short','target_seconds':TARGET_SECONDS,
    'output_width':OUTPUT_SIZE[0],'output_height':OUTPUT_SIZE[1], 'output_count':1,
    'captions_enabled':True,'fetch_assets':True,'llm_required':True,
    'refinement_iterations':REFINEMENT_ROUNDS,'local_asset_library':LOCAL_ASSET_LIBRARY or None,
    'asset_sources':{'local_library':True,'wikimedia':True,'pexels':True,'pixabay':True},
    'editorial_policy':{'source_audio':'preserve','supplementary_visuals':'require',
        'camera_motion':'dynamic','motion_graphics':'hero'},
    'editor':{'enabled':True,'narration':NARRATION,'narration_budget_seconds':12,
        'caption_style':'highlight','theme':'editorial','progress_bar':True,
        'music':'library' if MUSIC_PATH else 'procedural','procedural_sfx':True,
        'transition':'dissolve','min_graphics':2,'min_visual_layers':1,
        'required_features':REQUIRED_FEATURES,'visual_review_required':True,
        'strict_requirements':True,'publish_outputs':False},
    'sfx':{'mode':'adaptive','volume':0.20}, 'music_volume':0.12,
}
config={'source_path':str(source),'env_file':PRIVATE_ENV_FILE,'environment':environment,
    'work_dir':str(WORK),'payload':payload,'timeout_seconds':TIMEOUT_SECONDS,'keep_api':KEEP_LOCAL_API}
config_path=WORK/'private-config.json';config_path.write_text(json.dumps(config,indent=2));config_path.chmod(0o600)
# Run outside the notebook kernel: no stale imports and no foreground Uvicorn cell.
from IPython.display import Audio, FileLink, Video, display
import numpy as np

def beep():
    """Notify at readiness, completion, and failure; browser autoplay rules apply."""
    t=np.linspace(0,.5,4000,endpoint=False)
    display(Audio(np.sin(2*np.pi*440*t),rate=8000,autoplay=True))

beep()
print('Running the integrated editor. Evidence, drafts, and logs are saved under:',WORK,flush=True)
runner_log=WORK/'driver.log'
with runner_log.open('w') as stream:
    process=subprocess.Popen([str(PYTHON),'-u',str(CODE/'ai_compose_kaggle.py'),str(config_path)],
        cwd=CODE,stdout=stream,stderr=subprocess.STDOUT,env=setup_environment)
    offset=0
    try:
        while process.poll() is None:
            time.sleep(2)
            with runner_log.open() as current:
                current.seek(offset);message=current.read();offset=current.tell()
            if message: print(message,end='',flush=True)
    except KeyboardInterrupt:
        process.send_signal(2)
        try: process.wait(timeout=25)
        except subprocess.TimeoutExpired: process.kill();process.wait()
        beep()
        raise
with runner_log.open() as current:
    current.seek(offset);print(current.read(),end='')
beep()
if process.returncode:
    display(FileLink(str(runner_log)))
    display(FileLink(str(WORK/'server.log')))
    for diagnostic in WORK.glob('*-output.zip'): display(FileLink(str(diagnostic)))
    raise RuntimeError('No passing final render was produced. Review the saved diagnostics; requirements were not silently disabled.')
result=json.loads((WORK/'latest-output.json').read_text())
for output in result['outputs']:
    display(Video(output['local_path'],embed=True,width=360))
    display(FileLink(output['local_path']))
display(FileLink(result['archive']))
display(FileLink(result['receipt']))
if 'api_connection_file' in result:
    print('Loopback API remains available. Connection details:',result['api_connection_file'])
print('Save a Kaggle notebook version with outputs to retain these files after the session ends.')
